# ARC Prize 2026 — ARC-AGI-3 Submission

Built from `agent/my_agent.py`, `agent/arclog.py` and support modules via `scripts/build_notebook.py`. Do not edit cells directly — edit the source file and re-run `make submit`.

Only the competition gateway writes evaluated rows. Ordinary saves are logged as NOT_EVALUATED and emit a zero-row parquet for code submission. Diagnostics are saved under `/kaggle/working/diagnostics/`.

In [ ]:
"""Notebook diagnostics and gateway-only evaluation; embedded by the builder.

Only the competition gateway writes evaluated submission rows. Save runs emit
an empty schema-only output so Kaggle can select the file for a code rerun.
All runtime imports outside submission validation use the Python standard library.
"""
from __future__ import annotations

import hashlib
import importlib.metadata
import importlib.util
import json
import logging
import os
import shutil
import signal
import subprocess
import sys
import time
import traceback
from contextlib import contextmanager
from pathlib import Path
from textwrap import dedent


class NotebookRunner:
    """Keep notebook phases, subprocess output and failures visible on disk."""

    def __init__(self, build_id: str, *, working: Path = Path('/kaggle/working'),
                 bundle: Path = Path('/tmp/agent_bundle'),
                 competition: Path = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3'),
                 preflight: Path = Path('/tmp/agent_preflight'),
                 audit_script: Path = Path('/tmp/submission_audit.py')) -> None:
        self.build_id = build_id
        self.working, self.bundle = working, bundle
        self.competition, self.preflight = competition, preflight
        self.audit_script = audit_script
        self.framework = working / 'ARC-AGI-3-Agents'
        self.diagnostics = working / 'diagnostics'
        self.submission = working / 'submission.parquet'
        self.started = time.time()
        self.target_seconds = 29_700
        self.hard_limit_seconds = 30_600
        self.submission_wait_seconds = 30.0
        self.rerun = False
        self.verbose = os.getenv('ENABLE_NOTEBOOK_DIAGNOSTICS', '1') == '1'
        self.initial_submission: tuple[int, int, int] | None = None
        self.llm = None
        self.state: dict = {}
        self.logger = logging.getLogger(f'arc3.notebook.{id(self)}')
        self.logger.setLevel(logging.INFO)
        self.logger.propagate = False

    def elapsed(self) -> float:
        return time.time() - self.started

    def event(self, name: str, **details) -> None:
        record = {'event': name, 'elapsed_seconds': round(self.elapsed(), 3),
                  'build_id': self.build_id, **details}
        line = json.dumps(record, sort_keys=True)
        self.logger.info('%s', line)
        with (self.diagnostics / 'events.jsonl').open('a', encoding='utf-8') as stream:
            stream.write(line + '\n')

    def status(self, state: str, **details) -> None:
        self.state.update(status=state, elapsed_seconds=round(self.elapsed(), 3),
                          build_id=self.build_id, competition_rerun=self.rerun, **details)
        temporary = self.diagnostics / 'status.json.tmp'
        temporary.write_text(json.dumps(self.state, indent=2) + '\n')
        temporary.replace(self.diagnostics / 'status.json')

    @contextmanager
    def stage(self, name: str):
        began = time.time()
        self.status('RUNNING', stage=name)
        self.event('stage_started', stage=name)
        try:
            yield
        except Exception as error:
            detail = traceback.format_exc()
            with (self.diagnostics / 'traceback.log').open('a', encoding='utf-8') as stream:
                stream.write(f'\nStage: {name}\n' + detail)
            self.logger.exception('Stage %s failed', name)
            self.status('FAILED', stage=name, error_type=type(error).__name__, error=str(error),
                        submission_exists=self.submission.is_file())
            self.event('stage_failed', stage=name, error_type=type(error).__name__, error=str(error))
            raise
        else:
            self.event('stage_completed', stage=name, duration_seconds=round(time.time() - began, 3))

    def _signature(self) -> tuple[int, int, int] | None:
        if not self.submission.exists():
            return None
        stat = self.submission.stat()
        return stat.st_ino, stat.st_mtime_ns, stat.st_size

    def start(self) -> None:
        self.started = time.time()
        self.diagnostics.mkdir(parents=True, exist_ok=True)
        formatter = logging.Formatter('%(asctime)s %(levelname)s %(message)s')
        for handler in (logging.StreamHandler(sys.stdout),
                        logging.FileHandler(self.diagnostics / 'notebook.log', encoding='utf-8')):
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
        with self.stage('startup'):
            raw_flag = os.getenv('KAGGLE_IS_COMPETITION_RERUN', '')
            flag = raw_flag.strip().lower()
            if flag not in ('', '0', 'false', 'no', '1', 'true', 'yes'):
                raise ValueError('Unrecognized KAGGLE_IS_COMPETITION_RERUN value')
            self.rerun = flag in ('1', 'true', 'yes')
            os.environ['ARC3_RUN_STARTED'] = str(self.started)
            os.environ['ARC3_LOG_DIR'] = str(self.working / 'scratch')
            for name in ('ENABLE_SANDBOX_LIMITS', 'ENABLE_PREFIX_LOCK', 'ENABLE_LEVEL_BUDGET',
                         'ENABLE_ACTION_FALLBACK', 'ENABLE_NOTEBOOK_DIAGNOSTICS'):
                os.environ.setdefault(name, '1')
            (self.bundle / 'baseline').mkdir(parents=True, exist_ok=True)
            self.initial_submission = self._signature()
            self.event('execution_mode', mode='COMPETITION_RERUN' if self.rerun else 'SAVE_OR_INTERACTIVE',
                       rerun_flag=raw_flag, kernel_run_type=os.getenv('KAGGLE_KERNEL_RUN_TYPE', ''),
                       python=sys.version.split()[0], existing_submission=self.initial_submission is not None,
                       submission_policy='gateway-only', diagnostics=str(self.diagnostics))
            if self.verbose:
                gpu = shutil.which('nvidia-smi')
                if gpu:
                    try:
                        result = subprocess.run(
                            [gpu, '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True, timeout=5, check=False)
                        self.event('gpu_probe', returncode=result.returncode,
                                   output=(result.stdout + result.stderr)[-2000:])
                    except (OSError, subprocess.TimeoutExpired) as error:
                        self.event('gpu_probe_unavailable', error=str(error))
                else:
                    self.event('gpu_probe_unavailable', reason='nvidia-smi not installed')

    def command(self, name: str, args: list[str], *, timeout: float, cwd: Path | None = None) -> None:
        remaining = self.target_seconds - self.elapsed()
        if remaining <= 0:
            raise TimeoutError(f'Global run target exhausted before {name}')
        path = self.diagnostics / f'{name}.log'
        self.event('command_started', command=name, args=args, log=str(path))
        try:
            with path.open('w', encoding='utf-8') as stream:
                result = subprocess.run(args, cwd=cwd, stdout=stream, stderr=subprocess.STDOUT,
                                        timeout=min(timeout, remaining), check=True)
            self.event('command_completed', command=name, returncode=result.returncode)
        finally:
            # The complete stream stays on disk, including when a command fails.
            if path.exists():
                with path.open('rb') as stream:
                    stream.seek(max(0, path.stat().st_size - 16_384))
                    tail = stream.read().decode(errors='replace')
                if tail:
                    self.logger.info('%s output (tail):\n%s', name, tail)

    def install_dependencies(self) -> None:
        with self.stage('offline_dependencies'):
            wheelhouse = self.competition / 'arc_agi_3_wheels'
            if not wheelhouse.is_dir():
                raise FileNotFoundError(f'Competition wheelhouse is missing: {wheelhouse}')
            self.command('pip', [sys.executable, '-m', 'pip', 'install', '--no-index',
                                '--find-links', str(wheelhouse), 'arc-agi', 'python-dotenv'], timeout=600)
            self.event('dependency_versions', versions={name: importlib.metadata.version(name)
                       for name in ('arc-agi', 'arcengine', 'python-dotenv')})

    def verify_sources(self, expected: dict[str, str]) -> None:
        with self.stage('bundle_verification'):
            for name, digest in expected.items():
                path = self.bundle / name
                if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
                    raise ValueError(f'Bundled module hash mismatch: {name}')
                compile(path.read_text(), str(path), 'exec')
            self.event('bundle_verified', modules=len(expected))

    def prepare_framework(self) -> None:
        with self.stage('gateway_readiness'):
            self.command('gateway', ['curl', '--fail', '--show-error', '--silent',
                         '--connect-timeout', '5', '--max-time', '15', '--retry', '999',
                         '--retry-all-errors', '--retry-delay', '5', '--retry-max-time', '600',
                         'http://gateway:8001/api/games'], timeout=620)
        self.prepare_offline_framework()

    def prepare_offline_framework(self) -> None:
        """Exercise the deployed setup without contacting or starting a gateway."""
        with self.stage('framework_setup'):
            source = self.competition / 'ARC-AGI-3-Agents'
            if not (source / 'main.py').is_file():
                raise FileNotFoundError(f'Competition framework is missing: {source}')
            shutil.copytree(source, self.framework, dirs_exist_ok=True)
            shutil.copytree(self.bundle, self.framework / 'agents/templates', dirs_exist_ok=True)
            sandbox = self.working / '.venv_agent_sandbox'
            # The worker needs only stdlib. Avoid ensurepip and bundled pip
            # availability as an unnecessary dependency of isolated startup.
            self.command('sandbox_setup', [sys.executable, '-m', 'venv', '--without-pip',
                                          str(sandbox)], timeout=120)
            os.environ['ARC3_SANDBOX_PYTHON'] = str(sandbox / 'bin/python')
            self.preflight.mkdir(parents=True, exist_ok=True)
            for name, target in [('agent', self.bundle), ('.venv_agent_sandbox', sandbox)]:
                link = self.preflight / name
                if link.is_symlink():
                    if link.resolve() != target.resolve():
                        raise ValueError(f'Preflight symlink points at the wrong directory: {link}')
                else:
                    link.symlink_to(target, target_is_directory=True)
        with self.stage('containment_audit'):
            report_path = self.preflight / 'containment.json'
            # Do not mistake an earlier audit report for this invocation's result.
            report_path.unlink(missing_ok=True)
            try:
                self.command('containment', [sys.executable, str(self.audit_script)], timeout=90)
            finally:
                if report_path.is_file():
                    shutil.copy2(report_path, self.diagnostics / 'containment.json')
            report = json.loads(report_path.read_text())
            if report.get('containment_passed') is not True:
                raise RuntimeError('Containment audit did not pass; see diagnostics/containment.json')
        with self.stage('agent_registration'):
            registry = ('from typing import Type\nfrom dotenv import load_dotenv\n'
                        'from .agent import Agent, Playback\nfrom .swarm import Swarm\n'
                        'from .templates.random_agent import Random\n'
                        'from .templates.my_agent import MyAgent\n\nload_dotenv()\n\n'
                        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {'random': Random, 'myagent': MyAgent}\n")
            (self.framework / 'agents/__init__.py').write_text(registry)
            (self.framework / '.env').write_text(
                'SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\n'
                'ARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=competition\nENVIRONMENTS_DIR=\n'
                f'RECORDINGS_DIR={self.working / "server_recording"}\n')
            self.event('agent_registered', agent='myagent', action_log=os.environ['ARC3_LOG_DIR'],
                       operation_mode='competition',
                       backend='local flashnext' if os.getenv('ARC3_LLM_READY') == '1' else 'exploration')

    def preflight_framework(self) -> None:
        """Validate Kaggle's actual interpreter, containment and framework imports."""
        self.prepare_offline_framework()
        with self.stage('framework_imports'):
            self.command('framework_imports', [sys.executable, '-c',
                         dedent('''\
                         import os
                         os.environ['ENABLE_OFFLINE_BACKEND'] = '0'
                         from pathlib import Path
                         import numpy as np
                         from agents import AVAILABLE_AGENTS
                         from agents.agent import Agent
                         from arcengine import FrameDataRaw, GameState
                         agent_class = AVAILABLE_AGENTS['myagent']
                         assert issubclass(agent_class, Agent)
                         print('FRAMEWORK_IMPORTS_OK', flush=True)
                         # Synthetic public API data only; never instantiate an
                         # Arcade or read environment source during this check.
                         class Fixture:
                             def __init__(self):
                                 self.observation_space = FrameDataRaw(
                                     game_id='runtime-fixture', frame=[np.zeros((4, 4), dtype=np.int8)],
                                     state=GameState.NOT_FINISHED, win_levels=1, available_actions=[1])
                             def step(self, action, data=None, reasoning=None):
                                 self.observation_space.state = GameState.WIN
                                 self.observation_space.levels_completed = 1
                                 return self.observation_space
                         os.environ['ARC3_LOG_DIR'] = str(Path.cwd().parent / 'diagnostics/synthetic-agent')
                         agent = agent_class(card_id='fixture', game_id='runtime-fixture',
                                             agent_name='fixture', ROOT_URL='', record=False,
                                             arc_env=Fixture())
                         agent.main()
                         assert agent.action_counter == 1 and not agent._recovery_mode
                         print('FRAMEWORK_RUNTIME_OK synthetic_actions=1 gateway_actions=0', flush=True)
                         ''')],
                         timeout=120, cwd=self.framework)
        self.event('save_preflight_passed', gateway_contacted=False, gameplay_actions=0)

    def start_llm(self, *, startup_seconds: float = 900.0) -> None:
        with self.stage('llm_startup'):
            spec = importlib.util.spec_from_file_location('arc3_llm_daemon', '/tmp/arc3_llm_daemon.py')
            if spec is None or spec.loader is None:
                raise RuntimeError('Bundled LLM supervisor is unavailable')
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            remaining = self.target_seconds - self.elapsed()
            if remaining <= 0:
                raise TimeoutError('Run budget exhausted before LLM startup')
            self.llm = module.LLMDaemon(Path('/tmp/arc3_llm_launcher.py'), self.working,
                                        self.event, startup_seconds=min(startup_seconds, remaining))
            self.llm.start()

    def stop_llm(self) -> None:
        if self.llm is not None:
            self.llm.stop()

    def monitor(self, process: subprocess.Popen, output) -> None:
        last_heartbeat = -60.0
        try:
            while process.poll() is None:
                elapsed = self.elapsed()
                if elapsed >= self.hard_limit_seconds or elapsed >= self.target_seconds:
                    self.event('runtime_limit', elapsed=elapsed, action='request_scorecard_finalization')
                    break
                if self.verbose:
                    chunk = output.read(65_536)
                    if chunk:
                        print(chunk, end='', flush=True)
                    if elapsed - last_heartbeat >= 60:
                        self.event('framework_heartbeat', pid=process.pid,
                                   submission_exists=self.submission.is_file())
                        last_heartbeat = elapsed
                try:
                    process.wait(timeout=min(1.0, self.target_seconds - elapsed))
                except subprocess.TimeoutExpired:
                    continue
        finally:
            try:
                if process.poll() is None:
                    self.event('cleanup_started', pid=process.pid, signal='SIGINT')
                    try:
                        process.send_signal(signal.SIGINT)
                    except ProcessLookupError:
                        pass
                    remaining = max(0.0, self.hard_limit_seconds - self.elapsed())
                    process.wait(timeout=min(60.0, remaining))
            finally:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait()
                if self.verbose:
                    print(output.read(), end='', flush=True)
                self.event('framework_exited', pid=process.pid, returncode=process.returncode)
        if process.returncode:
            raise RuntimeError(f'Framework exited with code {process.returncode}; see diagnostics/framework.log')

    def run_framework(self) -> None:
        with self.stage('framework_execution'):
            if self.elapsed() >= self.target_seconds:
                raise TimeoutError('Global run target exhausted during setup')
            path = self.diagnostics / 'framework.log'
            with path.open('w', encoding='utf-8') as stream, path.open(errors='replace') as output:
                process = subprocess.Popen(
                    [sys.executable, '-u', 'main.py', '--agent', 'myagent'], cwd=self.framework,
                    env=dict(os.environ, MPLBACKEND='agg', PYTHONUNBUFFERED='1',
                             OPERATION_MODE='competition'),
                    stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
                self.event('framework_started', pid=process.pid, log=str(path))
                self.monitor(process, output)

    def validate_submission(self) -> None:
        with self.stage('submission_validation'):
            # Give the sidecar a bounded interval to finish writing after close.
            deadline = min(self.started + self.hard_limit_seconds,
                           time.time() + self.submission_wait_seconds)
            while self._signature() == self.initial_submission and time.time() < deadline:
                time.sleep(min(0.5, max(0.0, deadline - time.time())))
            if not self.submission.is_file():
                raise FileNotFoundError('Gateway produced no submission.parquet; inspect diagnostics/framework.log')
            if self._signature() == self.initial_submission:
                raise RuntimeError('submission.parquet is unchanged from startup; no fresh gateway result')
            import pyarrow.parquet as pq
            while True:
                try:
                    parquet = pq.ParquetFile(self.submission)
                    break
                except (OSError, ValueError):
                    # The sidecar may have created the file before its footer
                    # was flushed. Retry only inside the shared hard deadline.
                    if time.time() >= deadline:
                        raise
                    self.event('submission_write_pending')
                    time.sleep(min(0.5, max(0.0, deadline - time.time())))
            required = {'row_id', 'game_id', 'end_of_game', 'score'}
            if not required.issubset(parquet.schema_arrow.names) or parquet.metadata.num_rows < 1:
                raise ValueError('Gateway submission has missing columns or no rows')
            if parquet.metadata.num_rows == 1:
                row = parquet.read(columns=sorted(required)).to_pylist()[0]
                if row['row_id'] == '1_0' and row['game_id'] == '1' and row['end_of_game']:
                    raise ValueError('submission.parquet contains the old dummy record; not a gateway result')
            self.event('submission_validated', rows=parquet.metadata.num_rows,
                       bytes=self.submission.stat().st_size, columns=parquet.schema_arrow.names,
                       sha256=hashlib.sha256(self.submission.read_bytes()).hexdigest())
        self.status('COMPLETE', stage='submission_validation', submission_exists=True)

    def run(self) -> None:
        try:
            self._run()
        finally:
            self.stop_llm()

    def _run(self) -> None:
        if not self.rerun:
            if os.getenv('ENABLE_SUBMISSION_BOOTSTRAP', '1') == '1':
                # Create no scores or gameplay records. Kaggle requires a saved
                # output with this name before accepting a code submission.
                # Evaluation must replace it with a fresh, nonempty gateway file.
                import pyarrow as pa
                import pyarrow.parquet as pq
                schema = pa.schema([('row_id', pa.string()), ('game_id', pa.string()),
                                    ('end_of_game', pa.bool_()), ('score', pa.float64())])
                if self.submission.exists():
                    raise FileExistsError('Refusing to overwrite an existing submission during a save run')
                pq.write_table(pa.Table.from_batches([], schema=schema), self.submission)
                self.event('submission_bootstrap', rows=0, evaluated=False,
                           path=str(self.submission), purpose='Kaggle code submission output selector')
            self.status('NOT_EVALUATED', stage='competition_dispatch',
                        reason='Competition rerun flag is not enabled; gameplay gateway is unavailable')
            self.event('not_evaluated', message='Save & Run All does not play hidden games. '
                       'No gameplay rows were generated. The competition gateway runs during Submit to Competition.')
            return
        self.prepare_framework()
        self.run_framework()
        self.validate_submission()

runner = NotebookRunner(build_id='efbe032197ffb0a4')
runner.start()


In [ ]:
runner.install_dependencies()


In [ ]:
%%writefile /tmp/agent_bundle/arclog.py
"""Immutable visual evidence and pre-dispatch perception guards.

Grid/cell internals use (row, column); public coordinates and bounding boxes use
(x, y). Animation belongs to one response's ``frame`` list, never outer history.
The missing upstream FrameView base is adapted here; complete baseline diff,
animation, no-op and transition-evidence modules remain the shared algorithms.
"""
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
from collections import deque
from dataclasses import asdict, dataclass
from functools import cached_property
from pathlib import Path
from typing import Any

from .baseline.animation import collapse_frames, normalize_frames, summarize_animation
from .baseline.frame_diff import compute_frame_diff
from .baseline.grid_display import ARC_COLOR_CHARS, format_grid_ascii
from .baseline.noop_repeat_guard import NoopRepeatGuard, action_signature
from .baseline.transition_evidence import transition_evidence

# Optional accelerators. Equivalent stdlib fallbacks keep the offline notebook
# usable without assuming scientific wheels exist in the competition dataset.
try:
    import numpy as np
    from scipy import ndimage
except ImportError:
    np = None
    ndimage = None
try:
    import networkx as nx
except ImportError:
    nx = None

Grid = tuple[tuple[int, ...], ...]
XY = tuple[int, int]
Signature = tuple[str, str, int]


def enabled(name: str) -> bool:
    return os.getenv(name, "1") == "1"


def auditable() -> bool:
    """Default-on Generation 3 contract; 0 is the archived harness control."""
    return enabled('ENABLE_AUDITABLE_WORLD_MODEL')


def freeze_grid(raw: Any) -> Grid:
    grid: Grid = normalize_frames([raw])[0]
    if grid and (not grid[0] or any(len(row) != len(grid[0]) for row in grid)):
        raise ValueError("grid must be rectangular")
    if len(grid) > 256 or (grid and len(grid[0]) > 256):
        raise ValueError("grid too large")
    if any(not 0 <= cell < 16 for row in grid for cell in row):
        raise ValueError("grid colors must be in [0, 15]")
    return grid


def _object_signature(cells: tuple[XY, ...], color: int) -> tuple[str, int | None, int, str, str]:
    """Port the notebook's translation/rotation canonicalization, color-free pose.

    Exact hash convention matches Cell 3's segmentation hunks. SHA1 is only an
    opaque geometry identifier, never a security or provenance decision.
    """
    r0 = min(r for r, _ in cells)
    c0 = min(c for _, c in cells)
    norm = sorted((r-r0, c-c0) for r, c in cells)
    variants = [norm]
    for _ in range(3):
        max_r = max(r for r, _ in variants[-1])
        variants.append(sorted((c, max_r-r) for r, c in variants[-1]))

    def cell_hash(shape: list[XY], label: str) -> str:
        return hashlib.sha1(repr((label, shape)).encode(), usedforsecurity=False).hexdigest()[:16]

    hashes = [cell_hash(v, ARC_COLOR_CHARS[color]) for v in variants]
    shape_hashes = [cell_hash(v, "__shape__") for v in variants]
    k = shape_hashes.index(min(shape_hashes))
    symmetry = {1: 4, 2: 2, 4: 1}[len(set(shape_hashes))]
    rotation = None if symmetry == 4 else (((4-k) % 4)*90) % (180 if symmetry == 2 else 360)
    return hashes[k], rotation, symmetry, hashes[0], shape_hashes[0]


@dataclass(frozen=True)
class BoardObject:
    object_id: int
    color: int
    bounding_box: tuple[int, int, int, int]
    area: int
    centroid: tuple[float, float]
    shape_hash: str
    adjacency_list: tuple[int, ...]
    cells_rc: tuple[XY, ...]
    canonical_hash: str
    rotation: int | None
    rotational_symmetry: int
    pose_hash: str

    def node(self) -> dict[str, Any]:
        """Compatibility with the archived segmentation/frame-diff contract."""
        return {"id": self.object_id, "color": ARC_COLOR_CHARS[self.color],
                "hash": self.canonical_hash, "shape_hash": self.shape_hash,
                "pose_hash": self.pose_hash, "rotation": self.rotation,
                "rotational_symmetry": self.rotational_symmetry,
                "pixels": self.area, "boundary": self.cells_rc,
                "bbox_rc": (self.bounding_box[1], self.bounding_box[0],
                            self.bounding_box[3], self.bounding_box[2])}


def _components(grid: Grid) -> list[tuple[XY, ...]]:
    if not grid:
        return []
    if ndimage is not None and np is not None:
        array = np.asarray(grid)
        groups = []
        for color in np.unique(array):
            labeled: Any = ndimage.label(array == color)
            labels, count = labeled
            for label in range(1, count+1):
                groups.append(tuple((int(r), int(c)) for r, c in np.argwhere(labels == label)))
        return sorted(groups, key=lambda cells: cells[0])
    seen: set[XY] = set()
    groups = []
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if (r, c) in seen:
                continue
            queue = deque([(r, c)])
            seen.add((r, c))
            cells = []
            while queue:
                rr, cc = queue.popleft()
                cells.append((rr, cc))
                for nr, nc in ((rr-1, cc), (rr+1, cc), (rr, cc-1), (rr, cc+1)):
                    if (0 <= nr < len(grid) and 0 <= nc < len(row)
                            and (nr, nc) not in seen and grid[nr][nc] == color):
                        seen.add((nr, nc))
                        queue.append((nr, nc))
            groups.append(tuple(sorted(cells)))
    return groups


@dataclass(frozen=True)
class FrameView:
    grid: Grid
    level: int = 1
    episode: int = 0
    step: int = 0
    state: str = "NOT_FINISHED"

    @property
    def ascii(self) -> str:
        return format_grid_ascii(self.grid)

    @cached_property
    def segmentation(self) -> tuple[BoardObject, ...]:
        groups = _components(self.grid)
        owner = {cell: i for i, cells in enumerate(groups) for cell in cells}
        edges: set[tuple[int, int]] = set()
        for (r, c), i in owner.items():
            for other in ((r+1, c), (r, c+1)):
                j = owner.get(other)
                if j is not None and j != i:
                    edges.add((min(i, j), max(i, j)))
        graph = nx.Graph() if nx is not None else None
        if graph is not None:
            graph.add_nodes_from(range(len(groups)))
            graph.add_edges_from(edges)
        objects = []
        for i, cells in enumerate(groups):
            rows, cols = zip(*cells)
            color = self.grid[rows[0]][cols[0]]
            signature = _object_signature(cells, color)
            adjacent = (tuple(sorted(graph.neighbors(i))) if graph is not None else
                        tuple(sorted(b if a == i else a for a, b in edges if i in (a, b))))
            objects.append(BoardObject(
                i, color, (min(cols), min(rows), max(cols), max(rows)), len(cells),
                (sum(cols)/len(cells), sum(rows)/len(cells)), signature[4], adjacent,
                cells, signature[0], signature[1], signature[2], signature[3]))
        return tuple(objects)

    @cached_property
    def _owners(self) -> dict[XY, BoardObject]:
        return {cell: obj for obj in self.segmentation for cell in obj.cells_rc}

    def object_at(self, x: int, y: int) -> BoardObject | None:
        return self._owners.get((y, x))


def _cell(grid: Grid, x: int, y: int) -> int | None:
    return grid[y][x] if y < len(grid) and x < len(grid[y]) else None


@dataclass(frozen=True)
class CellChange:
    x: int
    y: int
    old: int | None
    new: int | None


@dataclass(frozen=True)
class AnimationChange:
    frame_index: int
    x: int
    y: int
    old: int | None
    new: int | None


def _diff(before: Grid, after: Grid) -> tuple[CellChange, ...]:
    return tuple(CellChange(x, y, _cell(before, x, y), _cell(after, x, y))
                 for y in range(max(len(before), len(after)))
                 for x in range(max(len(before[y]) if y < len(before) else 0,
                                    len(after[y]) if y < len(after) else 0))
                 if _cell(before, x, y) != _cell(after, x, y))


@dataclass(frozen=True)
class Transition:
    step: int
    action: str
    coordinates: XY | None
    signature: Signature | None
    before: FrameView
    intermediate: tuple[Grid, ...]
    after: FrameView
    raw_settled: tuple[CellChange, ...]
    settled: tuple[CellChange, ...]
    raw_transient: tuple[AnimationChange, ...]
    transient: tuple[AnimationChange, ...]
    hud_mask: frozenset[XY]
    reset: bool
    terminal: bool
    level_changed: bool
    full_reset: bool
    object_diffs_json: str
    evidence_json: str
    remaining_step_budget_estimate: int | None

    @property
    def zero_causal_diff(self) -> bool:
        return not self.settled and not self.transient

    def to_dict(self) -> dict[str, Any]:
        result = asdict(self)
        result['hud_mask'] = sorted(self.hud_mask)
        result['object_diffs'] = json.loads(result.pop('object_diffs_json'))
        result['evidence'] = json.loads(result.pop('evidence_json'))
        result['zero_causal_diff'] = self.zero_causal_diff
        return result


class PerceptionLog:
    """One environment's append-only evidence; no mutable process-global state."""

    def __init__(self, log_path: Path | None = None) -> None:
        self.log_path = log_path
        self.current = FrameView(())
        self._records: list[Transition] = []
        self._blocked: list[str] = []
        self._scope: list[dict[str, Any]] = []
        self._history: list[dict[str, Any]] = []
        self._zeros: dict[Signature, int] = {}
        self._reference = NoopRepeatGuard()
        self.last_action: str | None = None
        self._hud: frozenset[XY] = frozenset()
        self.remaining_step_budget_estimate: int | None = None
        self._journal_hash = hashlib.sha256()

    @property
    def records(self) -> tuple[Transition, ...]:
        return tuple(self._records)

    def history(self) -> tuple[Transition, ...]:
        """Read and authenticate the complete journal before issuing evidence.

        The host owns this read; generated code receives JSON through stdin.
        Missing, edited, extra or truncated transitions cannot authorize a move.
        Non-transition audit events are deliberately kept out of mechanics.
        """
        if self.log_path is None:
            raise ValueError('retrodiction requires persistent log.txt')
        raw = self.log_path.read_bytes()
        if hashlib.sha256(raw).digest() != self._journal_hash.digest():
            raise ValueError('log.txt append-only evidence was modified')
        entries = [json.loads(line) for line in raw.splitlines()]
        logged = [entry for entry in entries if entry.get('type') == 'transition']
        if len(logged) != len(self._records):
            raise ValueError('log.txt transition count differs from immutable history')
        for entry, transition in zip(logged, self._records, strict=True):
            expected = json.loads(json.dumps(transition.to_dict()))
            if any(entry.get(key) != value for key, value in expected.items()):
                raise ValueError(f'log.txt evidence changed at step {transition.step}')
        return self.records

    @property
    def dead_signatures(self) -> tuple[Signature, ...]:
        return tuple(sorted(key for key, count in self._zeros.items() if count >= 2))

    @property
    def blocked_proposals(self) -> tuple[dict[str, Any], ...]:
        return tuple(json.loads(entry) for entry in self._blocked)

    def _emit(self, payload: dict[str, Any]) -> None:
        if self.log_path is not None:
            self.log_path.parent.mkdir(parents=True, exist_ok=True)
            line = json.dumps(payload, separators=(',', ':'))+'\n'
            with self.log_path.open('a', encoding='utf-8') as stream:
                stream.write(line)
            self._journal_hash.update(line.encode())

    @staticmethod
    def _snapshot(frame: FrameView, action: str = '', **flags: Any) -> dict[str, Any]:
        return {'grid': frame.grid, 'step': frame.step, 'level': frame.level,
                'episode': frame.episode, 'action': action, 'result': flags}

    def start(self, grids: Any, *, level: int, state: str = 'NOT_FINISHED') -> None:
        if self._records:
            raise ValueError('cannot restart an existing log')
        frames = normalize_frames(grids)
        self.current = FrameView(freeze_grid(frames[-1]) if frames else (), level, state=state)
        self._scope = [self._snapshot(self.current)]
        self._history = list(self._scope)
        self._reference.note_level(level)
        if auditable():
            self._emit({'type': 'initial_observation', **self._snapshot(self.current)})

    def synchronize(self, grids: Any, *, level: int, state: str) -> None:
        """An out-of-band observation is evidence, never an invented action."""
        frames = normalize_frames(grids)
        before = self.current
        self.current = FrameView(freeze_grid(frames[-1]) if frames else before.grid,
                                 level, before.episode+1, before.step, state)
        self._zeros.clear()
        self._hud = frozenset()
        self.remaining_step_budget_estimate = None
        self._scope = [self._snapshot(self.current)]
        self._history.append(self._scope[0])
        self._emit({'type': 'observation_resync', 'live_action': False,
                    'before': asdict(before), 'after': asdict(self.current)})

    def _state_hash(self) -> str:
        cells = tuple((x, y, color) for y, row in enumerate(self.current.grid)
                      for x, color in enumerate(row) if (x, y) not in self.get_hud_mask())
        return hashlib.sha256(repr(cells).encode()).hexdigest()

    def signature(self, action: str, coordinates: XY | None) -> Signature | None:
        if coordinates is None:
            return None
        obj = self.current.object_at(*coordinates)
        return (action, obj.shape_hash, obj.color) if obj is not None else None

    def is_dead_signature(self, action_type: str, target_shape_hash: str, target_color: int) -> bool:
        return (enabled('ENABLE_DEAD_SIGNATURE')
                and self._zeros.get((action_type, target_shape_hash, target_color), 0) >= 2)

    def blocked_reason(self, action: str, coordinates: XY | None = None,
                       *, expected_step: int | None = None, reference: bool = True) -> str | None:
        if expected_step is not None and expected_step != self.current.step:
            return 'stale_observation'
        if (enabled('ENABLE_DOUBLE_RESET_GUARD') and action == 'RESET'
                and self.last_action == 'RESET'):
            return 'consecutive_reset'
        signature = self.signature(action, coordinates)
        if signature is not None and self.is_dead_signature(*signature):
            return 'dead_signature'
        # Preserve the archived state/position block-then-insist reference path.
        # It is weaker/different than the new hard shape+color filter and retains
        # the parent activation boundary (level >= 2).
        if (reference and not enabled('ENABLE_DEAD_SIGNATURE') and self.current.level >= 2
                and action != 'RESET' and self._reference.should_block(
                    self._state_hash(), self._action_sig(action, coordinates))):
            return 'reference_noop'
        return None

    @staticmethod
    def _action_sig(action: str, coordinates: XY | None) -> str:
        return action_signature({'action': action, 'row': coordinates[1], 'col': coordinates[0]}
                                if coordinates is not None else action)

    def note_blocked(self, action: str, coordinates: XY | None, reason: str) -> None:
        payload = {'type': 'blocked_proposal', 'step': self.current.step,
                   'level': self.current.level, 'episode': self.current.episode,
                   'action': action, 'coordinates': coordinates, 'reason': reason,
                   'live_action': False}
        self._blocked.append(json.dumps(payload))
        self._emit(payload)

    def note_replan(self, reason: str, action: str) -> None:
        self._emit({'type': 'replan', 'step': self.current.step, 'reason': reason,
                    'proposed_action': action, 'live_action': False})

    def note_dispatch_error(self, action: str, coordinates: XY | None, error_type: str) -> None:
        self._emit({'type': 'dispatch_error', 'step': self.current.step,
                    'action': action, 'coordinates': coordinates,
                    'error_type': error_type, 'execution_status': 'unknown'})

    def _detect_hud(self) -> frozenset[XY]:
        self.remaining_step_budget_estimate = None
        if not enabled('ENABLE_HUD_MASK') or len(self._scope) < 4:
            return frozenset()
        history = self._scope[-4:]
        varied_actions = len({f['action'] for f in history[1:]}) >= 2
        if not varied_actions and (not auditable() or not self._hud):
            return frozenset()
        grids: list[Grid] = [f['grid'] for f in history]
        h, w = len(grids[-1]), len(grids[-1][0]) if grids[-1] else 0
        if not h or not w or any((len(g), len(g[0]) if g else 0) != (h, w) for g in grids):
            return frozenset()
        timeline = [grids[0], *(g for f in history[1:]
                                for g in (f.get('animation_grids') or (f['grid'],)))]
        if any((len(g), len(g[0]) if g else 0) != (h, w) for g in timeline):
            return frozenset()
        mask: set[XY] = set()
        estimates = []
        borders = [tuple((x, y) for x in range(w)) for y in sorted({0, h-1})]
        borders += [tuple((x, y) for y in range(h)) for x in sorted({0, w-1})]
        for cells in borders:
            if not varied_actions and not set(cells) <= self._hud:
                continue  # A learned strip still needs monotonic visual evidence.
            stripes = [tuple(g[y][x] for x, y in cells) for g in grids]
            animated = [tuple(g[y][x] for x, y in cells) for g in timeline]
            palette = set().union(*(set(line) for line in animated))
            if len(palette) != 2 or any(sum(a != b for a, b in itertools.pairwise(line)) > 1
                                       for line in stripes):
                continue
            for color in sorted(palette):
                occupied = [{i for i, v in enumerate(line) if v == color} for line in stripes]
                micro_occupied = [{i for i, v in enumerate(line) if v == color} for line in animated]
                if (not all(b < a for a, b in itertools.pairwise(occupied))
                        or not all(b <= a for a, b in itertools.pairwise(micro_occupied))):
                    continue
                mask.update(cells)
                counts = [len(part) for part in occupied]
                deltas = [a-b for a, b in itertools.pairwise(counts)]
                if len(set(deltas)) == 1:
                    estimates.append(math.ceil(counts[-1]/deltas[0]))
        self.remaining_step_budget_estimate = min(estimates) if estimates else None
        return frozenset(mask)

    def record(self, action: str, coordinates: XY | None, grids: Any, *,
               level: int, state: str = 'NOT_FINISHED', full_reset: bool = False,
               level_completed: bool = False) -> Transition:
        """Record a confirmed response. Refusals/failures never create transitions."""
        before = self.current
        frames = tuple(freeze_grid(g) for g in normalize_frames(grids))
        # Empty terminal responses carry no rendered animation; preserve the last
        # board and terminal flags without inventing an observed board effect.
        post = frames[-1] if frames else before.grid
        reset = action == 'RESET'
        terminal = state in ('GAME_OVER', 'WIN')
        level_changed = level != before.level or level_completed
        boundary = reset or terminal or level_changed or full_reset
        after = FrameView(post, level, before.episode + int(boundary), before.step+1, state)
        target = self.signature(action, coordinates)
        snapshot = self._snapshot(after, action, executed=True, game_over=state == 'GAME_OVER',
                                  level_completed=level_changed, run_complete=state == 'WIN')
        snapshot['animation_grids'] = frames
        self._history.append(snapshot)
        self._scope.append(snapshot)
        self._hud = self._detect_hud() if not boundary else frozenset()
        raw_settled = _diff(before.grid, post)
        chain = collapse_frames(before.grid, frames)
        summary = summarize_animation(chain)
        repeated: set[XY] = set()
        events = [AnimationChange(i, d.x, d.y, d.old, d.new)
                  for i, (a, b) in enumerate(itertools.pairwise(chain)) for d in _diff(a, b)]
        seen: set[XY] = set()
        for event in events:
            xy = (event.x, event.y)
            if xy in seen:
                repeated.add(xy)
            seen.add(xy)
        raw_transient = tuple(e for e in events if (e.x, e.y) in repeated
                              or (auditable() and frames[:-1]))
        transient = tuple(e for e in raw_transient if (e.x, e.y) not in self._hud
                          ) if enabled('ENABLE_ANIMATION_DIFF') else ()
        settled = tuple(d for d in raw_settled if (d.x, d.y) not in self._hud)
        same_shape = (len(before.grid), len(before.grid[0]) if before.grid else 0) == (
            len(post), len(post[0]) if post else 0)
        object_diffs: dict[str, Any] = (compute_frame_diff(before.grid, post,
                                          [o.node() for o in before.segmentation],
                                          [o.node() for o in after.segmentation], max_group_match=64)
                        if same_shape else {'status': 'shape_changed'})
        # The archived object diff is raw evidence; explicitly provide a second
        # view in which only the currently inferred HUD delta is neutralized.
        if self._hud and same_shape:
            causal_grid = tuple(tuple(before.grid[y][x] if (x, y) in self._hud else value
                                      for x, value in enumerate(row)) for y, row in enumerate(post))
            causal_view = FrameView(causal_grid)
            object_diffs['causal_board'] = compute_frame_diff(
                before.grid, causal_grid, [o.node() for o in before.segmentation],
                [o.node() for o in causal_view.segmentation], max_group_match=64)
        else:
            object_diffs['causal_board'] = dict(object_diffs)
        evidence = transition_evidence(self._history[-64:], ARC_COLOR_CHARS)
        evidence['animation'] = summary if enabled('ENABLE_ANIMATION_DIFF') else None
        evidence['hud_classification'] = 'provisional monotonic border evidence; not a proven resource'
        evidence['rendered_response'] = bool(frames)
        evidence['causal_effect_status'] = 'observed' if frames and before.grid else 'unavailable'
        record = Transition(after.step, action, coordinates, target, before, frames[:-1], after,
                            raw_settled, settled, raw_transient, transient, self._hud,
                            reset, terminal, level_changed, full_reset,
                            json.dumps(object_diffs), json.dumps(evidence),
                            self.remaining_step_budget_estimate if not boundary else None)
        if not boundary and frames and before.grid:
            if target is not None:
                self._zeros[target] = self._zeros.get(target, 0)+1 if record.zero_causal_diff else 0
            self._reference.observe(self._state_hash(), self._action_sig(action, coordinates),
                                    gameplay_changed=not record.zero_causal_diff)
            self._reference.note_executed(self._state_hash(), self._action_sig(action, coordinates))
        self.last_action = action
        self.current = after
        if boundary:
            self._zeros.clear()
            self._scope = [snapshot]
            self._hud = frozenset()
            self.remaining_step_budget_estimate = None
        self._reference.note_level(level)
        self._records.append(record)
        payload = record.to_dict()
        payload.update(type='transition', live_action=True)
        payload['[DIFF]'] = [f'({d.x}, {d.y}): {d.old} -> {d.new}'
                             for d in (settled if auditable() else raw_settled)]
        payload['[RAW_SETTLED_DIFF]'] = [asdict(d) for d in raw_settled]
        payload['[CAUSAL_BOARD_DIFF]'] = [asdict(d) for d in settled]
        payload['[TRANSIENT_ANIMATION_DIFF]'] = [asdict(e) for e in transient]
        payload['[HUD_STEP_BUDGET_STRIP]'] = sorted(record.hud_mask)
        payload['dead_signatures'] = self.dead_signatures
        self._emit(payload)
        return record

    def get_board_objects(self) -> tuple[BoardObject, ...]:
        return self.current.segmentation

    def get_step_diffs(self, index: int = -1) -> Transition | None:
        return self._records[index] if self._records else None

    def get_hud_mask(self) -> frozenset[XY]:
        return self._hud if enabled('ENABLE_HUD_MASK') else frozenset()


def get_board_objects(source: PerceptionLog | FrameView) -> tuple[BoardObject, ...]:
    return source.get_board_objects() if isinstance(source, PerceptionLog) else source.segmentation


def get_step_diffs(log: PerceptionLog, index: int = -1) -> Transition | None:
    return log.get_step_diffs(index)


def get_hud_mask(log: PerceptionLog) -> frozenset[XY]:
    return log.get_hud_mask()


def is_dead_signature(log: PerceptionLog, action_type: str,
                      target_shape_hash: str, target_color: int) -> bool:
    return log.is_dead_signature(action_type, target_shape_hash, target_color)


In [ ]:
%%writefile /tmp/agent_bundle/baseline/__init__.py
"""Inspectable baseline mechanisms; see manifest.json for provenance and gaps."""


In [ ]:
%%writefile /tmp/agent_bundle/baseline/animation.py
"""Animation awareness.

``arcengine`` renders a frame after every internal ``step()``, so a single
action can come back as a short animation. ``GameState.frame`` is only the last
of those, and the harness consumed nothing else - so every intermediate frame
was discarded before the agent could see it. On some games the whole effect of
an action lives there: the board before and the board after are identical, and
reading the final frame cannot possibly reveal what happened.

Two shapes occur. Where cells change and change BACK, the information is
genuinely hidden and worth spending prompt tokens on. Where every cell changes
at most once, the frames are motion interpolation toward a state the final
frame already shows, and there is nothing to find - those animations are
reported nowhere, because a line about them would cost tokens to say that
nothing is missing.
"""
from __future__ import annotations

import itertools
from collections import defaultdict
from collections.abc import Sequence
from typing import Any

from .grid_display import ARC_COLOR_CHARS, format_grid_ascii

Grid = tuple[tuple[int, ...], ...]

# Retrieval budget.
#
# Why a diff timeline rather than the frames themselves: one 64x64 grid renders
# as roughly 4,100 characters, and a single action can return dozens of frames.
# Whole frames are never affordable against a ~1,024-token tool budget, so
# retrieval collapses identical frames and reports only the cells that changed
# between the rest. A single frame can still be read verbatim, cropped.
# No step limit: the timeline crosses the pipe as data and the model prints
# only what it chooses, so dropping steps served a prompt budget that no longer
# applies. It also cost a real misreading - a step count reported as omitted
# was read as "those steps had no changes", the exact inverse.
ANIMATION_MAX_CELLS_PER_STEP = 24
ANIMATION_MAX_TOTAL_CELLS = 80


def normalize_frames(raw_frames: Any) -> list[Grid]:
    """Engine frame list to hashable, comparable grids."""
    frames: list[Grid] = []
    for frame in raw_frames or ():
        rows = frame.tolist() if hasattr(frame, "tolist") else frame
        frames.append(tuple(tuple(int(cell) for cell in row) for row in rows or ()))
    return frames


def collapse_frames(before: Grid | None, frames: Sequence[Grid]) -> list[Grid]:
    """The pre-action board, then each distinct state the action passed through.

    Runs of identical grids collapse to one entry; non-adjacent repeats stay
    separate, because a grid at positions 2 and 7 with something between them
    is two distinct moments and the timeline should show both transitions.

    ``before`` is prepended and is NOT optional. A cell showing X before the
    action, Y during it and X again at the end arrives from the engine as
    [Y, Y, X] - one transition, which would not register as transient. With the
    board before it is [X, Y, Y, X], which gives two. Without this the very
    case the feature exists for is invisible.
    """
    collapsed: list[Grid] = []
    if before is not None:
        collapsed.append(before)
    for grid in frames:
        if collapsed and collapsed[-1] == grid:
            continue
        collapsed.append(grid)
    return collapsed


def _transient_cells(chain: Sequence[Grid]) -> list[tuple[int, int]]:
    """Cells whose value changes two or more times across the chain.

    A cell that changes once and stays changed is visible in the final frame,
    so the agent loses nothing by never seeing the frames between. A cell that
    changes twice has shown something that is no longer there - a rejected
    click, a consumed attempt, a sprite crossing a background cell. Those are
    exactly the pixels no frame the agent can reach will ever contain.
    """
    changes: defaultdict[tuple[int, int], int] = defaultdict(int)
    for before, after in itertools.pairwise(chain):
        rows = max(len(before), len(after))
        for row in range(rows):
            before_row = before[row] if row < len(before) else ()
            after_row = after[row] if row < len(after) else ()
            for col in range(max(len(before_row), len(after_row))):
                old = before_row[col] if col < len(before_row) else None
                new = after_row[col] if col < len(after_row) else None
                if old != new:
                    changes[(row, col)] += 1
    return sorted(cell for cell, count in changes.items() if count >= 2)


def summarize_animation(chain: Sequence[Grid]) -> dict[str, Any] | None:
    """Compact metadata for one action, or ``None`` if it did not animate.

    ``None`` for the single-frame case means ordinary actions carry no extra
    tokens at all - the key is simply absent from the action result.

    ``frames`` counts engine frames after collapsing and excludes the prepended
    pre-action board, so it answers "how many distinct states did this action
    pass through" rather than describing the internal representation.
    """
    if len(chain) <= 2:
        return None
    transient = _transient_cells(chain)
    summary: dict[str, Any] = {
        "frames": len(chain) - 1,
        "transient_pixels": len(transient),
    }
    if transient:
        rows = [cell[0] for cell in transient]
        cols = [cell[1] for cell in transient]
        # Inclusive [top, left, bottom, right]: enough to point at the region
        # without spending tokens on a coordinate list.
        summary["transient_bbox"] = [min(rows), min(cols), max(rows), max(cols)]
    return summary


def _bbox_phrase(bbox: Sequence[int] | None) -> str:
    """Where the transient cells are, collapsing degenerate ranges.

    A single cell reads as "at (20,30)", a single row or column as a span in
    the other axis only; "rows 20-20, cols 30-30" is noise the model has to
    parse before noticing it means one cell.
    """
    if not bbox:
        return ""
    top, left, bottom, right = bbox
    if top == bottom and left == right:
        return f" at ({top},{left})"
    if top == bottom:
        return f" in row {top}, cols {left}-{right}"
    if left == right:
        return f" in col {left}, rows {top}-{bottom}"
    return f" around rows {top}-{bottom}, cols {left}-{right}"


def describe_animation(
    summary: dict[str, Any] | None,
    *,
    action_display: str | None,
    gameplay_changed: bool | None,
    game_over: bool = False,
    timeline: bool = True,
) -> str:
    """The opener line, or "" when there is nothing worth saying.

    Silent unless cells changed and changed back. Where every cell changed at
    most once the final frame already shows the outcome, and a line saying so
    would spend tokens to report that nothing is missing.
    """
    if not summary:
        return ""
    transient = int(summary.get("transient_pixels") or 0)
    if transient <= 0:
        return ""
    frames = summary.get("frames")
    cells = "1 cell changed and changed back" if transient == 1 else (
        f"{transient} cells changed and changed back"
    )
    where = _bbox_phrase(summary.get("transient_bbox"))
    name = f" (`{action_display}`)" if action_display else ""
    # point at whichever accessor is actually available
    where_to_look = (
        "`last_animation_timeline`"
        if timeline
        else "`last_animation_frames`"
    )

    if game_over:
        # Follows the BAR RULE block, which has just walked the model through
        # history[-3] and history[-2]. This says what those two frames cannot
        # contain, which is precisely what "transient" means.
        return (
            f"That fatal action animated over {frames} frames, and inside it "
            f"{cells}{where}. Those cells are in neither "
            f"`history[-3].frame` nor `history[-2].frame` - read {where_to_look} to see "
            "what happened between them."
        )
    if gameplay_changed is False:
        return (
            f"The last executed action{name} animated over {frames} frames and left the "
            f"board area unchanged, but it was not a no-op: {cells}{where}. "
            "Whatever this action did is visible only there - "
            f"read {where_to_look} before concluding it failed."
        )
    return (
        f"The last executed action{name} animated over {frames} frames. "
        f"{cells[0].upper()}{cells[1:]}{where}. {where_to_look} shows them."
    )


def _color_char(value: int | None) -> str:
    if value is None:
        return "?"
    return ARC_COLOR_CHARS[max(0, min(15, int(value)))]


def _cell_at(grid: Grid, row: int, col: int) -> int | None:
    if row >= len(grid):
        return None
    line = grid[row]
    return line[col] if col < len(line) else None


def _diff_cells(before: Grid, after: Grid) -> list[tuple[int, int, int | None, int | None]]:
    rows = max(len(before), len(after))
    changes: list[tuple[int, int, int | None, int | None]] = []
    for row in range(rows):
        cols = max(
            len(before[row]) if row < len(before) else 0,
            len(after[row]) if row < len(after) else 0,
        )
        for col in range(cols):
            old = _cell_at(before, row, col)
            new = _cell_at(after, row, col)
            if old != new:
                changes.append((row, col, old, new))
    return changes


def _bbox_text(cells: Sequence[tuple[int, int, Any, Any]]) -> str:
    rows = [cell[0] for cell in cells]
    cols = [cell[1] for cell in cells]
    return f"rows {min(rows)}-{max(rows)}, cols {min(cols)}-{max(cols)}"


def _format_changes(
    cells: Sequence[tuple[int, int, int | None, int | None]], budget: int
) -> list[str]:
    """Changed cells grouped by colour transition, one compact line each.

    Strings rather than nested coordinate lists because the caller renders with
    json.dumps(indent=2), which would put every [row, col] pair on four lines.
    """
    grouped: dict[str, list[str]] = {}
    for row, col, old, new in cells[:budget]:
        key = f"{_color_char(old)}>{_color_char(new)}"
        grouped.setdefault(key, []).append(f"({row},{col})")
    lines = [f"{key} @ {' '.join(points)}" for key, points in grouped.items()]
    if len(cells) > budget:
        lines.append(f"... {len(cells) - budget} further changed cells omitted")
    return lines


def build_animation_frames(record: dict[str, Any] | None) -> list[dict[str, Any]]:
    """Every distinct state the action passed through, as FrameView payloads.

    The pre-action board is dropped here: the model already has it as
    `previous_frame`, and including it would make index 0 mean something
    different from the step numbers in the timeline.

    Sends the grid as well as the ascii so `segmentation` works on the far
    side. That doubles the payload, but it crosses a pipe rather than the
    prompt - what reaches the model is only what its own code prints.
    """
    if not record:
        return []
    chain: list[Grid] = list(record.get("chain") or ())
    if len(chain) <= 2:
        return []
    step = record.get("step") or 0
    level = record.get("level") or 0
    return [
        {
            "ascii": format_grid_ascii(grid),
            "step": step,
            "level": level,
            "shape": (len(grid), max((len(row) for row in grid), default=0)),
            "grid": [list(row) for row in grid],
        }
        for grid in chain[1:]
    ]


def build_animation_view(record: dict[str, Any] | None) -> dict[str, Any]:
    """The diff timeline: which cells changed at each step.

    The cheap first look. Reading a frame verbatim is no longer this
    function's job - `last_transition.frames()` hands back FrameView objects
    and the model crops their ascii the same way it crops current_frame, with
    its own helpers rather than a region parameter it has to guess.
    """
    empty = {
        "error": (
            "No animation is available: the last executed action returned a single frame."
        )
    }
    if not record:
        return empty
    chain: list[Grid] = list(record.get("chain") or ())
    if len(chain) <= 2:
        return empty

    header: dict[str, Any] = {
        "action": record.get("action_display"),
        # excludes the prepended pre-action board, matching the metadata
        "frames": len(chain) - 1,
    }

    steps: list[dict[str, Any]] = []
    for position in range(1, len(chain)):
        cells = _diff_cells(chain[position - 1], chain[position])
        if cells:
            steps.append({"_position": position, "_cells": cells})

    rendered: list[dict[str, Any]] = []
    remaining = ANIMATION_MAX_TOTAL_CELLS
    for step in steps:
        cells = step["_cells"]
        entry: dict[str, Any] = {
            "step": step["_position"],
            "changed": len(cells),
            "bbox": _bbox_text(cells),
        }
        budget = min(ANIMATION_MAX_CELLS_PER_STEP, max(0, remaining))
        if len(cells) > ANIMATION_MAX_CELLS_PER_STEP or budget == 0:
            # Too broad to enumerate usefully: a colour-transition census says
            # more per token than an arbitrarily truncated cell list.
            census: dict[str, int] = {}
            for _, _, old, new in cells:
                key = f"{_color_char(old)}>{_color_char(new)}"
                census[key] = census.get(key, 0) + 1
            entry["transitions"] = census
        else:
            entry["changes"] = _format_changes(cells, budget)
            remaining -= min(len(cells), budget)
        rendered.append(entry)

    return {**header, "steps": rendered}


In [ ]:
%%writefile /tmp/agent_bundle/baseline/bounded_search.py
"""Generic unit-cost BFS, also embedded verbatim into the isolated Python tool.

No environment access, puzzle identifiers, transition rules or action execution.
Time limits are cooperative; the sandbox's outer timeout remains authoritative.
"""

from collections import deque as _bfs_deque
from collections.abc import Callable, Iterable
from copy import deepcopy as _bfs_copy
from math import isfinite as _bfs_finite
from time import monotonic as _bfs_clock
from typing import Any


def bounded_bfs(start, successors: Callable[[Any], Iterable[tuple[Any, Any]]], is_goal, *, key=None, max_nodes=50000,
                max_depth=128, timeout_s=2.0, max_edges=250000):
    """Search a model supplied by the caller, never the real environment.

    successors(state) yields (action, next_state). is_goal(state) is a predicate.
    States must be deepcopy-able and hashable, or key(state) must be hashable.
    Keys must distinguish every state variable relevant to transitions and goals.
    Callbacks must be pure. Copies isolate ordinary mutable state/action aliases.

    Result status: found, exhausted, or limit (reason: nodes/edges/depth/time).
    Only found has a list of actions; [] means the start already satisfies goal.
    Exhausted refers only to the fully explored supplied model, NOT the real game.
    Hitting max_depth conservatively reports limit even if its boundary is a leaf.
    Unit-cost shortest paths only; there is no heuristic or weighted-edge mode.
    Valid oversized requests are capped at unchanged hard maxima: 100000 nodes,
    1024 depth, 1000000 edges and 5 seconds. Defaults remain unchanged.
    Results include effective_limits and clamped_limits (names, not input values).
    Zero depth remains valid; other limits must be positive. No type coercion.
    Callback exceptions propagate; no automatic retry or partial plan is returned.
    """
    # Validate types/lower bounds before capping: do not coerce malformed inputs.
    effective_limits = {}
    clamped_limits = []
    for name, value, lower, upper in (
        ('max_nodes', max_nodes, 1, 100000),
        ('max_depth', max_depth, 0, 1024),
        ('max_edges', max_edges, 1, 1000000),
    ):
        if type(value) is not int or value < lower:
            raise ValueError(f'{name} must be an integer >= {lower}')
        effective_limits[name] = min(value, upper)
        if value > upper:
            clamped_limits.append(name)
    if (isinstance(timeout_s, bool) or not isinstance(timeout_s, (int, float))
            or (isinstance(timeout_s, float) and not _bfs_finite(timeout_s))
            or timeout_s <= 0):
        raise ValueError('timeout_s must be finite and greater than 0')
    # Python integers are finite; cap before converting even very large integers.
    effective_limits['timeout_s'] = float(min(timeout_s, 5.0))
    if timeout_s > 5.0:
        clamped_limits.append('timeout_s')
    max_nodes = effective_limits['max_nodes']
    max_depth = effective_limits['max_depth']
    max_edges = effective_limits['max_edges']
    timeout_s = effective_limits['timeout_s']
    if not callable(successors) or not callable(is_goal) or (key is not None and not callable(key)):
        raise TypeError('successors and is_goal must be callable; key must be callable or None')

    began = _bfs_clock()
    deadline = began + timeout_s
    states, parents, incoming, depths = [], [], [], []
    seen = {}
    expanded = 0
    edges = 0
    depth_cut = False

    def timed_out():
        return _bfs_clock() >= deadline

    def result(status, reason=None, node=None):
        actions = None
        depth = None
        if status == 'found':
            assert node is not None
            actions = []
            depth = depths[node]
            while parents[node] != -1:
                actions.append(_bfs_copy(incoming[node]))
                node = parents[node]
            actions.reverse()
            # Reconstruction/copying is charged to the same cooperative budget.
            if timed_out():
                status, reason, actions, depth = 'limit', 'time', None, None
        return {'status': status, 'reason': reason, 'actions': actions,
                'depth': depth, 'expanded': expanded, 'discovered': len(states),
                'edges': edges, 'elapsed_s': _bfs_clock() - began,
                'effective_limits': dict(effective_limits),
                'clamped_limits': list(clamped_limits)}

    def state_key(state):
        value = key(_bfs_copy(state)) if key is not None else state
        try:
            hash(value)
        except TypeError as exc:
            raise TypeError('Unhashable search state/key; provide key(state) with a hashable, complete state identity') from exc
        return _bfs_copy(value)

    root = _bfs_copy(start)
    root_key = state_key(root)
    states.append(root); parents.append(-1); incoming.append(None); depths.append(0)
    seen[root_key] = 0
    if timed_out():
        return result('limit', 'time')
    solved = is_goal(_bfs_copy(root))
    if timed_out():
        return result('limit', 'time')
    if solved:
        return result('found', node=0)
    frontier = _bfs_deque([0])

    while frontier:
        if timed_out():
            return result('limit', 'time')
        node = frontier.popleft()
        if depths[node] >= max_depth:
            depth_cut = True
            continue
        expanded += 1
        iterator = iter(successors(_bfs_copy(states[node])))
        if timed_out():
            return result('limit', 'time')
        while True:
            if timed_out():
                return result('limit', 'time')
            if edges >= max_edges:
                return result('limit', 'edges')
            try:
                edge = next(iterator)
            except StopIteration:
                break
            if timed_out():
                return result('limit', 'time')
            edges += 1
            if not isinstance(edge, (tuple, list)) or len(edge) != 2:
                raise ValueError('successors(state) must yield pairs (action, next_state)')
            action, candidate = edge
            candidate = _bfs_copy(candidate)
            identity = state_key(candidate)
            if timed_out():
                return result('limit', 'time')
            if identity in seen:
                continue
            if len(states) >= max_nodes:
                return result('limit', 'nodes')
            child = len(states)
            # Store parent indices separately: no model-written predecessor tuples.
            states.append(candidate); parents.append(node)
            incoming.append(_bfs_copy(action)); depths.append(depths[node] + 1)
            seen[identity] = child
            solved = is_goal(_bfs_copy(candidate))
            if timed_out():
                return result('limit', 'time')
            if solved:
                return result('found', node=child)
            frontier.append(child)
    return result('limit', 'depth') if depth_cut else result('exhausted')


In [ ]:
%%writefile /tmp/agent_bundle/baseline/checked_execution.py
"""Generic checked execution. Environment access is supplied by the sandbox adapter.

No game identifiers, decoders, transition rules, retries, persistence or search.
The existing outer Python-tool deadline is the hard timeout.
"""
from copy import deepcopy as _exec_copy


def checked_execute(actions, observe, predict, send_one, *, expected_start,
                    max_steps=32):
    """Run a finite concrete plan, checking every nonterminal transition.

    observe() returns a deepcopy-able model state with ordinary equality.
    predict(state, action) returns its predicted successor. Neither may act.
    send_one(action) sends exactly one action via the existing guarded interface.
    The adapter owns callback action blocking and syntax validation of actions.

    A completed plan is NOT a won game. Reports retain confirmed execution even
    on ordinary callback errors. BaseException guards and hard timeouts propagate
    unchanged; the host's action records remain authoritative in those cases.
    """
    if type(actions) not in (list, tuple) or len(actions) > 128:
        raise ValueError('actions must be a finite list/tuple of at most 128 single actions')
    if type(max_steps) is not int or not 1 <= max_steps <= 128:
        raise ValueError('max_steps must be an integer in [1, 128]')
    if not all(callable(f) for f in (observe, predict, send_one)):
        raise TypeError('observe, predict and send_one must be callable')
    report = {'status': None, 'phase': 'prepare', 'attempted': 0,
              'confirmed_executed': 0, 'executed_actions': [],
              'execution_uncertain': False, 'last_action_result': None,
              'expected': None, 'observed': None, 'error': None}

    def finish(status):
        report['status'] = status
        return report

    def failure(status, exc):
        try:
            message = str(exc)[:600]
        except Exception:  # noqa: BLE001 - archived callback/retention failure boundary
            message = '<exception message unavailable>'
        report['error'] = type(exc).__name__ + ': ' + message
        return finish(status)

    try:
        plan = _exec_copy(list(actions))
        expected = _exec_copy(expected_start)
        report['expected'] = _exec_copy(expected)
        if not plan:
            return finish('plan_complete')
        report['phase'] = 'observe_start'
        state = _exec_copy(observe())
        report['observed'] = _exec_copy(state)
        if state != expected:
            return finish('start_mismatch')
    except Exception as exc:  # noqa: BLE001 - archived callback/retention failure boundary
        return failure('callback_error', exc)

    for index, item in enumerate(plan):
        if index >= max_steps:
            report['phase'] = 'step_limit'
            return finish('limit')
        try:
            report['phase'] = 'predict'
            predicted = _exec_copy(predict(_exec_copy(state), _exec_copy(item)))
            report['expected'] = _exec_copy(predicted)
            # Save the exact request before dispatch; copies cannot trigger a
            # post-action bookkeeping failure before execution is recorded.
            sent = _exec_copy(item)
        except Exception as exc:  # noqa: BLE001 - archived callback/retention failure boundary
            return failure('callback_error', exc)
        report['phase'] = 'action'
        report['attempted'] += 1
        report['observed'] = None
        report['last_action_result'] = None
        try:
            reply = send_one(sent)
        except Exception as exc:  # noqa: BLE001 - archived callback/retention failure boundary
            # No retry: transport/refresh errors can happen after a real action.
            report['execution_uncertain'] = True
            return failure('action_error', exc)
        if not isinstance(reply, dict):
            report['execution_uncertain'] = True
            return failure('action_error', TypeError('action result must be a dict'))
        report['last_action_result'] = reply
        count = reply.get('executed')
        confirmed = count is True or (type(count) is int and count == 1)
        if confirmed:
            report['confirmed_executed'] += 1
            report['executed_actions'].append(item)
        elif count is not False and not (type(count) is int and count == 0):
            report['execution_uncertain'] = True
        # Terminal flags MUST precede old-level decoding and comparison.
        report['phase'] = 'action_result'
        if reply.get('run_complete') or reply.get('done'):
            return finish('run_complete')
        if reply.get('game_over'):
            return finish('game_over')
        if reply.get('level_completed'):
            return finish('level_completed')
        if report['execution_uncertain']:
            return finish('action_error')
        if not confirmed:
            return finish('not_executed')
        if reply.get('no_op') or reply.get('gameplay_changed') is False:
            return finish('no_op')
        if reply.get('stop_reason'):
            return finish('stopped')
        try:
            report['phase'] = 'observe_after'
            observed = _exec_copy(observe())
            report['observed'] = _exec_copy(observed)
            if observed != predicted:
                return finish('mismatch')
            state = observed
        except Exception as exc:  # noqa: BLE001 - archived callback/retention failure boundary
            return failure('callback_error', exc)
    report['phase'] = 'complete'
    return finish('plan_complete')


In [ ]:
%%writefile /tmp/agent_bundle/baseline/frame_diff.py
"""Object-level frame diff core, shared by the sandbox runtime and the host.

This module must stay dependency-free (stdlib only, no imports): its source is
injected verbatim into the sandbox bootstrap template (see
python_tool_sandbox.py), which runs under ``python -I -S`` and cannot import
project modules. The host imports it normally for the auto frame-diff feature.
"""


def _node_topleft(node):
    points = node.get("boundary") or []
    if not points:
        return (0, 0)
    return (min(p[0] for p in points), min(p[1] for p in points))

def _hungarian(cost):
    # Minimum-cost assignment (Jonker-Volgonant style with potentials).
    # cost: n x m matrix with n <= m. Returns list row -> column.
    n = len(cost)
    m = len(cost[0]) if n else 0
    INF = float("inf")
    u = [0.0] * (n + 1)
    v = [0.0] * (m + 1)
    p = [0] * (m + 1)      # p[j] = row assigned to column j (1-based; 0 = none)
    way = [0] * (m + 1)
    for i in range(1, n + 1):
        p[0] = i
        j0 = 0
        minv = [INF] * (m + 1)
        used = [False] * (m + 1)
        while True:
            used[j0] = True
            i0 = p[j0]
            delta = INF
            j1 = 0
            for j in range(1, m + 1):
                if used[j]:
                    continue
                cur = cost[i0 - 1][j - 1] - u[i0] - v[j]
                if cur < minv[j]:
                    minv[j] = cur
                    way[j] = j0
                if minv[j] < delta:
                    delta = minv[j]
                    j1 = j
            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while True:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break
    assignment = [-1] * n
    for j in range(1, m + 1):
        if p[j]:
            assignment[p[j] - 1] = j - 1
    return assignment

def _node_bbox(node):
    points = node.get("boundary") or []
    if not points:
        return (0, 0, 0, 0)
    rows = [p[0] for p in points]; cols = [p[1] for p in points]
    return (min(rows), min(cols), max(rows), max(cols))

def _node_center2(node):
    # bbox center in half-cell units (ints): (r0+r1, c0+c1)
    points = node.get("boundary") or []
    if not points:
        return (0, 0)
    rows = [p[0] for p in points]; cols = [p[1] for p in points]
    return (min(rows) + max(rows), min(cols) + max(cols))

def compute_frame_diff(grid_a, grid_b, nodes_a, nodes_b, max_group_match=None):
    # Object-level diff between two same-shaped grids with precomputed
    # segmentation nodes. Returns only what changed; unchanged objects are
    # omitted. When max_group_match is set, identical-hash groups larger than
    # it skip pairwise matching and report plain appeared/disappeared entries
    # (guards against O(n^3) assignment on e.g. hundreds of identical pixels).
    changed = set()
    for r, (row_a, row_b) in enumerate(zip(grid_a, grid_b)):
        if row_a == row_b:
            continue
        for c, (va, vb) in enumerate(zip(row_a, row_b)):
            if va != vb:
                changed.add((r, c))
    if not changed:
        return {"changed_cell_count": 0, "moved": [], "rotated": [],
                "appeared": [], "disappeared": [],
                "changed_color": [], "resized": []}

    def touched_nodes(nodes):
        touched = []
        for node in nodes:
            points = node.get("boundary") or []
            if not points:
                continue
            r0 = min(p[0] for p in points); r1 = max(p[0] for p in points)
            c0 = min(p[1] for p in points); c1 = max(p[1] for p in points)
            # dilate by 1 so objects that shrank/grew still register as
            # touched by the cells they vacated/claimed
            if any(r0 - 1 <= r <= r1 + 1 and c0 - 1 <= c <= c1 + 1 for (r, c) in changed):
                touched.append(node)
        return touched

    before_nodes = touched_nodes(nodes_a)
    after_nodes = touched_nodes(nodes_b)

    moved, appeared, disappeared = [], [], []
    rotated = []

    def _classify_match(node_a, node_b):
        rot_a = node_a.get("rotation")
        rot_b = node_b.get("rotation")
        symmetry = node_a.get("rotational_symmetry") or 1
        rotated_by = None
        if rot_a is not None and rot_b is not None:
            modulus = 180 if symmetry == 2 else 360
            delta = (rot_b - rot_a) % modulus
            if delta:
                rotated_by = delta
        pa, pb = _node_topleft(node_a), _node_topleft(node_b)
        base_entry = {"color": node_a.get("color"),
                      "pixels": node_a.get("pixels"),
                      "hash": node_a.get("hash")}
        if rotated_by is not None:
            ca, cb = _node_center2(node_a), _node_center2(node_b)
            in_place = (abs(ca[0] - cb[0]) <= 2 and abs(ca[1] - cb[1]) <= 2)
            base_entry["rotated_by"] = rotated_by
            if in_place:
                base_entry["at"] = list(pb)
                rotated.append(base_entry)
            else:
                base_entry["from"] = list(pa); base_entry["to"] = list(pb)
                moved.append(base_entry)
        elif pa != pb:
            base_entry["from"] = list(pa); base_entry["to"] = list(pb)
            moved.append(base_entry)
        # same hash, same position, same rotation: unchanged -> omit

    groups_before = {}
    for node in before_nodes:
        groups_before.setdefault(node.get("hash"), []).append(node)
    groups_after = {}
    for node in after_nodes:
        groups_after.setdefault(node.get("hash"), []).append(node)

    for key, group_a in groups_before.items():
        group_b = groups_after.pop(key, [])
        if (
            max_group_match is not None
            and max(len(group_a), len(group_b)) > max_group_match
        ):
            # too many identical objects to match affordably
            for node_a in group_a:
                disappeared.append({"color": node_a.get("color"),
                                    "pixels": node_a.get("pixels"),
                                    "hash": node_a.get("hash"),
                                    "shape_hash": node_a.get("shape_hash"),
                                    "at": list(_node_topleft(node_a)),
                                    "bbox": list(_node_bbox(node_a))})
            for node_b in group_b:
                appeared.append({"color": node_b.get("color"),
                                 "pixels": node_b.get("pixels"),
                                 "hash": node_b.get("hash"),
                                 "shape_hash": node_b.get("shape_hash"),
                                 "at": list(_node_topleft(node_b)),
                                 "bbox": list(_node_bbox(node_b))})
            continue
        if not group_b:
            for node_a in group_a:
                disappeared.append({"color": node_a.get("color"),
                                    "pixels": node_a.get("pixels"),
                                    "hash": node_a.get("hash"),
                                    "shape_hash": node_a.get("shape_hash"),
                                    "at": list(_node_topleft(node_a)),
                                    "bbox": list(_node_bbox(node_a))})
            continue
        # optimal assignment over bbox-center distances; rows = smaller side
        transposed = len(group_a) > len(group_b)
        rows, cols = (group_b, group_a) if transposed else (group_a, group_b)
        cost = []
        for row_node in rows:
            cr = _node_center2(row_node)
            cost.append([
                abs(cr[0] - _node_center2(col_node)[0])
                + abs(cr[1] - _node_center2(col_node)[1])
                for col_node in cols
            ])
        assignment = _hungarian(cost)
        matched_cols = set()
        for row_index, col_index in enumerate(assignment):
            if col_index < 0:
                continue
            matched_cols.add(col_index)
            if transposed:
                _classify_match(cols[col_index], rows[row_index])
            else:
                _classify_match(rows[row_index], cols[col_index])
        if transposed:
            for col_index, node_a in enumerate(cols):
                if col_index not in matched_cols:
                    disappeared.append({"color": node_a.get("color"),
                                        "pixels": node_a.get("pixels"),
                                        "hash": node_a.get("hash"),
                                    "shape_hash": node_a.get("shape_hash"),
                                        "at": list(_node_topleft(node_a)),
                                        "bbox": list(_node_bbox(node_a))})
        else:
            for col_index, node_b in enumerate(cols):
                if col_index not in matched_cols:
                    appeared.append({"color": node_b.get("color"),
                                     "pixels": node_b.get("pixels"),
                                     "hash": node_b.get("hash"),
                             "shape_hash": node_b.get("shape_hash"),
                                     "at": list(_node_topleft(node_b)),
                                     "bbox": list(_node_bbox(node_b))})
            continue
        # transposed: every after-node (a row) got matched; nothing appeared
    for group_b in groups_after.values():
        for node_b in group_b:
            appeared.append({"color": node_b.get("color"),
                             "pixels": node_b.get("pixels"),
                             "hash": node_b.get("hash"),
                             "shape_hash": node_b.get("shape_hash"),
                             "at": list(_node_topleft(node_b)),
                             "bbox": list(_node_bbox(node_b))})
    # Phase A - changed_color: EXACT pairs only. Same position, byte-
    # identical shape (color-independent pose signature), different
    # color. This asserts "these exact cells changed color" and nothing
    # more; fuzzy transformation claims are deliberately not made.
    changed_color = []
    used_gone, used_new = set(), set()
    for gi, gone in enumerate(disappeared):
        for ni, new in enumerate(appeared):
            if ni in used_new:
                continue
            if (gone["at"] == new["at"]
                    and gone.get("shape_hash")
                    and gone.get("shape_hash") == new.get("shape_hash")
                    and gone["color"] != new["color"]):
                changed_color.append({"at": gone["at"],
                                      "pixels": gone["pixels"],
                                      "before_color": gone["color"],
                                      "after_color": new["color"],
                                      "before_hash": gone.get("hash"),
                                      "after_hash": new.get("hash")})
                used_gone.add(gi); used_new.add(ni)
                break
    disappeared = [entry for index, entry in enumerate(disappeared) if index not in used_gone]
    appeared = [entry for index, entry in enumerate(appeared) if index not in used_new]

    # Phase B - resized: same-color pairs whose bboxes substantially
    # overlap (overlap area / smaller bbox area >= 0.5); greedy
    # one-to-one by descending overlap then size similarity.
    resized = []

    def _bbox_area(b):
        return max(0, b[2] - b[0] + 1) * max(0, b[3] - b[1] + 1)

    def _overlap_ratio(b1, b2):
        inter_r = min(b1[2], b2[2]) - max(b1[0], b2[0]) + 1
        inter_c = min(b1[3], b2[3]) - max(b1[1], b2[1]) + 1
        if inter_r <= 0 or inter_c <= 0:
            return 0.0
        smaller = min(_bbox_area(b1), _bbox_area(b2))
        return (inter_r * inter_c) / smaller if smaller else 0.0

    candidates = []
    for gi, gone in enumerate(disappeared):
        for ni, new in enumerate(appeared):
            if gone["color"] != new["color"]:
                continue
            bg = gone.get("bbox") or (0, 0, -1, -1)
            bn = new.get("bbox") or (0, 0, -1, -1)
            ratio = _overlap_ratio(bg, bn)
            if ratio >= 0.5:
                area_g, area_n = _bbox_area(bg), _bbox_area(bn)
                similarity = (min(area_g, area_n) / max(area_g, area_n)) if max(area_g, area_n) else 0.0
                candidates.append((ratio, similarity, gi, ni))
    candidates.sort(key=lambda item: (-item[0], -item[1]))
    used_gone, used_new = set(), set()
    for ratio, similarity, gi, ni in candidates:
        if gi in used_gone or ni in used_new:
            continue
        used_gone.add(gi); used_new.add(ni)
        gone, new = disappeared[gi], appeared[ni]
        resized.append({"at_before": gone["at"], "at_after": new["at"],
                        "color": gone["color"],
                        "before_pixels": gone["pixels"],
                        "after_pixels": new["pixels"],
                        "before_hash": gone.get("hash"),
                        "after_hash": new.get("hash")})
    disappeared = [entry for index, entry in enumerate(disappeared) if index not in used_gone]
    appeared = [entry for index, entry in enumerate(appeared) if index not in used_new]
    return {"changed_cell_count": len(changed), "moved": moved,
            "rotated": rotated,
            "appeared": appeared, "disappeared": disappeared,
            "changed_color": changed_color, "resized": resized}


In [ ]:
%%writefile /tmp/agent_bundle/baseline/grid_coverage.py
"""Bounded repeated-rectangle lattice evidence and exact pixel crop extraction.
No game IDs, palette roles, terrain rules, paths, fixed origins or tile sizes.
Pitch-equals-component-size is a hypothesis, not a universal board detector.
Dependency-free so this module can be injected into the isolated Python tool.
"""


def _gc_grid(grid):
    if type(grid) not in (list, tuple) or not grid or len(grid) > 256:
        return None
    if type(grid[0]) not in (list, tuple) or not grid[0] or len(grid[0]) > 256:
        return None
    w = len(grid[0])
    if len(grid)*w > 8192 or any(type(row) not in (list, tuple) or len(row) != w or any(type(v) is not int or not 0 <= v < 16 for v in row) for row in grid):
        return None
    return grid


def _gc_components(grid):
    h,w = len(grid),len(grid[0])
    seen = [[False]*w for _ in range(h)]
    nodes = []
    for r in range(h):
        for c in range(w):
            if seen[r][c]:
                continue
            if len(nodes) >= 1024:
                return None
            queue = [(r,c)];seen[r][c] = True
            r0=r1=r;c0=c1=c
            for rr,cc in queue:
                r0=min(r0,rr);r1=max(r1,rr);c0=min(c0,cc);c1=max(c1,cc)
                for nr,nc in ((rr-1,cc),(rr+1,cc),(rr,cc-1),(rr,cc+1)):
                    if 0<=nr<h and 0<=nc<w and not seen[nr][nc] and grid[nr][nc]==grid[r][c]:
                        seen[nr][nc]=True;queue.append((nr,nc))
            nodes.append({'id':len(nodes),'bbox_rc':[r0,c0,r1,c1],'pixels':len(queue)})
    return nodes


def grid_coverage_candidates(grid):
    """Find repeated component-box candidates. Extents are inclusive.

At least four solid rectangles, >=2 distinct row AND column starts, and a
common size/residue are required. Non-solid same-sized aligned components can
extend the supported extent, but do not supply the initial repetition evidence.
Singleton markers never define a pitch. Thin HUD lines are not assigned a role.
Multiple aligned regions may share a candidate: grouping remains unresolved.
"""
    result = {'status':'ok','candidates':[],'candidates_truncated':False,
              'limits':{'cells':8192,'dimension':256,'components':1024,'candidates':8},
              'coverage':'observed repeated-component evidence only, not proof of complete board bounds'}
    g = _gc_grid(grid)
    if g is None:
        result.update(status='unavailable',reason='invalid_or_oversized_grid');return result
    nodes = _gc_components(g)
    if nodes is None:
        result.update(status='unavailable',reason='component_limit');return result
    groups = {}
    for n in nodes:
        r0,c0,r1,c1=n['bbox_rc'];ph,pw=r1-r0+1,c1-c0+1
        if ph<2 or pw<2:
            continue
        key=(ph,pw,r0%ph,c0%pw)
        groups.setdefault(key,[]).append(n)
    candidates = []
    for (ph,pw,rr,cc),group in groups.items():
        solid=[n for n in group if n['pixels']==ph*pw]
        if len(solid)<4 or len({n['bbox_rc'][0] for n in solid})<2 or len({n['bbox_rc'][1] for n in solid})<2:
            continue
        r0=min(n['bbox_rc'][0] for n in group);c0=min(n['bbox_rc'][1] for n in group)
        r1=max(n['bbox_rc'][2] for n in group);c1=max(n['bbox_rc'][3] for n in group)
        shape=[(r1-r0+1)//ph,(c1-c0+1)//pw]
        partial=[]
        # Nearby smaller components at an aligned start can indicate clipped or
        # partial edge cells. Report, don't extend a rectangle through them.
        for n in nodes:
            a,b,c,d=n['bbox_rc'];nh,nw=c-a+1,d-b+1
            outside=a<r0 or b<c0 or c>r1 or d>c1
            near=r0-ph<=a<=r1+ph and c0-pw<=b<=c1+pw
            if outside and near and nh<=ph and nw<=pw and (nh<ph or nw<pw) and a%ph==rr and b%pw==cc:
                partial.append(n['id'])
        candidates.append({'origin_rc':[r0,c0],'pitch_rc':[ph,pw],'shape_rc':shape,
            'bbox_rc':[r0,c0,r1,c1],'solid_support':len(solid),'box_support':len(group),
            'support_component_ids':[n['id'] for n in group],
            'nearby_partial_component_ids':partial[:8],'nearby_partial_count':len(partial),
            'interpretation':'candidate only; pitch equals component size; separate aligned regions may be grouped; roles/identity unresolved'})
    candidates.sort(key=lambda x:(-x['solid_support'],-x['box_support'],x['pitch_rc'],x['origin_rc']))
    result['candidates_truncated']=len(candidates)>8
    result['candidates']=candidates[:8]
    return result


def _gc_pair(value, name, positive=False):
    if type(value) not in (list,tuple) or len(value)!=2 or any(type(x) is not int for x in value):
        raise TypeError(name+' must be an integer row/column pair; no rounding')
    if any(x < (1 if positive else 0) for x in value):
        raise ValueError(name+' is outside the supported range')
    return tuple(value)


def extract_grid_view(grid, colors, *, candidate=None, origin_rc=None, pitch_rc=None, shape_rc=None, topology=None):
    """Return requested raw tiles/samples AND a geometric coverage audit.

Never silently expand/clip the requested crop, classify walkability, infer a
hazard model, or certify an unreachable result. Explicit candidate selection is
required if multiple interpretations exist. Manual geometry works without a
candidate, but its coverage is then unverified. Geometry is re-derived from raw
pixels, not from the caller's mutable cached candidate dictionary.
"""
    g=_gc_grid(grid)
    if g is None:
        raise ValueError('grid view requires a valid grid within the documented bounds')
    report=grid_coverage_candidates(g)
    choices=report['candidates']
    manual=any(x is not None for x in (origin_rc,pitch_rc,shape_rc))
    if candidate is not None and manual:
        raise ValueError('choose a candidate OR supply all three geometry pairs')
    if not manual:
        if report['status']!='ok' or not choices:
            raise ValueError('no supported candidate; supply explicit geometry and treat coverage as unverified')
        if candidate is None:
            if len(choices)!=1 or report['candidates_truncated']:
                raise ValueError('ambiguous candidates; select candidate=index or supply explicit geometry')
            candidate=0
        if type(candidate) is not int or not 0<=candidate<len(choices):
            raise ValueError('candidate must be an in-range integer index')
        chosen=choices[candidate]
        origin_rc,pitch_rc,shape_rc=(chosen[k] for k in ('origin_rc','pitch_rc','shape_rc'))
    elif any(x is None for x in (origin_rc,pitch_rc,shape_rc)):
        raise ValueError('explicit geometry requires origin_rc, pitch_rc AND shape_rc')
    r0,c0=_gc_pair(origin_rc,'origin_rc');ph,pw=_gc_pair(pitch_rc,'pitch_rc',True);nr,nc=_gc_pair(shape_rc,'shape_rc',True)
    r1,c1=r0+nr*ph-1,c0+nc*pw-1
    if r1>=len(g) or c1>=len(g[0]):
        raise ValueError('requested full tiles extend outside the frame; no silent clipping')
    matching=[x for x in choices if x['pitch_rc']==[ph,pw] and (r0-x['origin_rc'][0])%ph==0 and (c0-x['origin_rc'][1])%pw==0]
    omissions=[]
    for x in matching:
        a,b,c,d=x['bbox_rc']
        if a<r0 or b<c0 or c>r1 or d>c1:
            omissions.append({'observed_bbox_rc':x['bbox_rc'],'suggested_origin_rc':x['origin_rc'],
                              'suggested_shape_rc':x['shape_rc'],
                              'outside_sides':[name for name,test in (('top',a<r0),('left',b<c0),('bottom',c>r1),('right',d>c1)) if test]})
    coverage='incomplete_observed_extent' if omissions else ('covers_observed_extent' if matching else 'unverified')
    tiles=[];samples=[];mixed=[];color_cells={};color_counts=[]
    # Even-sized tiles have no central pixel. Explicit convention: upper/left
    # of the central pair. Preserve the full tile, never synthesize a color.
    sr,sc=(ph-1)//2,(pw-1)//2
    for r in range(nr):
        row=[];srow=[];counts_row=[]
        for c in range(nc):
            a,b=r0+r*ph,c0+c*pw
            tile=[''.join(colors[v] for v in line[b:b+pw]) for line in g[a:a+ph]]
            row.append(tile);srow.append(colors[g[a+sr][b+sc]])
            counts={}
            for line in tile:
                for color in line:counts[color]=counts.get(color,0)+1
            for color in sorted(counts):color_cells.setdefault(color,[]).append([r,c])
            counts_row.append(counts)
            if len(counts)>1:mixed.append([r,c])
        tiles.append(row);samples.append(srow);color_counts.append(counts_row)
    view = {'origin_rc':[r0,c0],'pitch_rc':[ph,pw],'shape_rc':[nr,nc],
            'bbox_rc':[r0,c0,r1,c1],'sample_offset_rc':[sr,sc],
            'samples':samples,'tiles':tiles,'mixed_cells_rc':mixed,
            'sample_rows':[''.join(row) for row in samples],
            'color_cells':color_cells,'color_counts':color_counts,
            'schema':{'samples':'[r][c] single observed color',
                      'sample_rows':'[r] string of sample colors',
                      'tiles':'[r][c][pixel_row] string of raw pixels; NOT a color list',
                      'color_cells':'color -> distinct [r,c] tiles containing that pixel color',
                      'color_counts':'[r][c] -> color: pixel count'},
            'coverage':coverage,'omitted_extent_evidence':omissions,
            'candidate_report_status':report['status'],'candidate_report_truncated':report['candidates_truncated'],
            'nearby_partial_count':sum(x['nearby_partial_count'] for x in matching),
            'meaning':'samples are observed pixel colors, not semantic terrain; coverage is not proof of complete board or safe reachability'}

    view['topology'] = None if topology is None else _gc_topology(view,topology,colors)
    return view


def grid_coverage_note(report):
    if report['status']!='ok' or not report['candidates']:
        return ''
    parts=[]
    for i,x in enumerate(report['candidates'][:2]):
        parts.append(str(i)+': origin='+str(x['origin_rc'])+' pitch='+str(x['pitch_rc'])+' rows,cols='+str(x['shape_rc'])+' inclusive_bbox='+str(x['bbox_rc']))
    return ('[grid_coverage] '+'; '.join(parts)+
        '; candidate extents, not proven board bounds. current_frame.grid_view() audits crop coverage; include observed last rows/columns before concluding unreachable.')[:650]


def grid_view_warning(view):
    if view['coverage']=='incomplete_observed_extent':
        return '[grid_coverage] requested crop omits aligned observed extent: '+str(view['omitted_extent_evidence'][:2])[:420]+'; do not conclude unreachable from this crop alone. No crop expansion or action performed.'
    if view['coverage']=='unverified':
        return '[grid_coverage] no matching repeated-component extent; requested samples returned, coverage UNVERIFIED. No completeness/safety claim.'
    if view['nearby_partial_count'] or view['candidate_report_truncated']:
        return '[grid_coverage] covers selected observed extent, but nearby partial components or truncated candidates leave bounds unresolved.'
    return ''


def _gc_topology(view, spec, colors):
    """Static graph under EXPLICIT caller color/stride hypotheses, never safety.

Node samples select candidate floor; pixel presence selects occupied nodes.
An occupied node's terrain remains unknown. Intermediate connector samples
must match the caller's connector colors. No diagonals, wraps or inferred roles.
"""
    keys={'node_colors','occupied_colors','connector_colors','stride_rc','start_rc','goal_rc'}
    if type(spec) is not dict or any(type(k) is not str for k in spec) or set(spec)!=keys:
        raise ValueError('topology requires exactly node_colors, occupied_colors, connector_colors, stride_rc, start_rc, goal_rc')
    def palette(name):
        xs=spec[name]
        if type(xs) not in (list,tuple,str) or len(xs)>16:
            raise TypeError(name+' must be at most 16 single-character colors')
        if any(type(x) is not str or len(x)!=1 or x not in colors for x in xs):
            raise ValueError(name+' contains an invalid color')
        return set(xs)
    floor=palette('node_colors');occupied=palette('occupied_colors');connector=palette('connector_colors')
    stride=_gc_pair(spec['stride_rc'],'stride_rc',True)
    if max(stride)>256:raise ValueError('stride exceeds supported dimension')
    start=_gc_pair(spec['start_rc'],'start_rc');goal=_gc_pair(spec['goal_rc'],'goal_rc')
    nr,nc=view['shape_rc']
    if any(r>=nr or c>=nc for r,c in (start,goal)):raise ValueError('endpoint outside requested view')
    occupied_rc={tuple(rc) for color in occupied for rc in view['color_cells'].get(color,[])}
    nodes=sorted({(r,c) for r,row in enumerate(view['samples']) for c,color in enumerate(row) if color in floor}|occupied_rc)
    index={rc:i for i,rc in enumerate(nodes)};neighbors=[[] for rc in nodes]
    for i,(r,c) in enumerate(nodes):
        for dr,dc in ((-stride[0],0),(stride[0],0),(0,-stride[1]),(0,stride[1])):
            dest=(r+dr,c+dc)
            if dest not in index:continue
            steps=abs(dr)+abs(dc);rr=(dr>0)-(dr<0);cc=(dc>0)-(dc<0)
            if all(view['samples'][r+k*rr][c+k*cc] in connector for k in range(1,steps)):
                neighbors[i].append(index[dest])
    missing=[name for name,rc in (('start',start),('goal',goal)) if rc not in index]
    connected=None
    if not missing:
        seen={index[start]};queue=[index[start]]
        for i in queue:
            for j in neighbors[i]:
                if j not in seen:seen.add(j);queue.append(j)
        connected=index[goal] in seen
    return {'status':'missing_endpoint' if missing else ('connected' if connected else 'disconnected'),
            'nodes_rc':[list(rc) for rc in nodes],'neighbors':neighbors,
            'start_index':index.get(start),'goal_index':index.get(goal),
            'missing_endpoints':missing,'connected':connected,
            'occupied_rc':[list(rc) for rc in sorted(occupied_rc)],
            'underlay_unknown_rc':[list(rc) for rc in sorted(occupied_rc)],
            'hypothesis':{'node_colors':sorted(floor),'occupied_colors':sorted(occupied),
                          'connector_colors':sorted(connector),'stride_rc':list(stride)},
            'meaning':'caller-hypothesis static topology ONLY; occupied nodes retained, underlying terrain unknown; no hazard, collision, resource, route-safety or complete-board certification'}


def grid_contract_note(view):
    note='[grid_contract] sample_rows[r] is a string; samples[r][c] is one pixel color. color_cells[color] lists tiles with that color; color_counts[r][c] counts pixels. tiles[r][c] is a LIST OF ROW STRINGS, not colors.'
    t=view['topology']
    if t is not None:
        note+=' Topology='+t['status']+'; nodes='+str(len(t['nodes_rc']))+'; missing='+str(t['missing_endpoints'])+'.'
    return note+' Occupancy is not terrain; static connectivity is NOT route safety.'


In [ ]:
%%writefile /tmp/agent_bundle/baseline/grid_display.py
"""Local display adapter for the archived animation helper's missing dependency.

Hexadecimal display is a port convention; it is not a recovered upstream palette.
"""
from collections.abc import Sequence

ARC_COLOR_CHARS = "0123456789ABCDEF"


def format_grid_ascii(grid: Sequence[Sequence[int]]) -> str:
    return "\n".join("".join(ARC_COLOR_CHARS[max(0, min(15, int(c)))] for c in row) for row in grid)


In [ ]:
%%writefile /tmp/agent_bundle/baseline/noop_repeat_guard.py
"""Known-no-op repeat guard.

Records ``(level, interior_state_hash, action_signature)`` triples that were
observed to change nothing outside the board border, and refuses to execute
them again in a bit-identical state.

Two deliberate choices:

* The state key uses ``_interior_state_hash`` - the same border-cropped hash
  the death ledger uses - so "the same state" means the same thing everywhere
  in the harness, and a ticking HUD timer does not make every state unique.

* Blocking is overridable. A blocked action arms a single slot; if the model
  immediately re-issues that same action, it executes. This avoids the
  self-sealing failure of a hard block: once an action is on the list, a hard
  block guarantees no contradicting evidence can ever arrive, so a mechanic
  gated on hidden state (a counter, a cooldown, an off-screen timer) would be
  foreclosed forever. Each insistence costs exactly one action and the entry
  survives, so the model can always insist again.

Known limitation: the harness does not receive per-action frame counts, so an
action whose effect appears only in intermediate animation frames (a rejected
click, a bounce) looks identical to one that did nothing, and will be
recorded. The override path is the only recourse there, which is why this
guard defaults to off.
"""
from __future__ import annotations

import re
from collections import OrderedDict
from collections.abc import Callable
from typing import Any

_DISPLAY_COORDS_RE = re.compile(
    r"^([A-Z0-9_]+)\(\s*(?:ROW\s*=\s*)?(-?\d+)\s*,\s*(?:COL\s*=\s*)?(-?\d+)\s*\)$"
)


def action_signature(action: Any) -> str:
    """Stable signature for one action. Mouse coordinates are part of the
    identity: a click is only "the same action" at the same cell.

    Accepts both shapes the harness uses - the dict form the sandbox passes
    (`{"action": "MOUSE", "row": 4, "col": 7}`) and the display form the death
    ledger stores (`"MOUSE(row=4, col=7)"`) - and normalizes them to the same
    string, so a verdict recorded through one path is found through the other.
    Callers must pass MODEL-facing names (UP/DOWN/...), not engine names
    (ACTION1/ACTION2/...).
    """
    if isinstance(action, dict):
        name = str(action.get("action") or action.get("name") or "").strip().upper()
        row = action.get("row")
        col = action.get("col")
        if row is not None or col is not None:
            return f"{name}({row},{col})"
        return name
    text = " ".join(str(action or "").split()).upper()
    match = _DISPLAY_COORDS_RE.match(text)
    if match:
        return f"{match.group(1)}({match.group(2)},{match.group(3)})"
    return text


class NoopRepeatGuard:
    """Per-level store of state+action combinations known to do nothing."""

    def __init__(self, *, max_states_per_level: int = 512, max_actions_per_state: int = 16) -> None:
        self._max_states = max(1, int(max_states_per_level))
        self._max_actions = max(1, int(max_actions_per_state))
        self._states: OrderedDict[str, OrderedDict[str, None]] = OrderedDict()
        self._level: Any = None
        # One slot per guard kind. A shared slot let whichever guard ran first
        # consume the other's override: the death check runs before the no-op
        # check, cleared the arm on every non-match, and re-armed it, so the
        # no-op guard's "issue it again and it will execute" never fired. A
        # mouse-only game observed 291 replies and one executed action.
        self._armed_noop: tuple[str, str] | None = None
        self._armed_death: tuple[str, str] | None = None

    # -- level scoping -------------------------------------------------------
    def note_level(self, level: Any) -> None:
        """Drop everything when the level changes: a new layout means old
        state hashes describe boards that no longer exist. Deaths and resets
        deliberately do NOT clear the store - the board returns to the level
        start, so recorded no-ops still hold and carry across attempts."""
        if level != self._level:
            self._level = level
            self._states.clear()
            self._armed_noop = None
            self._armed_death = None

    # -- recording -----------------------------------------------------------
    def observe(self, state_hash: str, action_sig: str, *, gameplay_changed: Any) -> None:
        if not state_hash or not action_sig:
            return
        if gameplay_changed is False:
            entry = self._states.get(state_hash)
            if entry is None:
                entry = OrderedDict()
                self._states[state_hash] = entry
                while len(self._states) > self._max_states:
                    self._states.popitem(last=False)
            entry[action_sig] = None
            while len(entry) > self._max_actions:
                entry.popitem(last=False)
        elif gameplay_changed is True:
            # Evidence contradicts a recorded no-op: drop it. Reachable via the
            # override path, where a blocked action is executed after all.
            entry = self._states.get(state_hash)
            if entry is not None and action_sig in entry:
                del entry[action_sig]
                if not entry:
                    del self._states[state_hash]

    # -- blocking ------------------------------------------------------------
    def should_block(self, state_hash: str, action_sig: str) -> bool:
        """True when this combination is a known no-op AND the model has not
        just been told so. Arms the override slot when it blocks."""
        if not state_hash or not action_sig:
            return False
        key = (state_hash, action_sig)
        if self._armed_noop == key:
            # the model saw the refusal and insisted: let it through, and
            # re-arm nothing so a third attempt is blocked again
            self._armed_noop = None
            return False
        entry = self._states.get(state_hash)
        if not entry or action_sig not in entry:
            self._armed_noop = None
            return False
        self._armed_noop = key
        return True

    def note_executed(self, state_hash: str, action_sig: str) -> None:
        """Any executed action that is not the armed one clears the slot, so
        the override only applies to an immediate repeat."""
        key = (state_hash, action_sig)
        if self._armed_noop and self._armed_noop != key:
            self._armed_noop = None
        if self._armed_death and self._armed_death != key:
            self._armed_death = None

    # -- death verdicts ------------------------------------------------------
    def should_block_death(
        self,
        state_hash: str,
        action_sig: str,
        is_known_fatal: Callable[[str, str], bool],
    ) -> bool:
        """Same block-then-insist contract as `should_block`, but the verdict
        comes from the death ledger rather than this store.

        The ledger already keeps, per recorded attempt, an `actions` list and
        an index-aligned `hashes` list of interior state hashes taken BEFORE
        each action - so "this action, from this exact board, killed us" is a
        lookup, not a sequence match. Matching per position rather than by
        sequence prefix is what makes an interleaved or differently-routed
        approach to the same fatal move still get caught, while a batch whose
        earlier actions changed the board correctly does NOT match.

        Deaths are more likely than no-ops to hinge on state the board does
        not show (a hazard's facing, a timer), so blocking must stay
        overridable: under permadeath a hard block could foreclose the only
        path through a level with no way to ever discover the error.
        """
        if not state_hash or not action_sig:
            return False
        key = (state_hash, action_sig)
        if self._armed_death == key:
            self._armed_death = None
            return False
        if not is_known_fatal(state_hash, action_sig):
            # deliberately does NOT clear the slot: an action this guard has no
            # verdict on is none of its business, and clearing here would drop
            # an override armed for a different action
            return False
        self._armed_death = key
        return True


In [ ]:
%%writefile /tmp/agent_bundle/baseline/priority_scheduler.py
"""Pure scheduler inputs and scoring; never reads hidden game metadata."""
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any

# Expected future level-weight sum, conditional on completing the current level.
# 80k tokens are shared by ALL future levels; efficiency is NOT baked in.
# Source: arc3-levelup-chances, pooled_late_provisional, N6..10_H80000 / 0.8.
# KM duration distributions: separate levels 1-4, pooled observed levels 5-8
# for levels 5 onward. Levels 9-10 are extrapolated. Costs rounded up to 250
# tokens. 336 historical passes / 1,417 reached levels; not a contest fit.
# Tuples are indexed by current_level - 1. The final level has no future tail.
TAIL_LOOKUP_80K: dict[int, tuple[float, ...]] = {
    6: (3.9082328014, 5.0756939323, 5.3532912917, 5.6080111343,
        4.0898867400, 0.0),
    7: (3.9262240771, 5.1669653335, 5.6943514417, 6.6500041602,
        6.6562876217, 4.7715345300, 0.0),
    8: (3.9291260268, 5.1855629875, 5.7851444897, 7.0053702980,
        7.8471367942, 7.7045641090, 5.4531823200, 0.0),
    9: (3.9295002558, 5.1885168225, 5.8035897108, 7.0958580947,
        8.2469236991, 9.0442694281, 8.7528405964, 6.1348301100, 0.0),
    10: (3.9295397677, 5.1888936715, 5.8065473797, 7.1136691211,
         8.3474656955, 9.4884771003, 10.2414020621, 9.8011170838,
         6.8164779000, 0.0),
}


# Heuristic continuation value by levels remaining AFTER the current level:
# 3 or more -> 8; 2 -> 7; 1 -> 5; 0 -> 0. These are final B values,
# so neither score normalization nor the empirical efficiency discount applies.
TAIL_LOOKUP_REMAINING: dict[int, tuple[float, ...]] = {
    n: (8.0,) * (n - 3) + (7.0, 5.0, 0.0) for n in range(6, 11)
}


def parse_total_levels(value: Any) -> int | None:
    """Validate public level-count metadata, then clip to the table domain."""
    if value is None or isinstance(value, bool):
        return None
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    if not math.isfinite(number) or number <= 0 or not number.is_integer():
        return None
    return min(10, max(6, int(number)))


@dataclass
class ProgressPace:
    completed: int = 0
    log_cost_ratio: float = 0.0

    def record_completion(self, tokens_used: float, reference_tokens: float) -> None:
        observation = math.log(max(1.0, tokens_used) / max(1.0, reference_tokens))
        self.log_cost_ratio = (
            observation if self.completed == 0
            else 0.6 * self.log_cost_ratio + 0.4 * observation
        )
        self.completed += 1

    def cost_multiplier(self) -> float:
        confidence = self.completed / (self.completed + 3.0)
        return math.exp(min(math.log(2.0), max(math.log(0.5),
                                             confidence * self.log_cost_ratio)))


@dataclass(frozen=True)
class PrioritySnapshot:
    level: int
    actions: int
    tokens: float
    cost_multiplier: float = 1.0
    # Appended to preserve four-argument callers and older pickled snapshots.
    total_levels: int | None = None


def priority_value(
    state: PrioritySnapshot, *, endgame: bool = False,
    tail_fraction: float | None = None, human_actions: float = 25.0,
    action_scale: float = 115.0, token_scale: float = 62000.0,
    action_weight: float = 0.25, tail_base: float = 6.0, tail_cap: float = 6.0,
    normalize_score: bool = False, tail_lookup: bool | str = False,
    tail_efficiency: float = 0.8,
) -> int:
    """No phase offset: all candidates must be evaluated at the same time.

    tail_fraction=None retains the endgame rule. Supplying a fraction
    keeps the likelihood proxy active and fades the tail instead of switching.
    The likelihood proxy is heuristic, not a calibrated completion probability.

    tail_lookup=True uses the empirical 80k table; "remaining" uses the
    fixed 8/7/5/0 table without an efficiency discount. False keeps the capped
    tail. normalize_score scales only A, never B or the endgame's 0.25 floor.
    """
    level = max(1, state.level)
    total_levels = parse_total_levels(getattr(state, "total_levels", None)) or 10
    actions, tokens = max(0, state.actions), max(0.0, state.tokens)
    pace = min(2.0, max(0.5, state.cost_multiplier))
    efficiency = (human_actions / (human_actions + actions)) ** 2
    current_value = level * efficiency
    if normalize_score:
        # Normalize immediate reward A to a ten-level reference (sum 1..10).
        # B is a separately chosen continuation value, not normalized reward.
        current_value *= 55.0 / (total_levels * (total_levels + 1) / 2)
    if endgame and tail_fraction is None:
        # Preserve the original arithmetic when normalization is disabled.
        value = (
            level * 0.25 + current_value if normalize_score
            else level * (0.25 + efficiency)
        ) / pace * 100
    else:
        if tail_lookup == "remaining":
            index = min(total_levels, level) - 1
            tail = TAIL_LOOKUP_REMAINING[total_levels][index]
        elif tail_lookup:
            future_efficiency = (
                min(1.0, max(0.0, tail_efficiency))
                if math.isfinite(tail_efficiency) else 0.8
            )
            index = min(total_levels, level) - 1
            tail = TAIL_LOOKUP_80K[total_levels][index] * future_efficiency
        else:
            tail = min(tail_cap, tail_base + level - 1)
        if tail_fraction is not None:
            tail *= min(1.0, max(0.0, tail_fraction))
        chance = (
            action_weight * 0.5 ** ((actions / action_scale) ** 2)
            + (1.0 - action_weight) * 0.5 ** ((tokens / (token_scale * pace)) ** 2)
        )
        value = (current_value + tail) * chance / pace * 100
    return max(1, int(value))


In [ ]:
%%writefile /tmp/agent_bundle/baseline/replay_compare.py
"""Bounded, optional named-state replay and advisory progress text; stdlib only.
No game rules, actions, identity inference, global budgets or authorization.
"""
import json as _rc_json
import math as _rc_math
import time as _rc_time


def _rc_snapshot(value, root=True, depth=0, budget=None):
    if budget is None:
        budget = [4096]
    budget[0] -= 1
    if budget[0] < 0 or depth > 8:
        raise ValueError('named state exceeds 4096 values or depth 8')
    t = type(value)
    if root and (t is not dict or not value):
        raise TypeError('observe/predict must return a nonempty dict of named fields, not a positional tuple')
    if t is dict:
        if len(value) > 64 or any(type(k) is not str or len(k) > 80 for k in value):
            raise ValueError('state dict needs at most 64 string keys, each at most 80 characters')
        return {k: _rc_snapshot(v, False, depth+1, budget) for k, v in value.items()}
    if t in (list, tuple):
        if len(value) > 1024:
            raise ValueError('state sequence exceeds 1024 items')
        items = [_rc_snapshot(v, False, depth+1, budget) for v in value]
        return tuple(items) if t is tuple else items
    if t is str:
        if len(value) > 256:
            raise ValueError('state strings must be at most 256 characters')
        return value
    if t is int:
        if value.bit_length() > 256:
            raise ValueError('state integer exceeds 256 bits')
        return value
    if t is float:
        if not _rc_math.isfinite(value):
            raise ValueError('state numbers must be finite')
        return value
    if value is None or t is bool:
        return value
    raise TypeError('state values must be plain dict/list/tuple and finite scalar data')


def _rc_preview(value):
    text = _rc_json.dumps(value, ensure_ascii=True, separators=(',', ':'))
    return {'type': type(value).__name__, 'value_preview': text[:160], 'truncated': len(text) > 160}


def _rc_differences(predicted, observed):
    out = []; count = 0
    def add(path, kind, p=None, o=None, missing=None):
        nonlocal count
        count += 1
        if len(out) < 8:
            d = {'path': list(path), 'kind': kind}
            if missing != 'predicted': d['predicted'] = _rc_preview(p)
            if missing != 'observed': d['observed'] = _rc_preview(o)
            out.append(d)
    def walk(p, o, path):
        if type(p) is not type(o):
            add(path, 'type', p, o)
        elif type(p) is dict:
            for key in sorted(p.keys() | o.keys()):
                if key not in p: add(path+(key,), 'missing_predicted_field', o=o[key], missing='predicted')
                elif key not in o: add(path+(key,), 'extra_predicted_field', p=p[key], missing='observed')
                else: walk(p[key], o[key], path+(key,))
        elif type(p) in (list, tuple):
            if len(p) != len(o): add(path, 'length', len(p), len(o))
            for i in range(min(len(p),len(o))): walk(p[i], o[i], path+(i,))
        elif p != o:
            add(path, 'value', p, o)
    walk(predicted, observed, ())
    return out, count


def _rc_skip(record):
    before = getattr(record, 'before_frame', None)
    after = getattr(record, 'after_frame', None)
    action_name = getattr(record, 'action', None)
    result = getattr(record, 'result', None)
    if type(action_name) is not str or not action_name.strip() or len(action_name) > 256:
        return 'invalid_action_label'
    if action_name.strip().upper() == 'RESET': return 'reset'
    if type(result) is not dict: return 'missing_result'
    if result.get('execution_uncertain'): return 'execution_uncertain'
    if any(result.get(k) for k in ('game_over','level_completed','run_complete','done')): return 'terminal_boundary'
    if 'executed' in result and not result['executed']: return 'not_executed'
    if result.get('stop_reason'): return 'stopped'
    if before is None or after is None: return 'missing_frame'
    if getattr(before,'level',None) != getattr(after,'level',None): return 'level_boundary'
    bs = getattr(before,'step',None); es = getattr(after,'step',None)
    if type(bs) is not int or type(es) is not int or es != bs+1: return 'nonconsecutive_steps'
    return None


def replay_named_transitions(records, observe, predict, *, max_cases=16, timeout_s=2.0):
    """Each prediction starts at an independently decoded observed before-state.
    Only endpoint fields supplied by the caller are compared. A match is not
    safety, completeness, physical update-order proof or a plan execution ticket.
    """
    if type(records) not in (list,tuple): raise TypeError('records must be a finite list/tuple of TransitionView records')
    if not callable(observe) or not callable(predict): raise TypeError('observe and predict must be callable')
    if type(max_cases) is not int or max_cases < 1: raise ValueError('max_cases must be a positive integer')
    if type(timeout_s) not in (int,float) or not _rc_math.isfinite(timeout_s) or timeout_s <= 0:
        raise ValueError('timeout_s must be finite and positive')
    effective_cases = min(max_cases,64); effective_time = min(timeout_s,5.0)
    start = _rc_time.monotonic(); expired = lambda: _rc_time.monotonic()-start >= effective_time
    report = {'status':'empty','scope':'selected observed nonterminal transitions and supplied named fields only; not safety or model certification',
        'input_count':len(records),'scanned_count':0,'input_truncated':len(records)>128,
        'eligible_count':0,'selected_count':0,'selection_truncated':False,
        'counts':{'matched':0,'mismatched':0,'errors':0,'skipped':0},'skip_reasons':{},'cases':[],
        'effective_limits':{'max_cases':effective_cases,'timeout_s':effective_time,'scan_records':128,'diffs_per_case':8},
        'clamped_limits':[k for k,v in (('max_cases',max_cases>64),('timeout_s',timeout_s>5.0)) if v]}
    eligible=[]
    for i in range(max(0,len(records)-128),len(records)):
        if expired(): report['status']='limit';report['reason']='time';break
        record=records[i];report['scanned_count']+=1;why=_rc_skip(record)
        if why:
            report['counts']['skipped']+=1
            report['skip_reasons'][why]=report['skip_reasons'].get(why,0)+1
        else: eligible.append((i,record))
    report['eligible_count']=len(eligible)
    chosen=eligible[-effective_cases:];report['selected_count']=len(chosen)
    report['selection_truncated']=len(chosen)<len(eligible)
    if report['status']!='limit':
        for index,record in chosen:
            if expired(): report['status']='limit';report['reason']='time';break
            entry={'index':index,'before_step':record.before_frame.step,'after_step':record.after_frame.step,'level':record.before_frame.level,'action':record.action}
            phase='observe_before'
            try:
                before=_rc_snapshot(observe(record.before_frame))
                if expired(): report['status']='limit';report['reason']='time';break
                phase='observe_after';observed=_rc_snapshot(observe(record.after_frame))
                if expired(): report['status']='limit';report['reason']='time';break
                phase='predict';predicted=_rc_snapshot(predict(_rc_snapshot(before),record.action))
                if expired(): report['status']='limit';report['reason']='time';break
                diffs,total=_rc_differences(predicted,observed)
                entry.update(status='mismatch' if total else 'match',fields=sorted(observed) if isinstance(observed, dict) else [],differences=diffs,difference_count=total,differences_truncated=total>len(diffs))
                report['counts']['mismatched' if total else 'matched']+=1
            except Exception as exc:  # noqa: BLE001 - archived callback/retention failure boundary
                entry.update(status='callback_error',phase=phase,error_type=type(exc).__name__)
                # No repr/str on callback-owned exceptions or arbitrary objects.
                report['counts']['errors']+=1
                report['cases'].append(entry);report['status']='callback_error';break
            report['cases'].append(entry)
        if report['status'] not in ('limit','callback_error'):
            report['status']='mismatch' if report['counts']['mismatched'] else ('matches_observed' if report['counts']['matched'] else 'empty')
    report['elapsed_s']=round(_rc_time.monotonic()-start,6)
    return report


def replay_summary(report):
    c=report['counts'];text='[replay_transitions] status={} matched={:d} mismatched={:d} errors={:d} skipped={:d} selected={:d}; supplied fields only, not safety.'.format(report['status'],c['matched'],c['mismatched'],c['errors'],c['skipped'],report['selected_count'])
    first=next((x for x in report['cases'] if x['status']!='match'),None)
    if first:
        text+=' First issue: step {} {}'.format(first['after_step'],first['status'])
        if first.get('differences'): text+=' path='+_rc_json.dumps(first['differences'][0]['path'],ensure_ascii=True)[:120]
        if first.get('phase'):text+=' phase='+first['phase']+' type='+first['error_type']
    if report['input_truncated'] or report['selection_truncated']:text+=' Selected tail only; earlier evidence not checked.'
    return text[:480]


def planning_progress_note(previous, key, now, timing, executed=None):
    """Readonly session timing snapshot; no synthesized/extended game deadline.
    executed is None for opener, False for a call without confirmed action,
    True for confirmed actions. No-action count is not lack-of-learning proof.
    """
    if type(previous) is not dict or previous.get('key')!=key or executed is True:
        previous={'key':key,'since':now,'calls':0}
    current=dict(previous)
    if executed is False: current['calls']=min(999999,current.get('calls',0)+1)
    elapsed=max(0.0,now-current['since'])
    remaining=timing.get('time_remaining_seconds') if type(timing) is dict else None
    if not isinstance(remaining, (int, float)) or isinstance(remaining, bool) or not _rc_math.isfinite(remaining) or remaining<0:
        remaining=None
    clock='unknown' if remaining is None else f'{remaining:.1f}s'
    text='[planning_progress] game_remaining_at_note={}; no_action_calls={:d}; interval={:.1f}s. Clock snapshot ages during generation.'.format(clock,current['calls'],elapsed)
    if current['calls']>=3 or (remaining is not None and remaining<=120):
        text+=' Prefer a short evidence-checked execution or a probe that distinguishes predictions over another rewrite. A printed plan executes nothing; raw actions remain available. Never blindly execute an unvalidated route.'
    return current,text[:480]


In [ ]:
%%writefile /tmp/agent_bundle/baseline/retained_functions.py
"""Conservative source-only function retention; no snippet state is serialized.

Also embedded into the isolated Python runner. Keep this module stdlib-only.
"""
import ast
import dis
import sys
import types


def _import_statements(nodes):
    """One explicit absolute import per binding; never replay wildcard imports."""
    statements = {}
    for node in nodes:
        if isinstance(node, ast.Import):
            for alias in node.names:
                binding = alias.asname or alias.name.split('.')[0]
                statements[binding] = ast.unparse(ast.Import(names=[alias]))
        elif isinstance(node, ast.ImportFrom) and not node.level and node.module:
            for alias in node.names:
                if alias.name == '*':
                    raise ValueError('wildcard imports are not retained; use explicit imports inside the function')
                statements[alias.asname or alias.name] = ast.unparse(
                    ast.ImportFrom(module=node.module, names=[alias], level=0))
        else:
            raise ValueError('only explicit absolute imports can accompany retained functions')
    return statements


def _source_parts(source, retain_imports=True):
    tree = ast.parse(source, '<retained>', 'exec')
    if (not tree.body or not isinstance(tree.body[-1], ast.FunctionDef)
            or (not retain_imports and len(tree.body) != 1)):
        raise ValueError('only top-level function definitions can be retained')
    imports = _import_statements(tree.body[:-1])
    return tree.body[-1], imports


def function_signature(source):
    node, _ = _source_parts(source)
    return node.name + '(' + ast.unparse(node.args) + ')'


def _function_record(source, retain_imports=False):
    node, imports = _source_parts(source, retain_imports)
    if getattr(node, "type_params", []):
        raise ValueError("generic type parameters are not retained")
    if node.decorator_list:
        raise ValueError('decorated functions are not retained')
    args = [*node.args.posonlyargs, *node.args.args, *node.args.kwonlyargs]
    args += [a for a in (node.args.vararg, node.args.kwarg) if a is not None]
    if node.returns is not None or any(a.annotation is not None for a in args):
        raise ValueError('use an unannotated definition for retention')
    # Restoration must never evaluate model calls or snapshot a frame via defaults.
    for default in [*node.args.defaults, *node.args.kw_defaults]:
        if default is not None:
            try:
                ast.literal_eval(default)
            except Exception:  # noqa: BLE001 - archived callback/retention failure boundary
                raise ValueError('defaults must be literals; pass changing data as arguments') from None
    tree = ast.Module(body=[node], type_ignores=[])
    code = next(c for c in compile(tree, '<retained>', 'exec').co_consts
                if isinstance(c, types.CodeType))
    reads = set()
    pending = [code]
    while pending:
        current = pending.pop()
        for instruction in dis.get_instructions(current):
            if instruction.opname in ('LOAD_GLOBAL', 'LOAD_NAME'):
                reads.add(instruction.argval)
            elif instruction.opname in ('STORE_GLOBAL', 'DELETE_GLOBAL'):
                raise ValueError('functions that assign global variables are not retained')
        pending.extend(c for c in current.co_consts if isinstance(c, types.CodeType))
    return node.name, source, reads - imports.keys()


def _rejection(name, reason, hint=None, **metadata):
    result = {'name': name, 'reason': reason, **metadata}
    if hint:
        result['hint'] = hint
    return result


def _validation_hint(reason):
    if 'decorat' in reason:
        return 'Use a plain top-level def without decorators if you want to reuse it.'
    if 'annotat' in reason:
        return 'Remove parameter and return annotations if you want to retain this function.'
    if 'default' in reason:
        return 'Use literal defaults and pass changing values explicitly as arguments.'
    if 'global variables' in reason:
        return 'Return updated values instead of assigning global variables.'
    if 'import' in reason:
        return 'Put explicit, sandbox-supported imports inside the function.'
    return 'Use a plain top-level def if you want to retain this function.'


def prepare_functions(sources, available, *, retain_imports=False):
    """Validate sources without executing them; drop invalid dependency chains."""
    records, rejected = {}, []
    for source in sources:
        name = '<unknown>'
        try:
            parsed = ast.parse(source)
            if (parsed.body and isinstance(parsed.body[-1], ast.FunctionDef)
                    and (retain_imports or len(parsed.body) == 1)):
                name = parsed.body[-1].name
                records.pop(name, None)
            name, source, reads = _function_record(source, retain_imports)
            _, imports = _source_parts(source)
            if imports.keys() & set(available):
                raise ValueError('import alias conflicts with a provided global or builtin')
            records[name] = (source, reads)
        except Exception as exc:  # noqa: BLE001 - archived callback/retention failure boundary
            reason = str(exc)[:180]
            rejected.append(_rejection(name, reason, _validation_hint(reason)))
    # Import bundles must not overwrite another retained function at restore.
    function_names = set(records)
    for name, (source, _) in list(records.items()):
        _, imports = _source_parts(source)
        if imports.keys() & function_names:
            del records[name]
            rejected.append(_rejection(name, 'import alias conflicts with a retained function',
                                       'Choose a different import alias or function name.'))
    # A wrapper must not survive its unavailable dependency. Re-run to a fixed point.
    while True:
        dropped = []
        for name, (source, reads) in records.items():
            missing = reads - set(available) - records.keys()
            if missing:
                dropped.append(name)
                names = ', '.join(sorted(missing))
                rejected.append(_rejection(
                    name, 'unavailable dependencies: ' + names,
                    'Pass ' + names + ' as arguments, or put their imports inside the function.',
                    missing_dependencies=sorted(missing)))
        if not dropped:
            break
        for name in dropped:
            del records[name]
    return {name: value[0] for name, value in records.items()}, rejected


def restore_functions(sources, namespace, available, *, retain_imports=False,
                      repair_hints=False):
    """Restore definitions and explicit imports through the sandbox importer.

    Never replay assignments, expressions, decorators, or previous actions.
    """
    kept, rejected = prepare_functions(sources, available, retain_imports=retain_imports)
    failed = set()
    for name, source in kept.items():
        before = dict(namespace)
        try:
            if name in available:
                raise ValueError('name conflicts with a provided global or builtin')
            exec(compile(source, '<retained>', 'exec'), namespace, namespace)  # noqa: S102 - source-only helper restoration; call in sandbox
        except Exception:  # noqa: BLE001 - archived callback/retention failure boundary
            # A multi-import bundle can fail partway through. Restore bindings
            # so one failed helper does not replace another helper's globals.
            namespace.clear()
            namespace.update(before)
            failed.add(name)
            reason = ('could not restore this definition or its imports' if repair_hints else
                      'could not restore this definition; redefine it if needed')
            rejected.append(_rejection(name, reason,
                                       'Check its imports are supported by the sandbox, then redefine it if needed.'))
    survivors, dependencies = prepare_functions(
        [source for name, source in kept.items() if name not in failed], available,
        retain_imports=retain_imports)
    rejected.extend(dependencies)
    for name in kept.keys() - survivors.keys():
        if name not in available:
            namespace.pop(name, None)
    return survivors, rejected


def _matching_imports(nodes, namespace):
    """Recognize live import bindings without importing or evaluating anything.

    Only module bindings and callable/immutable exports are eligible. Ordinary
    snippet assignments and mutable imported data are never serialized. Looking
    in module dictionaries avoids invoking __getattr__ during bookkeeping.
    """
    matches, declared = {}, {}
    for node in nodes:
        if not isinstance(node, (ast.Import, ast.ImportFrom)):
            continue
        try:
            statements = _import_statements([node])
        except ValueError:
            continue
        declared.update(statements)
        for binding, statement in statements.items():
            single = ast.parse(statement).body[0]
            alias = single.names[0]
            if isinstance(single, ast.Import):
                # Even unaliased dotted imports must already have executed.
                if alias.name not in sys.modules:
                    continue
                module_name = alias.name if alias.asname else alias.name.split('.')[0]
                expected = sys.modules.get(module_name)
            else:
                module = sys.modules.get(single.module)
                expected = vars(module).get(alias.name) if type(module) is types.ModuleType else None
            eligible = type(expected) in (
                types.ModuleType, types.FunctionType, types.BuiltinFunctionType, types.MethodType,
                type, str, bytes, int, float, complex, bool,
            )
            if eligible and binding in namespace and namespace[binding] is expected:
                matches[binding] = statement
    return matches, declared


def collect_functions(code_text, previous, namespace, available, *, retain_imports=False,
                      repair_hints=False):
    """Collect from the actual namespace, including after partial execution.

    A later def may never have run. Match the live function's source location
    rather than choosing the last definition in the submitted snippet.
    """
    candidates = {}
    rejected = []
    try:
        tree = ast.parse(code_text, '<python_tool>')
    except SyntaxError:
        # Parsing failed before execution; restored functions are still valid.
        tree = ast.Module(body=[], type_ignores=[])
    fresh = {}
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            fresh.setdefault(node.name, []).append(node)
    import_nodes = []
    for source in previous.values():
        # Old source-only records remain valid; no migration is necessary.
        import_nodes.extend(ast.parse(source).body[:-1])
    import_nodes.extend(tree.body)
    imports, declared_imports = (_matching_imports(import_nodes, namespace)
                                if retain_imports or repair_hints else ({}, {}))
    for name in dict.fromkeys([*previous, *fresh]):
        value = namespace.get(name)
        if not repair_hints and (name in available or type(value) is not types.FunctionType
                                 or value.__name__ != name):
            rejected.append(_rejection(name, 'definition did not execute, was removed/replaced, or conflicts with a provided name'))
            continue
        if name in available:
            rejected.append(_rejection(name, 'name conflicts with a provided global or builtin',
                                       'Choose a different function name.'))
            continue
        if name not in namespace:
            reason = 'function was removed' if name in previous else 'definition was not reached or was removed'
            rejected.append(_rejection(name, reason, 'Define it before calling it if it is still needed.'))
            continue
        if type(value) is not types.FunctionType or value.__name__ != name:
            rejected.append(_rejection(name, 'function name was replaced by another value',
                                       'Use a separate variable name; redefine the function only if needed.'))
            continue
        if name in previous and value.__code__.co_filename == '<retained>':
            # Preserve the restored definition when execution stopped before a
            # replacement (or while evaluating that replacement's defaults).
            candidates[name] = previous[name]
            continue
        node = next((node for node in fresh.get(name, [])
                     if isinstance(node, ast.FunctionDef) and not node.decorator_list
                     and value.__code__.co_filename == '<python_tool>'
                     and value.__code__.co_firstlineno == node.lineno), None)
        if node is None:
            rejected.append(_rejection(name, 'definition was decorated, replaced outside a top-level def, or is unsupported',
                                       'Use a plain top-level def without decorators if you want to reuse it.'))
            continue
        candidates[name] = ast.get_source_segment(code_text, node)
    # Rebuild every bundle from the LIVE namespace. An old helper must not
    # silently restore an obsolete import when that alias was rebound/deleted.
    for name, source in (list(candidates.items()) if retain_imports else []):
        try:
            node, _ = _source_parts(source)
            definition = ast.get_source_segment(source, node)
            assert definition is not None
            _, _, reads = _function_record(definition)
            statements = [imports[n] for n in sorted(reads & imports.keys()) if n not in available]
            candidates[name] = '\n'.join([*statements, definition])
        except Exception:  # noqa: BLE001, S112 - validation below reports the rejection
            # Validation below produces the specific rejection and prunes callers.
            continue
    kept, invalid = prepare_functions(candidates.values(), available, retain_imports=retain_imports)
    for item in invalid:
        missing = item.get('missing_dependencies', [])
        if not missing:
            continue
        hints = []
        missing_imports = [n for n in missing if n in declared_imports]
        missing_helpers = [n for n in missing if n in candidates and n not in declared_imports]
        missing_data = sorted(set(missing) - set(missing_imports) - set(missing_helpers))
        if missing_imports:
            statements = '; '.join(declared_imports[n] for n in missing_imports)
            hints.append('Put `' + statements + '` inside this function if you want to reuse it.')
        if missing_helpers:
            hints.append('Make the helper dependencies retainable too: ' + ', '.join(missing_helpers) + '.')
        if missing_data:
            hints.append('Pass ' + ', '.join(missing_data) + ' as arguments; snippet variables are not retained.')
        item['hint'] = ' '.join(hints)
    return kept, rejected + invalid


In [ ]:
%%writefile /tmp/agent_bundle/baseline/state_contracts.py
"""Geometry, dispatch validation and bounded error context; no game mechanics.

No new persistence, model inference, automatic retry or action dispatch.
"""
import ast as _sc_ast
import json as _sc_json


def component_geometry(cells):
    r0 = min(p[0] for p in cells)
    c0 = min(p[1] for p in cells)
    r1 = max(p[0] for p in cells)
    c1 = max(p[1] for p in cells)
    return {'bbox_rc': [r0, c0, r1, c1],
            'bbox_center2_rc': [r0 + r1, c0 + c1]}


def validate_mouse_coordinates(actions, shape):
    """Validate the whole normalized batch BEFORE its first dispatch; no coercion."""
    for i, item in enumerate(actions, 1):
        if item.get('action', '').upper() != 'MOUSE':
            continue
        if 'row' not in item or 'col' not in item:
            raise ValueError(f'MOUSE action {i:d} requires both integer row and col.')
        r, c = item['row'], item['col']
        if type(r) is not int or type(c) is not int:
            raise TypeError(f'MOUSE action {i:d}: row and col must each be an integer, not nested coordinates, floats, strings or booleans. Use one explicit (row, col) pair.')
        if len(shape) != 2 or not (0 <= r < shape[0] and 0 <= c < shape[1]):
            raise ValueError(f'MOUSE action {i:d} coordinate is outside current_frame.shape.')


def _value_shape(v, depth=0):
    # Never call repr/str/len/iteration hooks on a user-defined object.
    t = type(v)
    if v is None:
        return 'None'
    for cls, label in ((bool, 'bool'), (int, 'int'), (float, 'float')):
        if t is cls:
            return label
    if t is str:
        return f'str(length={len(v):d})'
    if t is dict:
        keys = []
        for i, k in enumerate(v):
            if i >= 8:
                break
            if type(k) is str:
                keys.append(k[:24])
        return 'dict(keys={}{})'.format(','.join(keys), ',...' if len(v) > 8 else '')
    if t in (list, tuple):
        name = 'list' if t is list else 'tuple'
        if depth >= 2:
            return f'{name}(length={len(v):d})'
        return '{}(length={:d},first={})'.format(name, len(v),
                   ','.join(_value_shape(x, depth+1) for x in v[:3]))
    if t in (set, frozenset):
        return f'set(length={len(v):d})'
    return 'object'


def exception_shape_context(exc, code):
    """Type/shape facts at the failing generated frame, never candidate contents."""
    tb = exc.__traceback__
    selected = None
    while tb is not None:
        if tb.tb_frame.f_code.co_filename in ('<python_tool>', '<retained>'):
            selected = tb
        tb = tb.tb_next
    if selected is None:
        return ''
    f = selected.tb_frame
    names = []
    if f.f_code.co_filename == '<python_tool>':
        try:
            tree = _sc_ast.parse(code)
            for node in _sc_ast.walk(tree):
                if (isinstance(node, _sc_ast.Name) and node.lineno == selected.tb_lineno
                        and node.id not in names):
                        names.append(node.id)
        except (SyntaxError, ValueError):
            pass
    # Include local arguments of retained helpers and lambdas, with a firm cap.
    for name in f.f_code.co_varnames:
        if name not in names:
            names.append(name)
    shapes = {}
    for name in names[:10]:
        if name.startswith('__'):
            continue
        if name in f.f_locals:
            value = f.f_locals[name]
        elif name in f.f_globals:
            value = f.f_globals[name]
        else:
            shapes[name[:40]] = 'MISSING'
            continue
        shapes[name[:40]] = _value_shape(value)[:180]
    if not shapes:
        return ''
    omitted = 0
    text = _sc_json.dumps({'values': shapes, 'omitted': omitted}, separators=(',', ':'), ensure_ascii=True)
    while len(text) > 1800 and shapes:
        shapes.popitem(); omitted += 1
        text = _sc_json.dumps({'values': shapes, 'omitted': omitted}, separators=(',', ':'), ensure_ascii=True)
    return '\n[value_shapes] ' + text + '\nShapes only; no data restored or repaired. Pass snippet-local data explicitly into retained functions.'


def execution_receipt(results, frame, uncertain=False):
    """Host-owned facts: never infer exact counts from an ambiguous batch reply."""
    count = 0
    complete = True
    calls = 0
    for r in results:
        executed = r.get('executed')
        if executed:
            calls += 1
        n = r.get('executed_count')
        if type(n) is int and n >= 0:
            count += n
        elif type(executed) is int and executed >= 0:
            count += executed
        elif executed is True:
            names = r.get('executed_actions')
            if type(names) is list:
                count += len(names)
            else:
                complete = False
        elif executed not in (False, None):
            complete = False
    details = {'action_calls_confirmed': calls,
               'confirmed_actions': count if complete else None,
               'execution_uncertain': bool(uncertain)}
    for name in ('level', 'step'):
        value = frame.get(name) if type(frame) is dict else None
        if type(value) is int:
            details['last_confirmed_' + name] = value
    return ('\n[execution_receipt] ' + _sc_json.dumps(details, separators=(',', ':')) +
            '\nNo automatic retry. Re-read current_frame/history before repairing the code; never replay an already executed prefix. The last confirmed frame may be stale if execution is uncertain.')


In [ ]:
%%writefile /tmp/agent_bundle/baseline/transition_evidence.py
"""Bounded visual/terminal evidence, not a resource decoder or death classifier.
Standard library only; source is injected into the isolated Python tool.
"""
import itertools


def _te_grid(frame):
    g = frame.get('grid') if type(frame) is dict else None
    if type(g) not in (list, tuple) or not g or len(g) > 256:
        return None
    if type(g[0]) not in (list, tuple) or not g[0] or len(g[0]) > 256 or len(g)*len(g[0]) > 8192:
        return None
    w = len(g[0])
    if any(type(row) not in (list, tuple) or len(row) != w or any(type(x) is not int or not 0 <= x < 16 for x in row) for row in g):
        return None
    return tuple(tuple(row) for row in g)


def _te_contiguous(a, b):
    if type(a) is not dict or type(b) is not dict:
        return False
    if any(type(f.get(k)) is not int for f in (a,b) for k in ('step','level')):
        return False
    action = b.get('action')
    result = b.get('result')
    if type(action) is not str or not action or action.strip().upper().startswith('RESET') or type(result) is not dict:
        return False
    if any(result.get(k) is True for k in ('level_completed','run_complete','execution_uncertain')):
        return False
    if 'executed' in result and result['executed'] in (False,0,None):
        return False
    # A death frame itself is evidence; do not discard the terminal transition.
    return a['level'] == b['level'] and b['step'] == a['step']+1


def _te_stripes(frames, colors):
    if len(frames) < 3 or not all(_te_contiguous(a,b) for a,b in zip(frames[-3:],frames[-2:])):
        return []
    grids = [_te_grid(f) for f in frames[-3:]]
    if any(g is None for g in grids):
        return []
    grids = [g for g in grids if g is not None]
    if any(g is None for g in grids) or len({(len(g),len(g[0])) for g in grids}) != 1:
        return []
    results = []
    for axis in ('row','col'):
        lines = grids if axis == 'row' else [tuple(zip(*g)) for g in grids]
        for index in range(len(lines[0])):
            stripe = [g[index] for g in lines]
            # A contiguous two-color fill is only a CANDIDATE indicator. No edge,
            # palette role, game identifier or pixel coordinate is assumed.
            palette = set().union(*(set(line) for line in stripe))
            if len(palette) != 2 or any(sum(a != b for a,b in itertools.pairwise(line)) > 1 for line in stripe):
                continue
            for color in sorted(palette):
                counts = [line.count(color) for line in stripe]
                if counts[0] > counts[1] > counts[2]:
                    results.append({'axis':axis,'index':index,'color':colors[color],
                        'counts':counts,'steps':[f['step'] for f in frames[-3:]],
                        'length':len(stripe[0]),'empty_now':counts[-1] == 0,
                        'interpretation':'possible resource/counter or ordinary geometry; unresolved'})
    results.sort(key=lambda x:(not x['empty_now'],-(x['counts'][0]-x['counts'][-1]),x['axis'],x['index']))
    return results


def transition_evidence(frames, colors):
    """Frames: at most 64 recent host snapshots including action/result metadata.

Raw grids remain unchanged. No physical IDs, hazard rules, resource units,
remaining-action estimates or terminal causes are asserted.
"""
    out = {'status':'ok','stripe_candidates':[],'same_scene_comparisons':[],
           'recent_death':None,'cause':'unknown','candidates_truncated':False,
           'limits':{'history':64,'grid_cells':8192,'dimension':256,'stripes':4,'comparisons':2},
           'coverage':'narrow visual evidence, not complete resource/hazard state'}
    if type(frames) not in (list,tuple) or not frames or any(type(f) is not dict for f in frames[-64:]):
        out.update(status='unavailable',reason='invalid_or_missing_history');return out
    frames = frames[-64:]
    current = frames[-1]
    grid = _te_grid(current)
    if grid is None:
        out.update(status='unavailable',reason='invalid_or_oversized_grid');return out
    candidates = _te_stripes(frames,colors)
    out['candidates_truncated'] = len(candidates)>4
    out['stripe_candidates'] = candidates[:4]
    # Equal visible scene EXCEPT an explicitly identified candidate stripe does
    # not imply equivalent state. Both the differing counts and terminal flags
    # are retained, including comparisons across attempts in the same level.
    for stripe in candidates[:4]:
        axis,index = stripe['axis'],stripe['index']
        rows = grid if axis == 'row' else tuple(zip(*grid))
        key = rows[:index]+rows[index+1:]
        for old in reversed(frames[:-1]):
            if old.get('level') != current.get('level') or old.get('step') == current.get('step'):
                continue
            previous = _te_grid(old)
            if previous is None or (len(previous),len(previous[0])) != (len(grid),len(grid[0])):
                continue
            other = previous if axis == 'row' else tuple(zip(*previous))
            if other[index] == rows[index] or other[:index]+other[index+1:] != key:
                continue
            color = colors.index(stripe['color'])
            result_old = old.get('result') if type(old.get('result')) is dict else {}
            result_now = current.get('result') if type(current.get('result')) is dict else {}
            out['same_scene_comparisons'].append({'axis':axis,'index':index,'color':stripe['color'],
                'steps':[old.get('step'),current.get('step')],
                'counts':[other[index].count(color),rows[index].count(color)],
                'game_over':[result_old.get('game_over') is True,result_now.get('game_over') is True],
                'outside_stripe_equal':True,'state_equivalence':'not established'})
            break
        if len(out['same_scene_comparisons']) >= 2:
            break
    # Include a just-ended attempt even when the harness has appended RESET.
    # Do not leak a previous level's death or repeatedly warn many actions later.
    for i in range(len(frames)-1,max(-1,len(frames)-5),-1):
        end = frames[i]
        result = end.get('result')
        if end.get('level') != current.get('level') or type(result) is not dict or result.get('game_over') is not True:
            continue
        death = {'step':end.get('step'),'cause':'unknown','stripe_candidates':[],
                 'colors_lost_completely':[],'interpretation':'terminal evidence, not a collision/exhaustion verdict'}
        death['stripe_candidates'] = _te_stripes(frames[max(0,i-2):i+1],colors)[:4]
        if i and _te_contiguous(frames[i-1],end):
            before,after = _te_grid(frames[i-1]),_te_grid(end)
            if before and after and (len(before),len(before[0])) == (len(after),len(after[0])):
                ca,cb = [0]*16,[0]*16
                for row in before:
                    for x in row:ca[x]+=1
                for row in after:
                    for x in row:cb[x]+=1
                death['colors_lost_completely'] = [{'color':colors[c],'before_pixels':ca[c],'after_pixels':0} for c in range(16) if ca[c] and not cb[c]][:4]
        out['recent_death'] = death
        break
    return out


def transition_evidence_note(evidence):
    """Only internally generated plain data is formatted; never a user callback."""
    if evidence['status'] != 'ok':
        return ''
    parts = []
    death = evidence['recent_death']
    if death:
        parts.append('recent game_over step='+str(death['step'])+' cause=unknown')
        for stripe in death['stripe_candidates'][:1]:
            parts.append('terminal stripe '+stripe['axis']+'='+str(stripe['index'])+' '+stripe['color']+' counts='+str(stripe['counts'])+' empty='+str(stripe['empty_now']))
        if death['colors_lost_completely']:
            parts.append('colors absent after death='+str([x['color'] for x in death['colors_lost_completely']]))
    for stripe in evidence['stripe_candidates'][:1]:
        parts.append('candidate stripe '+stripe['axis']+'='+str(stripe['index'])+' '+stripe['color']+' counts='+str(stripe['counts']))
    for pair in evidence['same_scene_comparisons'][:1]:
        parts.append('same pixels outside stripe at steps='+str(pair['steps'])+' but counts='+str(pair['counts'])+' game_over='+str(pair['game_over']))
    if not parts:
        return ''
    return ('[transition_evidence] '+'; '.join(parts)+
        '; resource meaning unresolved. Do not equate these states or infer new hazard rules from death alone. Details: current_frame.transition_evidence.')[:850]


def frozen_frame_defaults(callback, frame_type):
    """Detect direct frame defaults on plain functions only; no arbitrary hooks.
    Not a universal stale-closure detector. Do not mutate or invoke the callback.
    """
    if type(callback) is not type(lambda:None):
        return False
    defaults = callback.__defaults__
    kw = callback.__kwdefaults__
    return (type(defaults) is tuple and any(type(x) is frame_type for x in defaults[:64])) or (type(kw) is dict and any(type(x) is frame_type for x in list(kw.values())[:64]))


In [ ]:
%%writefile /tmp/agent_bundle/local_backend.py
"""Bounded local OpenAI-compatible inference; generated code stays in the worker."""
from __future__ import annotations

import http.client
import json
import time
from collections.abc import Callable
from pathlib import Path
from typing import Any

PROTOCOL = '''Return exactly one JSON object, without markdown.
Use only the observed state, objects, actions and replay evidence in this conversation.
Return {"actions": [{"action": "ACTION1", "expect": [{"x": 0, "y": 0, "value": 0}],
"expect_level_up": false}]} for a concrete plan; coordinates x/y are required for a
complex action. Every expect must predict non-HUD cells; never invent observations.
Alternatively return {"hypotheses": [{"source": "def step(state, action): ...",
"probe": {"action": "ACTION1", "expect": [{"x": 0, "y": 0, "value": 0}],
"expect_level_up": false}, "alternative_expect": [{"x": 0, "y": 0, "value": 1}],
"scope": "level"}]}. A simulator request returns {"simulator_source": "def step(state, action): ..."}.
Synthesized step receives a JSON state with grid, level and state, and an action
dictionary. It must return that complete state; unknown mechanics stay unknown.
The example coordinates/colors above illustrate the schema only, not a game rule.
Your proposals undergo full historical replay and forward-prediction checks.
'''


class LocalBackend:
    def __init__(self, directory: Path, limits: Callable[[], tuple[int, float]],
                 context: Callable[[], str], *, model: str = 'flashnext') -> None:
        self.directory, self.limits, self.context, self.model = directory, limits, context, model

    def __call__(self, messages: list[dict[str, Any]]) -> dict[str, Any]:
        tokens, timeout = self.limits()
        if tokens <= 0 or timeout <= 0:
            raise TimeoutError('reasoning budget exhausted')
        # Send the immutable complete history verbatim. No timestamps, injected
        # early messages, pruning, or mutation of prior assistant turns here.
        payload = {'model': self.model, 'messages': messages, 'max_tokens': tokens,
                   'temperature': 0.7, 'top_p': 0.95,
                   'response_format': {'type': 'json_object'}}
        encoded = json.dumps(payload, separators=(',', ':')).encode()
        began = time.monotonic()
        connection = http.client.HTTPConnection('127.0.0.1', 8001, timeout=min(30.0, timeout))
        record: dict[str, Any] = {'context': self.context(), 'model': self.model,
                                  'messages': len(messages), 'cache_hit_rate': None}
        try:
            connection.request('POST', '/v1/chat/completions', body=encoded,
                               headers={'Content-Type': 'application/json'})
            result = connection.getresponse()
            raw = result.read(4_194_305)
            if result.status != 200 or len(raw) > 4_194_304:
                raise RuntimeError(f'local inference HTTP {result.status} or oversized response')
            data = json.loads(raw)
            content = data['choices'][0]['message']['content']
            response = json.loads(content)
            if not isinstance(response, dict) or not any(response.get(k) for k in (
                    'actions', 'hypotheses', 'simulator_source')):
                raise ValueError('local backend returned no structured proposal')
            usage = data.get('usage', {})
            prompt, completion = usage.get('prompt_tokens'), usage.get('completion_tokens')
            details = usage.get('prompt_tokens_details') or {}
            cached = details.get('cached_tokens')
            record.update(prompt_tokens=prompt, completion_tokens=completion, cached_tokens=cached,
                          status='response')
            if type(prompt) is int and prompt > 0 and type(cached) is int and 0 <= cached <= prompt:
                record['cache_hit_rate'] = cached / prompt
            return response
        except Exception as error:
            record.update(status='error', error_type=type(error).__name__, error=str(error))
            raise
        finally:
            connection.close()
            record['elapsed_seconds'] = time.monotonic() - began
            completion = record.get('completion_tokens')
            if type(completion) is int and record['elapsed_seconds'] > 0:
                record['completion_tokens_per_second'] = completion / record['elapsed_seconds']
            self.directory.mkdir(parents=True, exist_ok=True)
            with (self.directory / 'llm_requests.jsonl').open('a') as stream:
                stream.write(json.dumps(record, sort_keys=True) + '\n')


In [ ]:
%%writefile /tmp/agent_bundle/my_agent.py
"""Evidence-gated memory, sandbox retrodiction and checked action planning.

Generated mechanics implement step(state, action), returning a complete named
state: grid, level and state (public framework status). Actions use framework
enum names and optional x/y. No generated Python executes in the engine host.
An injected reasoning backend returns structured hypotheses, [ACTIONS] or
simulator source. The default Generation 3 contract halts if no verified route
exists. ENABLE_AUDITABLE_WORLD_MODEL=0 retains the archived exploration control.
"""
from __future__ import annotations

import ast
import hashlib
import inspect
import json
import logging
import os
import random
import tempfile
import time
import traceback
from collections import deque
from collections.abc import Callable
from pathlib import Path
from typing import Any

# When run inside the ARC-AGI-3-Agents framework (locally or on Kaggle)
# the `agents` package is on sys.path, so this import resolves.
from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState

try:
    from .arclog import (
        FrameView,
        PerceptionLog,
        Transition,
        auditable,
        enabled,
        freeze_grid,
    )
    from .baseline.bounded_search import bounded_bfs
    from .baseline.replay_compare import _rc_differences
    from .local_backend import PROTOCOL, LocalBackend
    from .runtime import TARGET_SECONDS, LevelBudget, PrefixLock, run_expired
    from .sandbox import run_sandbox
except ImportError:
    from agent.arclog import (
        FrameView,
        PerceptionLog,
        Transition,
        auditable,
        enabled,
        freeze_grid,
    )
    from agent.baseline.bounded_search import bounded_bfs
    from agent.baseline.replay_compare import _rc_differences
    from agent.local_backend import PROTOCOL, LocalBackend
    from agent.runtime import TARGET_SECONDS, LevelBudget, PrefixLock, run_expired
    from agent.sandbox import run_sandbox


def _copy(value: Any) -> Any:
    return json.loads(json.dumps(value))


def _state(frame: FrameView) -> dict[str, Any]:
    return {'grid': [list(row) for row in frame.grid], 'level': frame.level, 'state': frame.state}


def _action(name: str, coordinates: tuple[int, int] | None) -> dict[str, Any]:
    return {'action': name, **({'x': coordinates[0], 'y': coordinates[1]} if coordinates else {})}


def _bfs_source() -> str:
    # Remove the trusted helper's host-only typing annotation. The secured
    # worker does not need disk-backed typing or future imports to run BFS.
    tree = ast.parse(inspect.getsource(bounded_bfs))
    function = tree.body[0]
    assert isinstance(function, ast.FunctionDef)
    for argument in function.args.args:
        argument.annotation = None
    if auditable():
        # The embedded trusted helper gets 14 seconds for cooperative search;
        # the existing worker kills/reaps it at 15, including serialization.
        for node in ast.walk(function):
            if isinstance(node, ast.Constant) and type(node.value) is float and node.value == 5.0:
                node.value = 14.0
    return ast.unparse(tree)


def _masked(state: dict[str, Any], mask: frozenset[tuple[int, int]]) -> dict[str, Any]:
    raw = state['grid']
    if (not isinstance(raw, list) or not raw or any(not isinstance(row, list) or not row for row in raw)
            or any(type(cell) is not int for row in raw for cell in row)):
        raise ValueError('step grid must contain nonempty rows of integer colors')
    grid = freeze_grid(state['grid'])
    if type(state['level']) is not int or not isinstance(state['state'], str):
        raise ValueError('step must return grid, integer level and public state name')
    return {'grid': [[None if (x, y) in mask else value for x, value in enumerate(row)]
                     for y, row in enumerate(grid)], 'level': state['level'], 'state': state['state']}


class ProposalValidationError(ValueError):
    """A model proposal failed a pure schema/action/expect check, not a tool."""


class PlanningHalted(RuntimeError):
    """No authorized action: retain evidence and stop instead of live guessing."""


class Playbook:
    """Evidence-backed section memory, with append-only turns per context.

    Full turns append until explicit reset. The external journal and versioned
    playbook retain evidence independently of inference context limits.
    """

    def __init__(self, directory: Path, lock: PrefixLock, token_ceiling: int = 58 * 1024) -> None:
        self.directory = directory
        self.lock = lock
        self.token_ceiling = token_ceiling
        self.facts: dict[str, dict[str, Any]] = {}
        self.working: dict[str, Any] = {}
        self.version = 0
        self.context = 0
        self.messages: list[dict[str, Any]] = []
        self._fresh('No replay evidence yet.')

    def _fresh(self, snapshot: str, retained: list[dict[str, Any]] | None = None) -> None:
        self.context += 1
        self.messages = [{'role': 'system', 'content':
                          'Use observed evidence only. Mechanics require complete replay; '
                          'emit concrete [ACTIONS] with non-HUD expect cells and expect_level_up.\n'
                          + PROTOCOL + '\nPlaybook snapshot:\n' + snapshot}]
        if auditable():
            self.messages[0]['content'] += (
                '\nGeneration 3 contract overrides the example protocol: every actions/[ACTIONS] '
                'response also needs mechanic_source containing def step(state, action). '
                'It must predict ALL logged transitions across levels, including RESET. '
                'Empty history permits only one explicitly disambiguating hypothesis probe. '
                'Simulator requests require simulator_source, never guessed live actions. '
                'Each step returns the complete grid, level and public state. '
                'A failed proposal costs zero actions. Never bypass replay or expect. '
                'Cross-level mechanics and win conditions need step proofs; coordinates, '
                'sub-goals, inert targets and falsified paths belong to working memory.')
        self.messages.extend(_copy(retained or []))
        self.request()

    def request(self) -> list[dict[str, Any]]:
        self.lock.validate(str(self.context), self.messages)
        return _copy(self.messages)

    def remember(self, key: str, **fact: Any) -> None:
        self.facts[key] = _copy(fact)

    def append(self, role: str, content: Any) -> None:
        if auditable() and role == 'user' and isinstance(content, dict):
            if 'state' in content:
                self.working['latest_state'] = _copy(content['state'])
            if content.get('type') in ('replan_rejected', 'live_counterexample', 'prediction_counterexample'):
                self.working.setdefault('falsified_paths', []).append(_copy(content))
        entry = {'role': role, 'content': json.dumps(content, sort_keys=True)}
        # Conservative UTF-8 byte bound: no tokenizer/model assets are assumed.
        # Reset before appending so a ceiling crossed by the latest state
        # request cannot discard that request immediately before inference.
        if len(json.dumps([*self.messages, entry]).encode()) >= self.token_ceiling:
            self.distill('token_ceiling')
        self.messages.append(entry)
        self.request()

    def distill(self, reason: str, *, level_advance: bool = False) -> None:
        if auditable():
            if level_advance:
                self.working.clear()
                # Historical sources remain in log.txt and immutable versions;
                # stale level coordinates and falsifications leave active memory.
                self.facts = {key: value for key, value in self.facts.items()
                              if value.get('status') == 'verified' and value.get('scope') == 'game'}
            if not enabled('ENABLE_PLAYBOOK_RESET') and not level_advance:
                return  # Ablation retains the entire prefix; never FIFO eviction.
            self.version += 1
            lines = [f'# Playbook v{self.version}', f'Reset reason: {reason}',
                     'Historical consistency is evidence, not a universal proof.',
                     '\n## 1. Verified World Model']
            for key, value in sorted(self.facts.items()):
                if value.get('status') != 'verified' or value.get('scope') != 'game':
                    continue
                steps = value.get('proof_steps', [])
                if steps:
                    proof = ', '.join(str(step) for step in steps)
                    lines.append(f'- [Verified: steps {proof}] ' + json.dumps({'id': key, **value}, sort_keys=True))
            lines.append('\n## 2. Current Level Working Memory')
            lines.append(json.dumps(self.working, sort_keys=True))
            lines.extend(json.dumps({'id': key, **value}, sort_keys=True)
                         for key, value in sorted(self.facts.items())
                         if value.get('status') != 'verified' or value.get('scope') != 'game')
            self._write_snapshot('\n'.join(lines)+'\n')
            return
        if not enabled('ENABLE_PLAYBOOK_RESET'):
            retained = self.messages[1:]
            # Memory ablation keeps all exchanges under a fresh prefix identity.
            self._fresh('Persistent section memory:\n' + json.dumps(self.facts, sort_keys=True), retained)
            return
        self.version += 1
        lines = [f'# Playbook v{self.version}', f'Reset reason: {reason}',
                 'Replay consistency is scoped evidence, not a universal proof.']
        for status in ('verified', 'counterexample', 'unresolved'):
            lines.append('\n## ' + status)
            lines.extend(json.dumps({'id': key, **value}, sort_keys=True)
                         for key, value in sorted(self.facts.items()) if value['status'] == status)
        snapshot = '\n'.join(lines) + '\n'
        self._write_snapshot(snapshot)

    def _write_snapshot(self, snapshot: str) -> None:
        self.directory.mkdir(parents=True, exist_ok=True)
        (self.directory / f'playbook.v{self.version:04d}.md').write_text(snapshot)
        temporary = self.directory / 'playbook.tmp'
        temporary.write_text(snapshot)
        temporary.replace(self.directory / 'playbook.md')
        self._fresh(snapshot)


class WorldModel:
    """Host-owned evidence gates. Only this object issues live plan/probe tickets."""

    def __init__(self, perception: PerceptionLog, planner: Callable[[str], dict[str, Any]],
                 directory: Path, lock: PrefixLock) -> None:
        self.log = perception
        self.planner = planner
        self.memory = Playbook(directory, lock)
        self.directory = directory
        self.events: list[dict[str, Any]] = []
        self.queue: deque[dict[str, Any]] = deque()
        self.pending: dict[str, Any] | None = None
        self.queue_scope: tuple[int, int, int] | None = None
        self.candidates: dict[str, dict[str, Any]] = {}
        self.wasted_moves = 0
        self.backend: Callable[[list[dict[str, Any]]], dict[str, Any]] | None = None
        self._seen = 0
        self._tier_attempt: tuple[int, int] | None = None
        self.queue_kind = 'plan'
        self._tier_failures = 0
        self.halted = False

    def tier_due(self) -> bool:
        if not enabled('ENABLE_TIER1_SIMULATOR') or self.log.current.level < 3:
            return False
        if not auditable():
            return self.wasted_moves >= 40
        history = [t for t in self.log.records if t.before.level == self.log.current.level]
        return (sum(not t.reset for t in history) >= 40
                or any(t.reset and t.before.state != 'NOT_PLAYED' for t in history))

    def event(self, kind: str, **payload: Any) -> None:
        entry = {'type': kind, 'step': self.log.current.step, 'live_action': False, **payload}
        self.events.append(_copy(entry))
        self.directory.mkdir(parents=True, exist_ok=True)
        with (self.directory / 'world_model.jsonl').open('a') as stream:
            stream.write(json.dumps(entry) + '\n')
        self.memory.append('user', entry)

    def _worker(self, source: str, body: str) -> dict[str, Any]:
        # Candidate namespace is separate from the verification driver. The host
        # checks output shape, coverage and cells; a candidate cannot grant itself
        # permission by returning a boolean "verified" flag.
        program = ('import json\n'
                   f'namespace = {{}}\nexec(compile({source!r}, "<mechanic>", "exec"), namespace)\n'
                   'step = namespace["step"]\n' + body)
        result = self.planner(program)
        if not result.get('ok'):
            return {'ok': False, 'error': result}
        try:
            lines = result['stdout'].splitlines()
            if len(lines) != 1:
                raise ValueError('worker must return exactly one JSON result')
            return {'ok': True, 'value': json.loads(lines[0])}
        except (ValueError, KeyError, TypeError) as error:
            return {'ok': False, 'error': 'invalid_worker_output: ' + str(error)}

    def _relevant(self, scope: str) -> list[Transition]:
        if scope not in ('level', 'game'):
            raise ValueError('scope must be level or game')
        if auditable():
            return [t for t in self.log.history()
                    if scope == 'game' or t.before.level == self.log.current.level]
        return [t for t in self.log.records if not t.reset and not t.full_reset
                and t.before.grid and json.loads(t.evidence_json).get('rendered_response')
                and (scope == 'game' or t.before.level == self.log.current.level)]

    def replay_transitions(self, source: str, *, scope: str = 'level', chunk_size: int = 16,
                           timeout_s: float = 30.0) -> dict[str, Any]:
        """Full-history upgrade of archived replay; no tail scan or case clamp.

        Empty/limited/crashed/malformed replay never grants a complete pass.
        Chunk outputs contain complete successor states; comparison is host-owned.
        Reset/unrendered evidence is counted as excluded, never silently matched.
        """
        if not 1 <= chunk_size <= 64 or not 0 < timeout_s <= 120:
            raise ValueError('invalid replay limits')
        records = self._relevant(scope)
        digest = hashlib.sha256(json.dumps([t.to_dict() for t in records], sort_keys=True).encode()).hexdigest()
        report: dict[str, Any] = {
            'status': 'empty', 'complete': False, 'scope': scope, 'level': self.log.current.level,
            'source_hash': hashlib.sha256(source.encode()).hexdigest(), 'history_hash': digest,
            'history_count': len(self.log.records), 'eligible': len(records),
            'excluded': len(self.log.records)-len(records), 'checked': 0, 'matched': 0,
            'errors': 0, 'chunks': [], 'counterexamples': [], 'live_actions': 0}
        began = time.monotonic()
        for offset in range(0, len(records), chunk_size):
            if time.monotonic()-began >= timeout_s:
                report['status'] = 'limited'
                break
            chunk = records[offset:offset+chunk_size]
            evidence = [{'step': t.step, 'before': _state(t.before),
                         'action': _action(t.action, t.coordinates),
                         'intermediate': t.intermediate, 'hud_mask': sorted(t.hud_mask),
                         'reset': t.reset, 'full_reset': t.full_reset,
                         'rendered_response': json.loads(t.evidence_json).get('rendered_response')}
                        for t in chunk]
            body = (f'log_history = json.loads({json.dumps(evidence)!r})\n'
                    'def verify_hypothesis(log_history):\n'
                    '    return [step(t["before"], t["action"]) for t in log_history]\n'
                    'print(json.dumps(verify_hypothesis(log_history)))\n')
            result = self._worker(source, body)
            predictions = result.get('value')
            if (not result['ok'] or not isinstance(predictions, list)
                    or len(predictions) != len(chunk)):
                report.update(status='error', errors=report['errors']+1, failure=result)
                break
            try:
                for transition, predicted in zip(chunk, predictions, strict=True):
                    expected = _masked(predicted, transition.hud_mask)
                    actual = _masked(_state(transition.after), transition.hud_mask)
                    differences, count = _rc_differences(expected, actual)
                    report['checked'] += 1
                    if count:
                        report['counterexamples'].append({'step': transition.step,
                            'expected_cells': differences, 'actual_state': actual,
                            'difference_count': count})
                        if auditable():
                            break
                    else:
                        report['matched'] += 1
            except (KeyError, TypeError, ValueError, IndexError) as error:
                report.update(status='error', errors=report['errors']+1, failure=str(error))
                break
            report['chunks'].append({'offset': offset, 'checked': len(chunk)})
            if auditable() and report['counterexamples']:
                break
        if report['status'] not in ('error', 'limited'):
            report['complete'] = bool(records) and report['checked'] == len(records)
            report['status'] = ('counterexample' if report['counterexamples'] else
                                'pass' if report['complete'] else 'empty')
        if time.monotonic()-began >= timeout_s:
            report.update(status='limited', complete=False)
        self.event('replay', report=report)
        return report

    def _cells(self, value: Any, *, required: bool = True) -> list[dict[str, int]]:
        if isinstance(value, dict):
            if any(not isinstance(xy, tuple) or len(xy) != 2 for xy in value):
                raise ProposalValidationError('expect mapping keys must be x/y pairs')
            value = [{'x': xy[0], 'y': xy[1], 'value': color} for xy, color in value.items()]
        if not isinstance(value, list) or (required and not value):
            raise ProposalValidationError('non-empty expected settled non-HUD cells required')
        grid, mask = self.log.current.grid, self.log.get_hud_mask()
        seen: set[tuple[int, int]] = set()
        for cell in value:
            if not isinstance(cell, dict) or set(cell) != {'x', 'y', 'value'}:
                raise ProposalValidationError('expect needs x, y, value cells')
            x, y, color = cell['x'], cell['y'], cell['value']
            if (any(type(n) is not int for n in (x, y, color)) or not 0 <= color < 16
                    or not 0 <= y < len(grid) or not 0 <= x < len(grid[y])
                    or (x, y) in mask or (x, y) in seen):
                raise ProposalValidationError('invalid, duplicate or HUD prediction cell')
            seen.add((x, y))
        return _copy(value)

    def _plan_item(self, item: dict[str, Any], legal: list[GameAction]) -> dict[str, Any]:
        if not isinstance(item, dict):
            raise ProposalValidationError('concrete action object required')
        name = item.get('action')
        if not isinstance(name, str) or name not in GameAction.__members__:
            raise ProposalValidationError('concrete framework action enum required')
        action = GameAction[name]
        if action is GameAction.RESET or action not in legal:
            raise ProposalValidationError('unavailable action or reset in plan')
        result: dict[str, Any] = {'action': name}
        if action.is_complex():
            x, y = item.get('x'), item.get('y')
            grid = self.log.current.grid
            if (type(x) is not int or type(y) is not int or not 0 <= y < len(grid)
                    or not 0 <= x < len(grid[y])):
                raise ProposalValidationError('concrete in-board x/y required')
            result.update(x=x, y=y)
        elif 'x' in item or 'y' in item:
            raise ProposalValidationError('simple action cannot carry coordinates')
        result['expect'] = self._cells(item.get('expect', []), required=enabled('ENABLE_EXPECT_QUEUE'))
        up = item.get('expect_level_up', False if not enabled('ENABLE_EXPECT_QUEUE') else None)
        if type(up) is not bool:
            raise ProposalValidationError('explicit expect_level_up boolean required')
        result['expect_level_up'] = up
        return result

    def execute_plan(self, actions: list[dict[str, Any]], legal: list[GameAction]) -> None:
        if auditable():
            raise ProposalValidationError('plan requires a completely replayed mechanic_source')
        self._install_plan(actions, legal)

    def _install_plan(self, actions: list[dict[str, Any]], legal: list[GameAction],
                      *, kind: str = 'plan') -> None:
        """Install an immutable concrete queue; actual dispatch stays in the host.

        Like checked_execute, terminal flags interrupt before old-level decoding.
        No callback can dispatch from within replay/search's isolated process.
        """
        if self.pending is not None:
            raise ValueError('cannot replace plan awaiting settled observation')
        if not isinstance(actions, list) or not 1 <= len(actions) <= 128:
            raise ProposalValidationError('plan requires 1..128 concrete actions')
        plan = [self._plan_item(item, legal) for item in actions]
        self.queue = deque(plan)
        self.queue_kind = kind
        current = self.log.current
        self.queue_scope = (current.level, current.episode, current.step)
        self.event('plan_queued', count=len(plan))

    def flush(self, reason: str) -> None:
        count = len(self.queue) + int(self.pending is not None)
        self.queue.clear()
        self.pending = None
        self.queue_scope = None
        if count:
            self.event('queue_flushed', reason=reason, remaining=count)

    def next_action(self, legal: list[GameAction]) -> dict[str, Any] | None:
        current = self.log.current
        if self.pending is not None:
            raise RuntimeError('settled observation required before another dispatch')
        if not self.queue:
            return None
        if auditable() and self.tier_due() and self.queue_kind != 'simulator':
            self.flush('simulator_required')
            return None
        if self.queue_scope != (current.level, current.episode, current.step):
            self.flush('stale_state')
            return None
        try:
            item = self._plan_item(self.queue[0], legal)
        except ValueError:
            self.flush('plan_no_longer_admissible')
            return None
        stored = self.queue.popleft()
        item['probe'] = bool(stored.get('probe'))
        item['simulator'] = self.queue_kind == 'simulator'
        if 'mechanic_key' in stored:
            item['mechanic_key'] = stored['mechanic_key']
        self.pending = item
        return _copy(item)

    def observe(self, transition: Transition) -> None:
        if transition.step <= self._seen:
            return
        self._seen = transition.step
        boundary = transition.reset or transition.full_reset or transition.level_changed or transition.terminal
        if self.pending is not None:
            item = self.pending
            action_matches = _action(transition.action, transition.coordinates) == {
                k: v for k, v in item.items() if k in ('action', 'x', 'y')}
            cells = item['expect']
            mask = transition.hud_mask
            grid = transition.after.grid
            match = action_matches and bool(transition.level_changed) == item['expect_level_up']
            match = match and all((c['x'], c['y']) not in mask
                                 and c['y'] < len(grid) and c['x'] < len(grid[c['y']])
                                 and grid[c['y']][c['x']] == c['value'] for c in cells)
            self.event('prediction', matched=match, predicted_cells=len(cells),
                       probing=bool(item.get('probe')), boundary=boundary)
            if auditable() and not match:
                self.event('prediction_counterexample', transition=transition.step,
                           action=_action(transition.action, transition.coordinates),
                           expected_cells=cells, actual_state=_state(transition.after))
            if not match and item.get('mechanic_key') in self.memory.facts:
                key = item['mechanic_key']
                fact = _copy(self.memory.facts[key])
                fact['status'] = 'counterexample'
                fact['counterexamples'].append({'step': transition.step,
                    'expected_cells': cells, 'actual_state': _state(transition.after)})
                self.memory.remember(key, **fact)
                self.candidates[key]['status'] = 'counterexample'
                self.event('live_counterexample', mechanic_key=key, transition=transition.step)
            if boundary or (enabled('ENABLE_EXPECT_QUEUE') and not match):
                self.flush('boundary' if boundary else 'cell_mismatch')
            else:
                self.pending = None
                self.queue_scope = (transition.after.level, transition.after.episode, transition.after.step)
        elif boundary:
            self.flush('boundary')
        if boundary:
            self.wasted_moves = 0
            self._tier_attempt = None
            if transition.level_changed:
                self._tier_failures = 0
        elif transition.zero_causal_diff:
            self.wasted_moves += 1
        # Record observations separately; a replay pass remains explicitly tied
        # to its evidence revision and must be refreshed before a new probe.
        self.memory.append('user', {'transition': transition.step, 'level': transition.before.level,
                                  'diff': [vars(cell) for cell in transition.settled],
                                  'transient_cells': len(transition.transient), 'boundary': boundary})
        if auditable():
            self.memory.working.update(latest_state=_state(transition.after),
                                       dead_signatures=self.log.dead_signatures)
        if boundary:
            self.memory.distill('level_boundary' if transition.level_changed else 'reset_or_terminal',
                                level_advance=transition.level_changed)

    def _predict(self, source: str, action: dict[str, Any]) -> dict[str, Any]:
        result = self._worker(source, f'print(json.dumps(step(json.loads({json.dumps(_state(self.log.current))!r}), '
                              f'json.loads({json.dumps(action)!r}))))\n')
        if not result['ok'] or not isinstance(result.get('value'), dict):
            raise ValueError('prediction failed: ' + str(result))
        _masked(result['value'], self.log.get_hud_mask())
        return result['value']

    def propose_hypothesis(self, source: str, probe: dict[str, Any], alternative_expect: Any,
                           legal: list[GameAction], *, scope: str = 'level') -> dict[str, Any]:
        """Replay first, then authorize exactly one differing, evidence-seeking probe."""
        if auditable():
            scope = 'game'
        source_hash = hashlib.sha256(source.encode()).hexdigest()
        key = f'{source_hash}:{scope}:{self.log.current.level if scope == "level" else "all"}'
        report = (self.replay_transitions(source, scope=scope) if enabled('ENABLE_RETRODICTION')
                  else {'status': 'ablation_bypass', 'complete': False})
        if auditable() and report['status'] == 'empty' and not self.log.records:
            # Vacuous historical consistency buys one explicit bootstrap probe,
            # never a verified rule or a multi-step plan.
            report.update(status='bootstrap', complete=True)
        status = ('verified' if report['status'] == 'pass' else 'counterexample'
                  if report['status'] == 'counterexample' else 'unresolved')
        fact = {'status': status, 'scope': scope, 'level': self.log.current.level,
                'source': source, 'report': report}
        self.candidates[key] = fact
        self.directory.mkdir(parents=True, exist_ok=True)
        source_path = self.directory / f'mechanic.{source_hash}.py'
        source_path.write_text(source)
        self.memory.remember(key, status=status, scope=scope, level=self.log.current.level,
                             source_file=source_path.name, mechanic_source=source, evidence={
                                 k: report[k] for k in ('status', 'complete', 'history_hash', 'checked', 'eligible')
                                 if k in report},
                             proof_steps=[t.step for t in self._relevant(scope)]
                             if status == 'verified' else [],
                             counterexamples=[{'step': c['step'], 'expected_cells': c['expected_cells']}
                                              for c in report.get('counterexamples', [])[:1]])
        if enabled('ENABLE_RETRODICTION') and not (report['status'] in ('pass', 'bootstrap') and report['complete']):
            self.event('hypothesis_rejected', source_hash=source_hash, reason=report['status'], live_actions=0)
            return report
        item = self._plan_item(probe, legal)
        alternative = self._cells(alternative_expect)
        if not item['expect'] or { (c['x'], c['y']) for c in alternative } != {
                (c['x'], c['y']) for c in item['expect']}:
            raise ProposalValidationError('probe alternatives must predict the same nonempty cells')
        predicted = self._predict(source, {k: v for k, v in item.items() if k in ('action', 'x', 'y')})
        if (not all(predicted['grid'][c['y']][c['x']] == c['value'] for c in item['expect'])
                or (predicted['level'] > self.log.current.level) != item['expect_level_up']):
            raise ProposalValidationError('probe expectation disagrees with mechanic')
        if sorted(alternative, key=lambda c: (c['x'], c['y'])) == sorted(item['expect'], key=lambda c: (c['x'], c['y'])):
            raise ProposalValidationError('probe must disambiguate concrete predictions')
        self._install_plan([item], legal, kind='probe')
        self.queue[0]['probe'] = True
        self.queue[0]['mechanic_key'] = key
        self.event('probe_authorized', source_hash=source_hash, count=1, replay_status=report['status'])
        return report

    def _lookup_source(self) -> str:
        # Conservative synthesis from observation only: no invented unknown edges.
        edges = [{'before': _state(t.before), 'action': _action(t.action, t.coordinates),
                  'after': _state(t.after)} for t in self._relevant('level')]
        return (f'edges = {edges!r}\n'
                'def step(state, action):\n'
                '    matches = [e["after"] for e in edges if e["before"] == state and e["action"] == action]\n'
                '    if matches and all(m == matches[0] for m in matches):\n'
                '        return matches[0]\n'
                '    return None\n')

    def verified_actions(self, source: Any, actions: list[dict[str, Any]],
                         legal: list[GameAction], *, kind: str = 'plan') -> dict[str, Any]:
        """Replay the journal, then validate the entire forward rollout offline."""
        if not isinstance(actions, list) or not 1 <= len(actions) <= 128:
            raise ProposalValidationError('plan requires 1..128 concrete actions')
        plan = [self._plan_item(item, legal) for item in actions]
        if not isinstance(source, str) or not source.strip():
            raise ProposalValidationError('plan requires mechanic_source with def step(state, action)')
        report = self.replay_transitions(source, scope='game')
        if report['status'] != 'pass' or not report['complete']:
            self.event('hypothesis_rejected', reason=report['status'], live_actions=0)
            return report
        predictions: list[dict[str, Any]] = []
        current = _state(self.log.current)
        # Keep long full-board rollouts below the worker's 1 MiB output cap.
        for offset in range(0, len(plan), 16):
            chunk = plan[offset:offset+16]
            body = (f'state = json.loads({json.dumps(current)!r})\n'
                    f'actions = json.loads({json.dumps(chunk)!r})\n'
                    'predictions = []\n'
                    'for item in actions:\n'
                    '    state = step(json.loads(json.dumps(state)), '
                    '{k: v for k, v in item.items() if k in ("action", "x", "y")})\n'
                    '    predictions.append(json.loads(json.dumps(state)))\n'
                    'print(json.dumps(predictions))\n')
            result = self._worker(source, body)
            values = result.get('value')
            if not result['ok'] or not isinstance(values, list) or len(values) != len(chunk):
                raise RuntimeError('forward simulator failed: ' + str(result))
            predictions.extend(values)
            current = values[-1]
        level = self.log.current.level
        mask = self.log.get_hud_mask()
        for index, (item, state) in enumerate(zip(plan, predictions, strict=True)):
            _masked(state, mask)
            grid = state['grid']
            if (any(c['y'] >= len(grid) or c['x'] >= len(grid[c['y']])
                    or grid[c['y']][c['x']] != c['value'] for c in item['expect'])
                    or (state['level'] > level) != item['expect_level_up']):
                raise ProposalValidationError(f'plan expectation disagrees with mechanic at index {index}')
            if (state['level'] != level or state['state'] in ('WIN', 'GAME_OVER')) and index < len(plan)-1:
                raise ProposalValidationError('plan must end at level or terminal boundary')
            # Check all settled board cells, so a partial declared expect cannot
            # conceal a counterexample elsewhere on the board.
            item['expect'] = [{'x': x, 'y': y, 'value': value} for y, row in enumerate(grid)
                              for x, value in enumerate(row) if (x, y) not in mask]
            level = state['level']
        digest = hashlib.sha256(source.encode()).hexdigest()
        key = f'{digest}:game:all'
        path = self.directory / f'mechanic.{digest}.py'
        path.write_text(source)
        fact = {'status': 'verified', 'scope': 'game', 'source_file': path.name,
                'mechanic_source': source,
                'proof_steps': [t.step for t in self._relevant('game')],
                'evidence': {k: report[k] for k in ('history_hash', 'checked', 'eligible')},
                'counterexamples': []}
        self.memory.remember(key, **fact)
        self.candidates[key] = _copy(fact)
        self._install_plan(plan, legal, kind=kind)
        for item in self.queue:
            item['mechanic_key'] = key
        return report

    def search_plan(self, source: str, legal: list[GameAction], *, tier: int = 1) -> dict[str, Any]:
        """Compile and replay current-level history before bounded sandbox BFS."""
        compile_result = self._worker(source, 'assert callable(step)\nprint(json.dumps(True))\n')
        self.event('simulator_compile', success=compile_result['ok'], result=compile_result)
        if not compile_result['ok']:
            return {'status': 'compile_failed'}
        # Simulator validation is mandatory even with the general replay ablated.
        replay = self.replay_transitions(source, scope='level')
        self.event('simulator_replay', success=replay['status'] == 'pass', report=replay)
        if replay['status'] != 'pass' or not replay['complete']:
            return {'status': 'replay_failed', 'replay': replay}
        actions = [_action(a.name, None) for a in legal if not a.is_complex()]
        actions.extend(_action(t.action, t.coordinates) for t in self._relevant('level')
                       if GameAction[t.action] in legal and t.coordinates is not None)
        if auditable():
            grid, mask = self.log.current.grid, self.log.get_hud_mask()
            targets = ([(x, y) for y, row in enumerate(grid) for x in range(len(row))
                        if (x, y) not in mask] if tier >= 2 else
                       [(int(o.centroid[0]), int(o.centroid[1]))
                        for o in self.log.get_board_objects()])
            actions.extend(_action(a.name, xy) for a in legal if a.is_complex()
                           for xy in targets if xy not in mask)
        actions = list({json.dumps(a, sort_keys=True): a for a in actions}.values())
        limits = ('max_nodes=50000, max_depth=128, max_edges=250000, timeout_s=14.0'
                  if auditable() else 'max_nodes=2000, max_depth=32, max_edges=10000, timeout_s=2.0')
        mask_source = f'mask = {set(self.log.get_hud_mask())!r}\n'
        compact_cells = ('    cells = [c for c in cells if (c["x"], c["y"]) not in mask][:1]\n'
                         if auditable() else '')
        # Reuse the extracted BFS verbatim, providing already-bootstrapped aliases.
        body = ('from collections import deque as _bfs_deque\nfrom time import monotonic as _bfs_clock\n'
                '_bfs_copy = lambda value: json.loads(json.dumps(value))\n'
                '_bfs_finite = __import__("math").isfinite\n' + _bfs_source() + '\n'
                f'actions = json.loads({json.dumps(actions)!r})\n'
                f'start = json.loads({json.dumps(_state(self.log.current))!r})\n'
                'def successors(state):\n'
                '    for action in actions:\n'
                '        candidate = step(_bfs_copy(state), _bfs_copy(action))\n'
                '        if candidate is not None:\n'
                '            yield action, candidate\n'
                'report = bounded_bfs(start, successors, lambda s: s["level"] > start["level"] '
                'or s["state"] == "WIN", key=lambda s: json.dumps(s, sort_keys=True), '
                + limits + ')\n'
                'plan = []\nstate = start\n' + mask_source +
                'for action in report.get("actions") or []:\n'
                '    after = step(_bfs_copy(state), _bfs_copy(action))\n'
                '    cells = [{"x": x, "y": y, "value": v} for y, row in enumerate(after["grid"]) '
                'for x, v in enumerate(row)]\n' + compact_cells +
                '    plan.append(dict(action, expect=cells, expect_level_up=after["level"] > state["level"]))\n'
                '    state = after\n'
                'report["plan"] = plan\nprint(json.dumps(report))\n')
        result = self._worker(source, body)
        if not result['ok']:
            self.event('search_failed', result=result)
            return {'status': 'search_error'}
        report = result['value']
        self.event('search', report=report)
        if report['status'] == 'found' and report['plan']:
            mask = self.log.get_hud_mask()
            for item in report['plan']:
                item['expect'] = [c for c in item['expect'] if (c['x'], c['y']) not in mask]
            if auditable():
                proof = self.verified_actions(source, report['plan'], legal, kind='simulator')
                if proof['status'] != 'pass':
                    return {'status': 'history_rejected', 'replay': proof}
            else:
                self.execute_plan(report['plan'], legal)
        return report

    def reason(self, legal: list[GameAction]) -> None:
        """Backend integration boundary: validate immutable prefix before inference."""
        tier_due = self.tier_due()
        response: dict[str, Any] = {}
        if self.backend is not None:
            self.memory.append('user', {'request': 'simulator' if tier_due else 'hypotheses_or_actions',
                                      'state': _state(self.log.current),
                                      'available_actions': [a.name for a in legal],
                                      'objects': [o.node() for o in self.log.get_board_objects()],
                                      'hud_mask': sorted(self.log.get_hud_mask()),
                                      'dead_signatures': self.log.dead_signatures,
                                      'simulator_tier': min(2, self._tier_failures+1) if tier_due else None,
                                      'transition_count': len(self.log.records)})
            response = self.backend(self.memory.request())
            self.memory.append('assistant', response)
            if not isinstance(response, dict):
                raise ProposalValidationError('structured proposal object required')
            if auditable() and isinstance(response.get('working_memory'), dict):
                for key in ('active_coordinates', 'sub_goals', 'falsified_paths'):
                    if key in response['working_memory']:
                        self.memory.working[key] = _copy(response['working_memory'][key])
        if tier_due:
            if auditable():
                self.flush('simulator_required')
                tier = min(2, self._tier_failures+1)
                source = response.get('simulator_source')
                if not isinstance(source, str) or not source.strip():
                    raise ProposalValidationError('live moves halted: synthesize simulator_source with def step')
                self.event('tier1_required' if tier == 1 else 'tier2_required',
                           level_actions=sum(t.before.level == self.log.current.level and not t.reset
                                             for t in self.log.records), live_actions=0)
                report = self.search_plan(source, legal, tier=tier)
                if report['status'] != 'found' or not self.queue:
                    self._tier_failures += 1
                    self.event('simulator_rejected', tier=tier, report=report, live_actions=0)
                    raise ProposalValidationError('simulator failed: ' + json.dumps(report))
                return
            revision = (self.log.current.level, len(self.log.records))
            if self._tier_attempt != revision:
                self._tier_attempt = revision
                source = response.get('simulator_source') or self._lookup_source()
                self.event('tier1_required', wasted_moves=self.wasted_moves,
                           synthesis='backend' if response.get('simulator_source') else 'observed_lookup')
                report = self.search_plan(source, legal)
                if report['status'] != 'found':
                    self.event('fallback', reason='tier1_' + report['status'])
            return
        hypotheses = response.get('hypotheses', [])
        if not isinstance(hypotheses, list):
            raise ProposalValidationError('hypotheses must be a list')
        for hypothesis in hypotheses:
            if (not isinstance(hypothesis, dict)
                    or not isinstance(hypothesis.get('source'), str)
                    or not isinstance(hypothesis.get('probe'), dict)
                    or 'alternative_expect' not in hypothesis
                    or hypothesis.get('scope', 'level') not in ('level', 'game')):
                raise ProposalValidationError('hypothesis requires source, probe, alternative_expect and valid scope')
            self.propose_hypothesis(hypothesis['source'], hypothesis['probe'],
                                    hypothesis['alternative_expect'], legal,
                                    scope=hypothesis.get('scope', 'level'))
            if self.queue:
                return
        if response.get('hypotheses'):
            # Rejected mechanics cannot fall through to an unchecked live probe
            # disguised as a second response field.
            return
        if response.get('actions'):
            if auditable():
                self.verified_actions(response.get('mechanic_source'), response['actions'], legal)
            else:
                self.execute_plan(response['actions'], legal)
        elif auditable() and response.get('[ACTIONS]'):
            self.verified_actions(response.get('mechanic_source'), response['[ACTIONS]'], legal)


class MyAgent(Agent):
    """Framework adapter; perception owns actions, world model owns reasoning."""

    # Upper bound on actions per game; the framework also enforces global limits.
    MAX_ACTIONS = 80

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        # A fixed control seed supports paired ablations without shared RNG state.
        seed = int(time.time() * 1_000_000) + hash(self.game_id) % 1_000_000
        self._rng = random.Random(int(os.getenv("ARC3_SEED", str(seed))))
        log_root = Path(os.getenv("ARC3_LOG_DIR", "scratch"))
        log_root.mkdir(parents=True, exist_ok=True)
        self.perception = PerceptionLog(Path(tempfile.mkdtemp(prefix="perception-", dir=log_root)) / "log.txt")
        self._perception_started = False
        self._recovery_mode = False
        self._observation: FrameData | None = None
        self._dispatch_approval: tuple[str, tuple[int, int] | None, int] | None = None
        self._reasoning_deadline: float | None = None
        self.prefix_lock = PrefixLock()
        self.level_budget: LevelBudget | None = None
        assert self.perception.log_path is not None
        self.world = WorldModel(self.perception, self.execute_planner,
                                self.perception.log_path.parent, self.prefix_lock)
        if enabled('ENABLE_OFFLINE_BACKEND') and os.getenv('ARC3_LLM_READY') == '1':
            self.world.backend = LocalBackend(self.perception.log_path.parent, self._inference_limits,
                                             lambda: str(self.world.memory.context))
            self.world.event('backend_connected', model='flashnext', endpoint='loopback:8001')
        else:
            self.world.event('backend_unavailable', reason=os.getenv('ARC3_LLM_FAILURE', 'not_ready_or_disabled'),
                             mode='halt_until_verified' if auditable() else 'exploration')

    def _inference_limits(self) -> tuple[int, float]:
        remaining = (float(os.getenv('ARC3_RUN_STARTED', str(time.time())))
                     + TARGET_SECONDS - time.time())
        tokens = 4096
        if self.level_budget is not None:
            remaining = min(remaining, self.level_budget.remaining())
            tokens = self.level_budget.generation_tokens(tokens)
        if self._reasoning_deadline is not None:
            remaining = min(remaining, self._reasoning_deadline - time.monotonic())
        return tokens, max(0.0, min(30.0, remaining))

    def _reason_with_replanning(self, legal: list[GameAction]) -> None:
        # All strict proposals, including simulator revisions, share one
        # bounded zero-action repair allowance. The control keeps its old guard.
        tier_due = self.world.tier_due()
        if (not enabled('ENABLE_ZERO_ACTION_REPLAN') or self.world.backend is None
                or (tier_due and not auditable())):
            self.world.reason(legal)
            return
        if self.world.pending is not None:
            raise RuntimeError('cannot replan while awaiting a settled observation')
        _, allowance = self._inference_limits()
        self._reasoning_deadline = time.monotonic() + allowance
        try:
            for attempt in range(1, 4):
                tokens, remaining = self._inference_limits()
                if tokens <= 0 or remaining <= 0:
                    self.world.flush('replan_budget_exhausted')
                    self.world.event('replan_exhausted', attempt=attempt-1, reason='budget', live_actions=0)
                    raise TimeoutError('zero-action replanning budget exhausted')
                self.world.event('replan_attempt', attempt=attempt, max_attempts=3, live_actions=0)
                before = len(self.world.events)
                error: str | None = None
                try:
                    self.world.reason(legal)
                except ProposalValidationError as invalid:
                    error = str(invalid)
                events = self.world.events[before:]
                rejected = [event for event in events if event['type'] == 'hypothesis_rejected']
                # A later malformed proposal or valid queue cannot reclassify
                # an earlier worker/limited replay failure as model repair.
                if any(event['reason'] not in ('counterexample', 'empty') for event in rejected):
                    self.world.flush('replan_tool_failure')
                    return
                if self.world.queue:
                    tokens, remaining = self._inference_limits()
                    if tokens <= 0 or remaining <= 0:
                        self.world.flush('replan_budget_exhausted')
                        self.world.event('replan_exhausted', attempt=attempt, reason='budget', live_actions=0)
                        raise TimeoutError('zero-action replanning budget exhausted')
                    self.world.event('replan_accepted', attempt=attempt, live_actions=0)
                    return
                if error is None and not rejected:
                    return
                counterexamples = [counterexample for event in events if event['type'] == 'replay'
                                   for counterexample in event['report'].get('counterexamples', [])]
                self.world.flush('replan_rejected')
                self.world.event('replan_rejected', attempt=attempt, reason=error or 'hypotheses_rejected',
                                 counterexamples=counterexamples[:2], live_actions=0)
            self.world.event('replan_exhausted', attempt=3, reason='attempt_cap', live_actions=0)
        finally:
            self._reasoning_deadline = None

    def execute_planner(self, source: str) -> dict[str, Any]:
        root = Path(__file__).resolve().parents[1]
        python = Path(os.getenv("ARC3_SANDBOX_PYTHON", str(root / ".venv_agent_sandbox/bin/python")))
        timeout = 15.0
        if self._reasoning_deadline is not None:
            _, remaining = self._inference_limits()
            if remaining <= 0:
                raise TimeoutError('zero-action replanning budget exhausted before worker')
            timeout = min(timeout, remaining)
        result = run_sandbox(source, python=python, cwd=root, timeout=timeout)
        if not result["ok"]:
            logging.getLogger(__name__).warning("planner failure: %s", result)
        return result

    @property
    def name(self) -> str:
        return f"{super().name}.{self.MAX_ACTIONS}"

    def main(self) -> None:
        try:
            super().main()
        except PlanningHalted as error:
            # The framework has no pause action. End this game explicitly and
            # close its public scorecard without turning a failed planner into
            # another live move or taking down other games in the runner.
            logging.getLogger(__name__).warning('planning halted: %s', error)
            self.cleanup()

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        # Stop once we win. Don't stop on GAME_OVER — we want to RESET and retry.
        return run_expired() or latest_frame.state is GameState.WIN or self.world.halted

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        try:
            if self._recovery_mode:
                return self._safe_fallback(latest_frame)
            # Public level metadata supplies weights; an unavailable count defers
            # scheduling rather than inventing an environment structure.
            if latest_frame.win_levels > 0:
                if self.level_budget is None:
                    self.level_budget = LevelBudget(
                        env_seconds=float(os.getenv("ARC3_ENV_SECONDS", str(TARGET_SECONDS))),
                        levels=latest_frame.win_levels,
                        run_started=float(os.getenv("ARC3_RUN_STARTED", str(time.time()))),
                    )
                level = min(latest_frame.levels_completed + 1, latest_frame.win_levels)
                if level > self.level_budget.level:
                    self.level_budget.begin_level(level)

            self._ensure_perception(latest_frame)
            for transition in self.perception.records:
                if transition.step > self.world._seen:
                    self.world.observe(transition)
            planned = None
            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                self.world.flush('reset_proposal')
                proposal = GameAction.RESET
            else:
                legal = self._legal_actions(latest_frame)
                if not legal:
                    raise RuntimeError("no advertised non-reset action; replan required")
                planned = self.world.next_action(legal)
                if planned is None:
                    try:
                        self._reason_with_replanning(legal)
                    except Exception as error:
                        self.world.flush('reasoning_failure')
                        self.world.event('reasoning_failure', error=type(error).__name__, detail=str(error))
                        raise
                    planned = self.world.next_action(legal)
                if planned is not None:
                    proposal = GameAction[planned['action']]
                    if proposal.is_complex():
                        proposal.set_data({'x': planned['x'], 'y': planned['y']})
                else:
                    if auditable():
                        raise PlanningHalted('no admissible evidence-gated plan; live exploration halted')
                    if self.world.backend is not None:
                        raise RuntimeError('backend produced no admissible evidence-gated plan')
                    self.world.event('fallback', reason='no_backend_or_validated_route',
                                     wasted_moves=self.world.wasted_moves)
                    proposal = self._sample(self._rng.choice(legal))
            action: Any = self._resolve_action(proposal, latest_frame)
            if planned is not None and _action(action.name, self._coordinates(action)) != {
                    k: v for k, v in planned.items() if k in ('action', 'x', 'y')}:
                self.world.flush('perception_blocked_plan')
                raise RuntimeError('planned action blocked; replan before dispatch')
            self._dispatch_approval = (action.name, self._coordinates(action), self.perception.current.step)
            action.reasoning = {"why": "expect-checked plan" if planned else "visible exploration fallback",
                                "evidence_step": self.perception.current.step,
                                "context_id": self.world.memory.context}
            if self.level_budget is not None and self.level_budget.generation_tokens(1) == 0:
                action.reasoning["budget"] = "level reasoning budget exhausted; best-effort action"
            return action
        except Exception as e:
            if auditable():
                self.world.halted = True
                self.world.queue.clear()
                self.world.pending = None
                self.world.queue_scope = None
                self._dispatch_approval = None
                self._log_action_failure(e)
                raise PlanningHalted('live moves halted: ' + str(e)) from e
            if not enabled('ENABLE_ACTION_FALLBACK'):
                raise
            # Do not call world.flush/event or perception here: these may be
            # exactly the subsystem that failed. No stale plan may dispatch.
            self._recovery_mode = True
            self._dispatch_approval = None
            self.world.queue.clear()
            self.world.pending = None
            self.world.queue_scope = None
            self._log_action_failure(e)
            return self._safe_fallback(latest_frame)

    @staticmethod
    def _log_action_failure(error: Exception) -> None:
        detail = ''.join(traceback.format_exception(error))
        try:
            directory = Path(os.getenv('ARC3_LOG_DIR', 'scratch'))
            directory.mkdir(parents=True, exist_ok=True)
            with (directory / 'choose_action_errors.log').open('a', encoding='utf-8') as stream:
                stream.write(detail + '\n')
        except OSError:
            # A full or read-only disk must not turn recovery into another crash.
            logging.getLogger(__name__).error('Unable to save action traceback: %s', detail)

    @staticmethod
    def _safe_fallback(observation: FrameData) -> GameAction:
        # ACTION1 is the framework's UP action. Use public action metadata only;
        # emergency mode never re-enters world-model or perception helpers.
        if observation.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        legal = [a for a in GameAction if a is not GameAction.RESET
                 and a.value in observation.available_actions]
        action: Any = next((a for a in legal if not a.is_complex()),
                      legal[0] if legal else GameAction.ACTION1)
        if action.is_complex():
            action.set_data({'x': 0, 'y': 0})
        action.reasoning = {'why': 'choose_action exception recovery; unverified fallback'}
        return action

    def _ensure_perception(self, observation: FrameData) -> None:
        previous = self._observation
        self._observation = observation
        if not self._perception_started:
            self.perception.start(observation.frame, level=observation.levels_completed+1,
                                  state=observation.state.name)
            self._perception_started = True
        elif auditable():
            grid = freeze_grid(observation.frame[-1]) if observation.frame else self.perception.current.grid
            current = self.perception.current
            if (grid != current.grid or observation.levels_completed+1 != current.level
                    or observation.state.name != current.state
                    or (previous is not observation and observation.full_reset)):
                self.world.flush('observation_resync')
                self.world.event('prediction_counterexample', reason='observation_resync',
                                 expected_state=_state(current), actual_grid=grid)
                advanced = observation.levels_completed+1 != current.level
                self.perception.synchronize(observation.frame, level=observation.levels_completed+1,
                                            state=observation.state.name)
                self.world.memory.distill('observation_resync', level_advance=advanced)

    @staticmethod
    def _legal_actions(observation: FrameData) -> list[GameAction]:
        return [a for a in GameAction if a is not GameAction.RESET
                and a.value in observation.available_actions]

    def _sample(self, action: GameAction) -> GameAction:
        if action.is_complex():
            grid = self.perception.current.grid
            height = len(grid) or 64
            width = len(grid[0]) if grid else 64
            action.set_data({"x": self._rng.randrange(width), "y": self._rng.randrange(height)})
        return action

    @staticmethod
    def _coordinates(action: GameAction) -> tuple[int, int] | None:
        if not action.is_complex():
            return None
        data = action.action_data.model_dump()
        return int(data['x']), int(data['y'])

    def _resolve_action(self, proposal: GameAction, observation: FrameData,
                        *, reference: bool = True) -> GameAction:
        coordinates = self._coordinates(proposal)
        legal = self._legal_actions(observation)
        reason = (self.perception.blocked_reason(proposal.name, coordinates, reference=reference)
                  if proposal is GameAction.RESET or proposal in legal else 'unavailable_action')
        if reason is None:
            return proposal
        self.perception.note_blocked(proposal.name, coordinates, reason)
        if auditable():
            raise ProposalValidationError('action blocked before dispatch: ' + reason)
        self._rng.shuffle(legal)
        for candidate in legal:
            grid = self.perception.current.grid
            h, w = len(grid), len(grid[0]) if grid else 0
            attempts = w*h if candidate.is_complex() else 1
            offset = self._rng.randrange(attempts) if attempts else 0
            for i in range(attempts):
                if candidate.is_complex():
                    cell = (offset+i) % attempts
                    candidate.set_data({'x': cell % w, 'y': cell // w})
                coordinates = self._coordinates(candidate)
                reason = self.perception.blocked_reason(candidate.name, coordinates)
                if reason is None:
                    return candidate
                self.perception.note_blocked(candidate.name, coordinates, reason)
        # The archived reference guard explicitly permits immediate insistence.
        # Reconsider only the final refused reference action, never a hard dead
        # signature or consecutive RESET. The event remains visible in the log.
        if legal and reason == 'reference_noop':
            candidate = legal[-1]
            coordinates = self._coordinates(candidate)
            if self.perception.blocked_reason(candidate.name, coordinates) is None:
                self.perception.note_replan('reference_noop_override', candidate.name)
                return candidate
        # If all object signatures are exhausted, a single first RESET starts a
        # fresh attempt. Keep hard refusals intact; log the replan rather than
        # silently reviving a dead target. A repeated RESET is still forbidden.
        if legal and self.perception.last_action != 'RESET':
            self.perception.note_replan('admissible_targets_exhausted', 'RESET')
            return GameAction.RESET
        raise RuntimeError("all advertised actions blocked; replan required before dispatch")

    def do_action_request(self, action: GameAction) -> FrameData:
        if auditable() and (self.world.halted or self._recovery_mode):
            raise PlanningHalted('live moves halted; no recovery dispatch permitted')
        if self._recovery_mode:
            # Remain in best-effort mode for this game. Failed perception must
            # not crash dispatch or invent transitions used as replay evidence.
            response = super().do_action_request(action)
            self._observation = response
            return response
        # Final interception also protects direct framework/tool calls that did
        # not originate in choose_action. Snapshot mutable enum data before step.
        if self._observation is None:
            self._ensure_perception(self._convert_raw_frame_data(self.arc_env.observation_space))
        assert self._observation is not None
        if auditable():
            lifecycle_reset = (action is GameAction.RESET and self._observation.state in (
                GameState.NOT_PLAYED, GameState.GAME_OVER))
            simulator = self.world.pending is not None and self.world.pending.get('simulator')
            if ((self.world.tier_due() and not simulator)
                    or (self.world.pending is None and not lifecycle_reset)):
                self.world.flush('unapproved_dispatch')
                raise PlanningHalted('live dispatch requires a verified plan or lifecycle reset')
        if action is GameAction.RESET:
            self.world.flush('reset_proposal')
        elif self.world.queue and self.world.pending is None:
            self.world.flush('unapproved_dispatch')
            raise RuntimeError('queued actions require next_action authorization')
        approval = (action.name, self._coordinates(action), self.perception.current.step)
        if self.world.pending is not None:
            pending = self.world.pending
            expected = {k: v for k, v in pending.items() if k in ('action', 'x', 'y')}
            frame = self.perception.current
            if (_action(action.name, self._coordinates(action)) != expected
                    or self.world.queue_scope != (frame.level, frame.episode, frame.step)):
                self.world.flush('dispatch_mismatch')
                raise RuntimeError('dispatch does not match evidence-gated plan')
        selected = self._resolve_action(action, self._observation,
                                        reference=approval != self._dispatch_approval)
        if self.world.pending is not None and (selected.name, self._coordinates(selected)) != approval[:2]:
            self.world.flush('perception_blocked_plan')
            raise RuntimeError('planned action blocked; replan before dispatch')
        self._dispatch_approval = None
        coordinates = self._coordinates(selected)
        try:
            response = super().do_action_request(selected)
        except Exception as error:
            self.world.flush('dispatch_error')
            self.perception.note_dispatch_error(selected.name, coordinates, type(error).__name__)
            logging.getLogger(__name__).exception("live dispatch failed; no observed transition recorded")
            raise
        transition = self.perception.record(selected.name, coordinates, response.frame,
                                            level=response.levels_completed+1, state=response.state.name,
                                            full_reset=response.full_reset,
                                            level_completed=bool(getattr(response, 'level_completed', False)))
        self.world.observe(transition)
        self._observation = response
        return response


In [ ]:
%%writefile /tmp/agent_bundle/runtime.py
"""Ablatable prefix contract and level-weighted deployment budgets."""
from __future__ import annotations

import json
import os
import time
from typing import Any

TARGET_SECONDS = 29_700.0
HARD_CAP_SECONDS = 32_400.0
_RUN_STARTED = time.time()


class PrefixLock:
    """Validate full backend requests. Reset requires a fresh context identity.

    Call validate immediately before each backend request, passing its complete
    message list. Serialized snapshots prevent mutation through caller aliases.
    """

    def __init__(self) -> None:
        self._contexts: dict[str, tuple[str, ...]] = {}
        self._active: str | None = None

    def validate(self, context_id: str, messages: list[dict[str, Any]]) -> None:
        if not context_id or not messages:
            raise ValueError("context identity and messages are required")
        current = tuple(json.dumps(m, sort_keys=True, separators=(",", ":")) for m in messages)
        previous = self._contexts.get(context_id)
        if os.getenv("ENABLE_PREFIX_LOCK", "1") == "1":
            if previous is not None and context_id != self._active:
                raise ValueError("closed context cannot be reopened")
            if previous is not None and current[:len(previous)] != previous:
                raise ValueError("prefix mutation requires a new context identity")
        self._contexts[context_id] = current
        self._active = context_id


class LevelBudget:
    """Per-environment weighted allowance with unused-time rollover.

    The caller supplies the measured environment allocation and actual level
    count; no game-specific assumptions. begin_level returns a deadline suitable
    for request timeouts. expired enforces both that deadline and global caps.
    """

    def __init__(self, *, env_seconds: float, levels: int,
                 run_started: float | None = None, now: float | None = None) -> None:
        if not 0 < env_seconds <= TARGET_SECONDS or levels < 1:
            raise ValueError("invalid environment allocation or level count")
        now = time.time() if now is None else now
        self.run_started = now if run_started is None else run_started
        self.target_deadline = self.run_started + TARGET_SECONDS
        self.hard_deadline = self.run_started + HARD_CAP_SECONDS
        self.env_deadline = min(now + env_seconds, self.target_deadline)
        self.env_seconds = env_seconds
        self.levels = levels
        self.weight_sum = levels * (levels + 1) / 2
        self.rollover = 0.0
        self.level = 0
        self.level_deadline = self.env_deadline

    def begin_level(self, level: int, *, now: float | None = None) -> float:
        now = time.time() if now is None else now
        if not 1 <= level <= self.levels or level <= self.level:
            raise ValueError("levels must advance monotonically")
        if self.level:
            self.rollover = self.level_deadline - now
        self.level = level
        enabled = os.getenv("ENABLE_LEVEL_BUDGET", "1") == "1"
        self.level_deadline = min(
            self.env_deadline,
            now + max(0.0, self.env_seconds * level / self.weight_sum + self.rollover)
        ) if enabled else self.env_deadline
        return self.level_deadline

    def remaining(self, *, now: float | None = None) -> float:
        now = time.time() if now is None else now
        deadline = min(self.hard_deadline, self.target_deadline, self.env_deadline)
        if os.getenv("ENABLE_LEVEL_BUDGET", "1") == "1":
            deadline = min(deadline, self.level_deadline)
        return max(0.0, deadline - now)

    def generation_tokens(self, requested: int, *, now: float | None = None) -> int:
        """Degrade reasoning near exhaustion; zero requests best-effort actions."""
        left = self.remaining(now=now)
        if left <= 0:
            return 0
        return min(requested, 256) if left < 15 else requested


def run_expired() -> bool:
    """Shared notebook start includes setup overhead and survives process launch."""
    started = float(os.getenv("ARC3_RUN_STARTED", str(_RUN_STARTED)))
    return time.time() >= started + min(TARGET_SECONDS, HARD_CAP_SECONDS)


In [ ]:
%%writefile /tmp/agent_bundle/sandbox.py
"""Engine-free compute worker: stdin/stdout only; Linux seccomp is mandatory.

Callers pass immutable evidence as source literals/JSON, never filesystem handles.
No worker filesystem access is allowed, including scratch (host mediates storage).
"""
from __future__ import annotations

import os
import signal
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Any

# Runs under the sandbox interpreter, not the engine-enabled host interpreter.
_WORKER = r'''
import collections
import ctypes
import errno
import json
import math
import os
import resource
import sys
import traceback

source = sys.stdin.read()
limited = os.environ.get("ENABLE_SANDBOX_LIMITS", "1") == "1"
if limited:
    for kind, limit in ((resource.RLIMIT_AS, 512 * 1024**2),
                        (resource.RLIMIT_NPROC, 32), (resource.RLIMIT_NOFILE, 32),
                        (resource.RLIMIT_FSIZE, 1024**2), (resource.RLIMIT_CORE, 0)):
        resource.setrlimit(kind, (limit, limit))
# The allowlist remains mandatory even in resource ablations. It cannot be
# removed by generated code, including ctypes/native code in the same process.
lib = ctypes.CDLL("libseccomp.so.2", use_errno=True)
lib.seccomp_init.argtypes = [ctypes.c_uint32]
lib.seccomp_init.restype = ctypes.c_void_p
lib.seccomp_syscall_resolve_name.argtypes = [ctypes.c_char_p]
lib.seccomp_syscall_resolve_name.restype = ctypes.c_int
lib.seccomp_rule_add.argtypes = [ctypes.c_void_p, ctypes.c_uint32, ctypes.c_int, ctypes.c_uint]
lib.seccomp_load.argtypes = [ctypes.c_void_p]
lib.seccomp_release.argtypes = [ctypes.c_void_p]
ctx = lib.seccomp_init(0x00050000 | errno.EPERM)
if not ctx:
    raise RuntimeError("seccomp initialization failed")
for name in ("read", "write", "close", "fstat", "lseek", "brk", "mmap", "mprotect",
             "munmap", "mremap", "madvise", "futex", "clock_gettime", "gettimeofday",
             "rt_sigaction", "rt_sigprocmask", "rt_sigreturn", "sigaltstack",
             "getpid", "gettid", "getrandom", "sched_yield", "exit", "exit_group"):
    number = lib.seccomp_syscall_resolve_name(name.encode())
    if number < 0 or lib.seccomp_rule_add(ctx, 0x7fff0000, number, 0) != 0:
        raise RuntimeError("seccomp rule failed: " + name)
if lib.seccomp_load(ctx) != 0:
    raise RuntimeError("seccomp load failed")
lib.seccomp_release(ctx)
print("SANDBOX_READY", flush=True)
try:
    exec(compile(source, "<planner>", "exec"), {"__name__": "__main__"})
except BaseException as error:
    # Avoid traceback source lookup, which would itself need file access.
    print(type(error).__name__ + ": " + str(error), file=sys.stderr)
    sys.exit(1)
'''


def run_sandbox(source: str, *, python: Path, cwd: Path,
                timeout: float = 15.0, controlled_fixture: bool = False) -> dict[str, Any]:
    """Execute a planner, report errors explicitly and always reap the worker.

    Resource ablations require a controlled fixture; security is never ablated.
    Optional shorter timeouts support tests; production cannot exceed 15 seconds.
    """
    started = time.monotonic()
    limited = os.getenv("ENABLE_SANDBOX_LIMITS", "1") == "1"
    if not limited and not controlled_fixture:
        raise ValueError("resource ablation requires controlled_fixture=True")
    if not python.is_file():
        return {"ok": False, "error": "missing_sandbox", "ready": False}
    if not 0 < timeout <= 15:
        raise ValueError("timeout must be in (0, 15]")
    env = {"ENABLE_SANDBOX_LIMITS": "1" if limited else "0"}
    # Regular files plus RLIMIT_FSIZE bound output without pipe deadlock.
    with tempfile.TemporaryFile() as output, tempfile.TemporaryFile() as errors:
        try:
            process = subprocess.Popen(
                [str(python.absolute()), "-I", "-c", _WORKER], stdin=subprocess.PIPE,
                stdout=output, stderr=errors, cwd=cwd, env=env,
                close_fds=True, start_new_session=True,
            )
        except OSError as error:
            return {"ok": False, "ready": False, "error": str(error)}
        timed_out = False
        try:
            process.communicate(source.encode(), timeout=timeout)
        except subprocess.TimeoutExpired:
            timed_out = True
        finally:
            # No child creation is permitted; reap the whole group defensively.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
        output.seek(0)
        errors.seek(0)
        stdout = output.read(1024**2).decode(errors="replace")
        stderr = errors.read(1024**2).decode(errors="replace")
    ready = stdout.startswith("SANDBOX_READY\n")
    return {"ok": ready and not timed_out and process.returncode == 0,
            "ready": ready, "timeout": timed_out, "returncode": process.returncode,
            "stdout": stdout.removeprefix("SANDBOX_READY\n"), "stderr": stderr,
            "error": "timeout" if timed_out else ("worker_failed" if process.returncode else None),
            "elapsed_seconds": time.monotonic() - started}


In [ ]:
runner.verify_sources({'arclog.py': '0c119b81ad71aa39e84069dbbbabba7562aa462c94f5f19e5a2177a957c0375e', 'baseline/__init__.py': 'af3b0c639320a3a3559a3b41dfd7b3060205b8ad94b1e2b6d46071784fb3c8dc', 'baseline/animation.py': 'cb47d19f35e2ceeee160541963c12cc0d82ef2d608215db91128184c1e3fce86', 'baseline/bounded_search.py': 'dfa22cd0a686cf2315fdb942b387c572eb756fb9ce73db8b5439129710709ef2', 'baseline/checked_execution.py': '829aebfef19e039d7994ecd84bff6fd918244138fe0b1e941a076b5b9c297621', 'baseline/frame_diff.py': '843be0ce9225ae19de91dd34409bd7b7cf69ebad1ff73e69759c506c161bdeee', 'baseline/grid_coverage.py': '1d0b524149a146bc029d03029076aa865cafa2bde0296db61d678d481fbb957b', 'baseline/grid_display.py': 'c536ffaa0a1250bdea7a2f9a29b6f56dfe3b5f46abb9495b76dfbad51a730b0a', 'baseline/noop_repeat_guard.py': 'efad18a4c803b8a778f9561ebb13335d3d47372ee5c80fd6297a5f78258c5957', 'baseline/priority_scheduler.py': '46d77544a6285bdb5dad0e0c7fcd6e546f06961640fa13341f38b2cd5043fd19', 'baseline/replay_compare.py': '3cd7873bf52eac5edab570ac92f29ce309dbe9f530adec8a857717ab6abe90c7', 'baseline/retained_functions.py': 'aaf7750b02c4b56a9bb3253d69e52992a18cf6bc65a7ed88e0d6aedf81852c9f', 'baseline/state_contracts.py': '437a7ccef673e68c16c4b4ccc5c9fdd863c294eb38cc6c61e0728f7832044457', 'baseline/transition_evidence.py': '0c82056d9a21ca7ed573213e786e0a93828834925611df705d465eb22c376947', 'local_backend.py': '4c006f6b56dfae61ff019724ab90fc07150462f46a43d874bc1d2e3aec5ad1b9', 'my_agent.py': '5e6a76bc2376407c045fed7f4a7eed0e254e658009955b0c304c89b281826bc9', 'runtime.py': '2b59095e38911d5c0de52c9104990a601f89b4730e6ba05aa7a3b358c29c342e', 'sandbox.py': '59a26176fbd45f3022110a7b7edf0f04b916a074141d1cd585f2882e635b4dfe'})


In [ ]:
%%writefile /tmp/submission_audit.py
"""Fail-closed Gate 1; synthetic canaries never inspect actual game files."""
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

ROOT = Path('/tmp/agent_preflight')
sys.path.insert(0, str(ROOT))
from agent.sandbox import run_sandbox

FORBIDDEN_TOKENS = ['environment_files', 'ar25', 'bp35', 'cd82', 'cn04', 'dc22', 'ft09', 'g50t', 'ka59', 'lf52', 'lp85', 'ls20', 'm0r0', 'r11l', 're86', 's5i5', 'sb26', 'sc25', 'sk48', 'sp80', 'su15', 'tn36', 'tr87', 'tu93', 'vc33', 'wa30']


def audit(root: Path = ROOT, *, loop_timeout: float = 15.0) -> dict:
    violations: list[str] = []
    checks: dict[str, bool] = {}
    evidence: dict = {}
    sources = sorted((root / "agent").rglob("*.py"))
    checks["deployed_sources_present"] = all(
        (root / "agent" / name).is_file() for name in ("my_agent.py", "arclog.py"))
    for path in sources:
        content = path.read_text(encoding="utf-8").lower()
        tokens = [t for t in FORBIDDEN_TOKENS if t in content]
        # Reject any short game identifier, including identifiers not in the list.
        tokens += re.findall(r"(?<![a-z0-9_])[a-z]{2}\d{2}(?:-[a-f0-9]+)?(?![a-z0-9_])", content)
        if tokens:
            violations.append(f"{path.relative_to(root)}: forbidden tokens {sorted(set(tokens))}")
    sandbox = root / ".venv_agent_sandbox" / "bin" / "python"
    checks["sandbox_present"] = sandbox.is_file()
    checks["engine_import_denied"] = False
    if sandbox.is_file():
        # A clean venv must fail specifically with ModuleNotFoundError. Any other
        # subprocess error (including a broken interpreter) fails closed.
        probe = """import sys
try:
    import arc_agi
except ModuleNotFoundError as error:
    if error.name == 'arc_agi':
        print('ENGINE_ABSENT')
        sys.exit(0)
sys.exit(1)
"""
        try:
            result = subprocess.run([str(sandbox), "-I", "-c", probe],
                                    check=False, capture_output=True, text=True, timeout=15,
                                    env={}, cwd=root)
            checks["engine_import_denied"] = (
                result.returncode == 0 and result.stdout == "ENGINE_ABSENT\n")
            evidence["engine_probe_returncode"] = result.returncode
            with tempfile.TemporaryDirectory(prefix="containment-") as temp:
                fixture = Path(temp)
                forbidden = fixture / "environment_files"
                forbidden.mkdir()
                canary = forbidden / "canary.txt"
                canary.write_text("synthetic-secret")
                scratch = fixture / "scratch"
                scratch.mkdir()
                (scratch / "linked").symlink_to(forbidden, target_is_directory=True)
                attempts = {
                    "absolute_read": f"open({str(canary)!r}).read()",
                    "traversal_read": "open('../environment_files/canary.txt').read()",
                    "symlink_read": "open('linked/canary.txt').read()",
                    "write": f"open({str(canary)!r}, 'w').write('changed')",
                    "symlink_write": "open('linked/new.txt', 'w').write('new')",
                    "directory_list": f"__import__('os').listdir({str(forbidden)!r})",
                    "subprocess_escape": "__import__('os').system('cat ../environment_files/canary.txt')",
                    "fork_escape": "__import__('os').fork()",
                    "exec_escape": "__import__('os').execv('/bin/true', ['/bin/true'])",
                    "native_open": f"__import__('ctypes').CDLL(None).open({str(canary).encode()!r}, 0)",
                }
                # Assert denial in-worker; a crash/initialization failure cannot pass.
                for name, expression in attempts.items():
                    denied_results = (-1, 127 << 8) if name == "subprocess_escape" else (-1,)
                    source = f"""try:
    result = {expression}
except PermissionError:
    print('DENIED')
else:
    assert isinstance(result, int) and result in {denied_results!r}, 'access succeeded'
    print('DENIED')
"""
                    run = run_sandbox(source, python=sandbox, cwd=scratch,
                                      controlled_fixture=True)
                    checks[name] = run["ok"] and run["stdout"] == "DENIED\n"
                    evidence[name] = run
                checks["canary_unchanged"] = canary.read_text() == "synthetic-secret"
                checks["write_not_created"] = not (forbidden / "new.txt").exists()
                run = run_sandbox("print(sum(range(10)))", python=sandbox, cwd=scratch,
                                  controlled_fixture=True)
                checks["compute_works"] = run["ok"] and run["stdout"] == "45\n"
                if os.getenv("ENABLE_SANDBOX_LIMITS", "1") == "1":
                    source = "try:\n    bytearray(600 * 1024**2)\nexcept MemoryError:\n    print('MEMORY_CAPPED')"
                    run = run_sandbox(source, python=sandbox, cwd=scratch,
                                      controlled_fixture=True)
                    checks["memory_capped"] = run["ok"] and run["stdout"] == "MEMORY_CAPPED\n"
                    evidence["memory"] = run
                run = run_sandbox("raise ValueError('planner-canary')", python=sandbox,
                                  cwd=scratch, controlled_fixture=True)
                checks["planner_failure_visible"] = (run["ready"] and not run["ok"]
                                                     and "planner-canary" in run["stderr"])
                run = run_sandbox("while True: pass", python=sandbox, cwd=scratch,
                                  timeout=loop_timeout, controlled_fixture=True)
                checks["loop_terminated"] = (run["ready"] and not run["ok"] and run["timeout"]
                                             and run["elapsed_seconds"] <= loop_timeout + 2)
                evidence["loop"] = run
                # Missing interpreter must never produce a successful worker.
                run = run_sandbox("print(1)", python=fixture / "missing", cwd=scratch)
                checks["missing_worker_fails"] = not run["ok"] and not run["ready"]
        except (OSError, subprocess.SubprocessError, ValueError, KeyError) as error:
            violations.append(f"sandbox probe failed: {type(error).__name__}: {error}")
    for name, passed in checks.items():
        if not passed:
            violations.append(f"failed check: {name}")
    return {"containment_passed": not violations and all(checks.values()),
            "sandbox_isolated": all(checks.values()), "checks": checks,
            "violations": violations, "evidence": evidence,
            "resource_limits_enabled": os.getenv("ENABLE_SANDBOX_LIMITS", "1") == "1"}


def main() -> int:
    try:
        report = audit()
    except Exception as error:  # noqa: BLE001 - audit exceptions must produce a failed report
        report = {"containment_passed": False, "sandbox_isolated": False,
                  "violations": [f"audit failed: {type(error).__name__}: {error}"]}
    (ROOT / "containment.json").write_text(json.dumps(report, indent=2) + "\n")
    print(json.dumps(report, indent=2))
    return 0 if report["containment_passed"] else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
if not runner.rerun:
    runner.preflight_framework()


In [ ]:
%%writefile /tmp/arc3_llm_daemon.py
"""Bounded loopback-only supervisor for the extracted Pennyroyal launcher."""
from __future__ import annotations

import hashlib
import http.client
import json
import os
import signal
import subprocess
import sys
import time
from collections.abc import Callable
from pathlib import Path
from typing import Any


class LLMDaemon:
    def __init__(self, launcher: Path, working: Path, event: Callable[..., Any],
                 *, startup_seconds: float = 900.0) -> None:
        if not 0 < startup_seconds <= 900:
            raise ValueError('LLM startup must fit within 900 seconds')
        self.launcher, self.working, self.event = launcher, working, event
        self.startup_seconds = startup_seconds
        self.process: subprocess.Popen | None = None
        self.ready = False
        self.ever_ready = False
        self.reason = 'not_started'
        self.log: Any = None

    def models_ready(self, timeout: float) -> bool:
        connection = http.client.HTTPConnection('127.0.0.1', 8001, timeout=timeout)
        try:
            connection.request('GET', '/v1/models')
            response = connection.getresponse()
            if response.status != 200:
                return False
            raw = response.read(1_048_577)
            if len(raw) > 1_048_576:
                return False
            data = json.loads(raw)
            return any(item.get('id') == 'flashnext' for item in data.get('data', []))
        finally:
            connection.close()

    def record(self) -> None:
        path = self.working / 'diagnostics/llm_status.json'
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps({'ready': self.ready, 'ever_ready': self.ever_ready,
                                  'reason': self.reason,
                                  'endpoint': 'http://127.0.0.1:8001/v1',
                                  'mode': 'llm' if self.ready else 'exploration',
                                  'startup_seconds': self.startup_seconds}, indent=2) + '\n')

    def startup_logs(self) -> None:
        """Keep this run's own startup failures visible in its notebook log."""
        for path in (self.working / 'diagnostics/llm_launcher.log', self.working / 'serve.log'):
            if not path.is_file():
                continue
            try:
                with path.open('rb') as stream:
                    stream.seek(0, 2)
                    stream.seek(max(0, stream.tell() - 16_384))
                    tail = stream.read(16_384).decode(errors='replace')
            except OSError as error:
                self.event('llm_startup_log_unavailable', file=path.name, error=str(error))
            else:
                if tail:
                    self.event('llm_startup_log', file=path.name, tail=tail)

    def start(self) -> bool:
        os.environ['ARC3_LLM_READY'] = '0'
        os.environ.pop('ARC3_LLM_FAILURE', None)
        if os.getenv('ENABLE_OFFLINE_BACKEND', '1') != '1':
            self.reason = 'disabled_by_ablation'
            self.record()
            self.event('llm_disabled', reason=self.reason)
            return False
        began = time.monotonic()
        deadline = began + self.startup_seconds
        try:
            # An occupied port is not evidence that our model is running. Never
            # adopt or terminate another service, including gateway:8001.
            import socket
            with socket.socket() as probe:
                probe.settimeout(min(1.0, self.startup_seconds))
                if probe.connect_ex(('127.0.0.1', 8001)) == 0:
                    raise RuntimeError('loopback_port_occupied')
            digest = hashlib.sha256(self.launcher.read_bytes()).hexdigest()
            (self.working / 'diagnostics').mkdir(parents=True, exist_ok=True)
            self.log = (self.working / 'diagnostics/llm_launcher.log').open('ab', buffering=0)
            environment = dict(os.environ, ARC3_LLM_WORKING=str(self.working),
                               ARC3_LLM_PREFIX=f'/tmp/arc3-pennyroyal-{digest[:16]}',
                               HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', UV_OFFLINE='1',
                               UV_PYTHON_DOWNLOADS='never')
            self.process = subprocess.Popen([sys.executable, '-u', str(self.launcher)],
                                            env=environment, stdout=self.log,
                                            stderr=subprocess.STDOUT, start_new_session=True)
            self.event('llm_started', pid=self.process.pid, launcher_sha256=digest)
            while time.monotonic() < deadline:
                if self.process.poll() is not None:
                    raise RuntimeError(f'launcher_exited:{self.process.returncode}')
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    break
                try:
                    if self.models_ready(min(2.0, remaining)):
                        self.ready, self.ever_ready = True, True
                        self.reason = 'models_endpoint_ready'
                        os.environ['ARC3_LLM_READY'] = '1'
                        os.environ['ARC3_LLM_MODEL'] = 'flashnext'
                        self.record()
                        self.event('llm_ready', elapsed_seconds=time.monotonic() - began)
                        return True
                except (OSError, ValueError, TypeError, AttributeError, http.client.HTTPException):
                    pass
                time.sleep(min(0.5, max(0.0, deadline - time.monotonic())))
            raise TimeoutError(f'startup_deadline_{self.startup_seconds:g}s')
        except (OSError, ValueError, RuntimeError, TypeError, AttributeError,
                subprocess.SubprocessError, http.client.HTTPException) as error:
            self.reason = f'{type(error).__name__}: {error}'
            self.stop()
            self.record()
            self.startup_logs()
            os.environ['ARC3_LLM_FAILURE'] = self.reason
            self.event('llm_fallback', reason=self.reason, mode='exploration')
            return False
        except BaseException:
            # Notebook interruption must not leave a half-installed launcher
            # or detached CUDA workers consuming resources.
            self.stop()
            raise

    def stop(self) -> None:
        self.ready = False
        os.environ['ARC3_LLM_READY'] = '0'
        if self.process is not None:
            try:
                os.killpg(self.process.pid, signal.SIGTERM)
                self.process.wait(timeout=2)
            except (ProcessLookupError, subprocess.TimeoutExpired):
                pass
            finally:
                try:
                    os.killpg(self.process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                self.process.wait()
                self.event('llm_stopped', returncode=self.process.returncode)
                self.process = None
        if self.log is not None:
            self.log.close()
            self.log = None
        if self.ever_ready:
            self.reason = 'stopped_after_ready'
            self.record()


In [ ]:
%%writefile /tmp/arc3_llm_launcher.py
import os, time
from pathlib import Path
WORKING_DIR = Path(os.environ["ARC3_LLM_WORKING"])
SERVED_MODEL_PORT = 8001
def resolve_path(candidates: list[str]) -> str:
    for c in candidates:
        if os.path.exists(c):
            return c
    first_path = Path(candidates[0])
    target_name = first_path.name
    for p in Path('/kaggle/input').rglob(target_name):
        if p.exists():
            return str(p)
    stem = first_path.stem
    for p in Path('/kaggle/input').glob(f'**/*{stem}*'):
        if p.exists():
            return str(p)
    return candidates[0]

WHEELHOUSE_DIR = resolve_path([
    '/kaggle/input/pennyroyal-v253',
    '/kaggle/input/datasets/dfranzen/pennyroyal-v253',
    '/kaggle/input/dfranzen/pennyroyal-v253',
])

MODEL_DIR = resolve_path(['/kaggle/input/intel-qwen3-8-flash-next-w4a16-autoround', '/kaggle/input/datasets/mirzamilanfarabi/intel-qwen3-8-flash-next-w4a16-autoround', '/kaggle/input/mirzamilanfarabi/intel-qwen3-8-flash-next-w4a16-autoround'])

DRAFT_MODEL_DIR = resolve_path(['/kaggle/input/albucino-qwen3-8-flash-next-drafter', '/kaggle/input/datasets/mirzamilanfarabi/albucino-qwen3-8-flash-next-drafter', '/kaggle/input/mirzamilanfarabi/albucino-qwen3-8-flash-next-drafter'])

SERVED_MODEL_NAME = 'flashnext'

SERVED_MODEL_PORT = 8001

import threading
from concurrent.futures import ThreadPoolExecutor
def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except OSError as error: print(f"Prefetch skipped: {error}", flush=True)

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except OSError as error: print(f"Prefetch skipped: {error}", flush=True)

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
precache_model_thread = None
if os.getenv("ENABLE_BASELINE_PREFETCH", "1") == "1":
    precache(WHEELHOUSE_DIR)
    precache_model_thread = threading.Thread(target=precache,
        args=(MODEL_DIR, DRAFT_MODEL_DIR) if os.getenv("ENABLE_SPECULATIVE_DRAFT", "1") == "1" else (MODEL_DIR,),
        kwargs={"delay": 60, "threads": 3}, daemon=True)
"""Inspect versioned offline inputs before the extracted launcher installs them."""

import hashlib
import json
import zipfile
from pathlib import Path

SGLANG_WHEEL = 'sglang-0.5.19+gd00d88efc8d6-cp312-cp312-linux_x86_64.whl'


def asset_digest(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def checkpoint_report(path: Path) -> dict:
    index_path = path / 'model.safetensors.index.json'
    index = json.loads(index_path.read_text())
    names = sorted(set(index['weight_map'].values()))
    if not names:
        raise ValueError('empty model shard index')
    shards = {}
    for name in names:
        relative = Path(name)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('unsafe shard path')
        shard = path / relative
        if not shard.is_file() or shard.stat().st_size < 8:
            raise FileNotFoundError(f'missing or empty weight shard: {shard}')
        shards[name] = shard.stat().st_size
    return {'path': str(path), 'config_sha256': asset_digest(path / 'config.json'),
            'index_sha256': asset_digest(index_path), 'shard_sizes': shards,
            'shard_content_hashes': 'not independently supplied; Kaggle input provenance is recorded separately'}


def verify_offline_assets(wheelhouse: Path, target: Path, draft: Path | None) -> dict:
    wheel_dirs = [wheelhouse / 'wheels'] if (wheelhouse / 'wheels').is_dir() else list(wheelhouse.rglob('wheels'))
    if len(wheel_dirs) != 1:
        raise FileNotFoundError('expected one offline wheels directory')
    wheels = wheel_dirs[0]
    locks = list(wheelhouse.rglob('requirements.lock'))
    if len(locks) != 1 or not (wheels / SGLANG_WHEEL).is_file():
        raise FileNotFoundError('missing lock or pinned Pennyroyal wheel')
    files = sorted(wheels.glob('*.whl'))
    if len(list(wheels.glob('uv-*.whl'))) != 1:
        raise FileNotFoundError('missing or ambiguous bundled uv wheel')
    hashes = {}
    for path in files:
        if not zipfile.is_zipfile(path):
            raise ValueError(f'invalid wheel archive: {path.name}')
        hashes[path.name] = asset_digest(path)
    config = json.loads((target / 'config.json').read_text())
    quantization = config.get('quantization_config', {})
    if quantization.get('quant_method') != 'auto-round' or quantization.get('bits') != 4:
        raise ValueError('target is not the pinned AutoRound INT4 format')
    report = {'wheelhouse': str(wheelhouse), 'wheel_sha256_observed': hashes,
              'requirements_sha256': asset_digest(locks[0]), 'target': checkpoint_report(target),
              'identity': 'versioned Kaggle inputs; observed hashes are recorded, not invented upstream signatures'}
    if draft is not None:
        candidates = []
        for path in draft.rglob('config.json'):
            quant = json.loads(path.read_text()).get('quantization_config', {})
            if quant.get('quant_method') == 'compressed-tensors' and 'mtp_routed_experts' in quant.get('config_groups', {}):
                candidates.append(path.parent)
        if len(candidates) != 1:
            raise ValueError('expected one compatible compressed-tensors draft checkpoint')
        report['draft'] = checkpoint_report(candidates[0])
    return report

asset_report = verify_offline_assets(Path(WHEELHOUSE_DIR), Path(MODEL_DIR),
    Path(DRAFT_MODEL_DIR) if os.getenv("ENABLE_SPECULATIVE_DRAFT", "1") == "1" else None)
asset_report_path = WORKING_DIR / "diagnostics/llm_assets.json"
asset_report_path.parent.mkdir(parents=True, exist_ok=True)
asset_report_path.write_text(json.dumps(asset_report, indent=2) + "\n")
# Paste this entire file into the launcher cell.
# Required notebook variables: MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR,
# WORKING_DIR, SERVED_MODEL_NAME, SERVED_MODEL_PORT.
# MODEL_DIR: Intel AutoRound INT4 target (BF16 PLE or albucino FP8 PLE).
# DRAFT_MODEL_DIR: downloaded albucino runtime/mtp-int4-g32, or its dataset root.
import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error
import copy, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path

PREFIX = os.environ["ARC3_LLM_PREFIX"]       # Separate from the previous fork's /tmp/sgl venv.
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=65536,
    MEMFRAC=0.88,
    MAXREQ=4,
    CUDAGRAPH_MAXBS=4,
    MAMBA_CACHE=16,
    CHUNK=8192,
    MAX_PREFILL=16384,
    SPEC=os.getenv("ENABLE_SPECULATIVE_DRAFT", "1") == "1",
    FRSPEC=True,
    AUTOTUNE=True,
    # Preserve the original Kaggle checkpoint prefetch/cache behavior.
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    LINEAR_BACKEND="flashinfer",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="bfloat16",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=3,
)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]
    # Content/path-specific directory makes notebook reruns idempotent without
    # overwriting an unrelated view. We never modify the read-only datasets.
    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view


# ---- validate notebook inputs / identify the offline bundle ----
MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
# Refuse to start a second server on an occupied port; do not kill other jobs.
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
        raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
Path(PREFIX).mkdir(parents=True, exist_ok=True)
Path(LOG).parent.mkdir(parents=True, exist_ok=True)
WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
wheel = sglang_wheels[0]
install_marker = Path(PREFIX) / "installed-bundle.txt"

# ---- one-time offline install, isolated from the old mratsim environment ----
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(SGLANG).exists() or not install_marker.is_file():
    print("=== installing Pennyroyal (offline) ===")
    # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
    uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
    assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
    uv_binary = Path(PREFIX) / "bin/uv"
    uv_binary.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_wheels[0]) as archive:
        members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
        assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
        binary = archive.read(members[0])
        assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
        uv_binary.write_bytes(binary)
    uv_binary.chmod(0o755)
    uv = [str(uv_binary)]
    if not Path(PYTHON).exists():
        run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
    install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--only-binary", ":all:", "--find-links", str(WHEELS)]
    run(install + ["-r", str(LOCK)], env=install_env)
    run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
    install_marker.write_text('install')
else:
    print(f"Matching offline bundle already installed at {VENV}")

# ---- bundled CUDA toolkit and linker symlinks ----
cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
    raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
CUDA_HOME = str(cu13[0])
libdir = Path(CUDA_HOME) / "lib"
lib64 = Path(CUDA_HOME) / "lib64"
if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
for so in sorted(libdir.glob("*.so.*")):
    link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
    if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"

# Clean sitecustomize in venv and clean_py_dir to prevent Kaggle host import errors (e.g. wrapt)
clean_py_dir = Path(PREFIX) / "clean_python"
clean_py_dir.mkdir(parents=True, exist_ok=True)
(clean_py_dir / "sitecustomize.py").write_text("# Clean sitecustomize\n")
# Clean sitecustomize in venv to prevent Kaggle host import errors (e.g. wrapt)
for pdir in Path(VENV).glob("lib/python*/site-packages"):
    (pdir / "sitecustomize.py").write_text("# Clean isolated venv sitecustomize\n")

# Unconditionally replace stub libcuda in libdir and lib64 with the host NVIDIA driver
candidates = []
for directory in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/lib64", "/usr/local/nvidia/lib"):
    candidates += [p for p in Path(directory).glob("libcuda.so*") if p.is_file() and "stub" not in str(p).lower()]
if not candidates:
    code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
    try:
        maps = run([PYTHON, "-c", code], env=install_env).stdout
        candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps) if "stub" not in p.lower()]
    except Exception:
        pass
if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
# Prioritize real versioned driver libraries over unversioned or symlinks
candidates = sorted(set(candidates), key=lambda p: (0 if ".so." in p.name else 1, -len(p.name)))
target_driver = candidates[0].resolve()
print(f"Host GPU driver libcuda: {target_driver} (from {len(candidates)} candidates: {[str(c) for c in candidates[:3]]})")
for d in {libdir, lib64}:
    if d.exists() and not d.is_symlink():
        for link_name in ["libcuda.so", "libcuda.so.1"]:
            link = d / link_name
            if link.is_symlink() or link.exists(): link.unlink()
            link.symlink_to(target_driver)
            print(f"Symlinked {link} -> {target_driver}")

CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
assert CXX, "Host C++ compiler not found"
CC = CXX.replace("g++", "gcc")
assert shutil.which(CC), f"Matching C compiler missing: {CC}"

# ---- runtime environment ----
env = dict(os.environ)
env["PYTHONPATH"] = str(clean_py_dir)
cache = f"{PREFIX}/cache"
for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
    Path(cache, name).mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
    "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:/usr/local/nvidia/lib64:/usr/lib/x86_64-linux-gnu:" + env.get("LD_LIBRARY_PATH", ""),
    "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
    "TORCH_CUDA_ARCH_LIST": "12.0",
    "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
    "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
    "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
    "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
    "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
    "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
    "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
    "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
    "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
    "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
    "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
    "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
    "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
})
run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
print("Host compiler:", CXX)
# Report the optional later caching patch. This cell does not modify wheel code.
run([PYTHON, "-c", '''import os, pathlib, sglang, subprocess, torch
print("sglang", sglang.__version__, "| torch", torch.__version__)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH"))
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout:
        print("nvidia-smi:")
        print(res.stdout)
    if res.stderr:
        print("nvidia-smi stderr:")
        print(res.stderr)
except Exception as e:
    print("nvidia-smi exec error:", e)

try:
    torch.cuda.init()
    print("torch.cuda.init() succeeded! Device count:", torch.cuda.device_count())
except Exception as e:
    print("torch.cuda.init() error:", repr(e))

if not torch.cuda.is_available():
    raise RuntimeError(f"CUDA is unavailable! Device count={torch.cuda.device_count()}")
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
'''], env=env)

# ---- validate target and create the read-only draft's SGLang view ----
target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
q = target_cfg.get("quantization_config", {})
assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
_, target_shards = indexed_shards(MODEL_DIR)
print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
DRAFT_VIEW = None
if CFG["SPEC"]:
    assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
    DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)

# ---- assemble Pennyroyal / AutoRound launch arguments ----
graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"]})
graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
    "--model-loader-extra-config", '{"enable_multithread_load":false}',
    "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
    "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
    "--dtype", "bfloat16", "--quantization", "auto-round",
    "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
    "--context-length", str(CFG["CTX"]), "--page-size", "64",
    "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
    "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
    "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
    "--cuda-graph-bs-decode", *map(str, graph_bs),
    "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
    "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
    "--mamba-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
    "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
    "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
    "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
    "--default-chat-template-kwargs", '{"preserve_thinking":true}',
    "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
    "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
tmpl = Path(MODEL_DIR, "chat_template.jinja")
if tmpl.is_file(): args += ["--chat-template", str(tmpl)]
if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
if CFG["SPEC"]:
    args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
        "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
        "--speculative-draft-model-path", str(DRAFT_VIEW),
        "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
        "--speculative-draft-kv-cache-dtype", CFG["KVDTYPE"],
        "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
        "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
    if CFG["FRSPEC"]:
        tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
        if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
        assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
        assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
        args += ["--speculative-token-map", str(tok)]


if precache_model_thread is not None:
    precache_model_thread.start()
print("SGLang launch:", args, flush=True)
with open(LOG, "ab", buffering=0) as logf:
    proc = subprocess.Popen(args, env=env, stdout=logf, stderr=subprocess.STDOUT)
raise SystemExit(proc.wait())


In [ ]:
runner.start_llm(startup_seconds=900)


In [ ]:
runner.run()
